In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import re

import numpy as np
import pandas as pd

# Здесь задаём пути к уже готовым финальным таблицам двух магазинов.
# Берём именно финальные файлы после ручных brand/product_type правил,
# потому что хотим строить blocking уже на максимально сильных полях.
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

SKL_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_with_manual_triggers.csv'
ALPHA_PATH = OUTPUT_DIR / 'alphamega_brand_product_type_with_manual_triggers.csv'

CANDIDATE_PAIRS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_v1.csv'
CANDIDATE_SUMMARY_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_blocking_summary_v1.csv'

sk_raw = pd.read_csv(SKL_PATH, encoding='utf-8-sig').fillna('')
alpha_raw = pd.read_csv(ALPHA_PATH, encoding='utf-8-sig').fillna('')

print('Sklavenitis rows:', len(sk_raw))
print('Alpha Omega rows:', len(alpha_raw))


Sklavenitis rows: 3342
Alpha Omega rows: 8672


# Основной пайплайн Sklavenitis

В этом ноутбуке собраны основные шаги подготовки данных для будущего матчинга товаров между магазинами.

In [2]:
base_path = Path(r"C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\raw_data")

sklavenitis = pd.read_csv(base_path / "sklavenitis-ypsonas.csv", encoding="utf-8")
alphamega = pd.read_csv(base_path / "alphamega.csv", encoding="utf-8")
alphamega
# list(alphamega.brand.drop_duplicates().sort_values())

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,image_url,product_url,sku,scraped_at
0,alphamega,Limassol,Frozen,Qualifood Salmon 4x125 g,Qualifood,14.99 EUR,29.98 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,802273,2026-03-13T21:04:22.682Z
1,alphamega,Limassol,Frozen,Creta Farms En Elladi Chicken Nuggets 400 g,Creta Farms,4.65 EUR,11.63 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,781899,2026-03-13T21:04:22.682Z
2,alphamega,Limassol,Frozen,Green Isle Frozen Blueberries 300 g,Green Isle,3.39 EUR,11.30 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,869876,2026-03-13T21:04:22.682Z
3,alphamega,Limassol,Frozen,Qualifood Salmon Fillet With Skin 4x125 g,Qualifood,11.99 EUR,23.98 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,879997,2026-03-13T21:04:22.682Z
4,alphamega,Limassol,Frozen,Party Ice Ice Cubes 1.2 kg,Party Ice,1.65 EUR,1.38 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,894580,2026-03-13T21:04:22.682Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8667,alphamega,Limassol,Food Cupboard,Twix Xtra Chocolate 75 g,Twix,1.09 EUR,14.53 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,794425,2026-03-13T21:04:22.682Z
8668,alphamega,Limassol,Food Cupboard,Pop A Corn Unsalted Plain Popcorn 75 g,Pop A Corn,1.59 EUR,21.20 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,789680,2026-03-13T21:04:22.682Z
8669,alphamega,Limassol,Food Cupboard,Lawry's Garlic Salt 85 g,Lawrys,3.60 EUR,42.35 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,786721,2026-03-13T21:04:22.682Z
8670,alphamega,Limassol,Food Cupboard,Agia Skepi Organic Lentils 500 g,Agia Skepi,2.95 EUR,5.90 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,785240,2026-03-13T21:04:22.682Z


In [3]:
alphamega

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,image_url,product_url,sku,scraped_at
0,alphamega,Limassol,Frozen,Qualifood Salmon 4x125 g,Qualifood,14.99 EUR,29.98 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,802273,2026-03-13T21:04:22.682Z
1,alphamega,Limassol,Frozen,Creta Farms En Elladi Chicken Nuggets 400 g,Creta Farms,4.65 EUR,11.63 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,781899,2026-03-13T21:04:22.682Z
2,alphamega,Limassol,Frozen,Green Isle Frozen Blueberries 300 g,Green Isle,3.39 EUR,11.30 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,869876,2026-03-13T21:04:22.682Z
3,alphamega,Limassol,Frozen,Qualifood Salmon Fillet With Skin 4x125 g,Qualifood,11.99 EUR,23.98 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,879997,2026-03-13T21:04:22.682Z
4,alphamega,Limassol,Frozen,Party Ice Ice Cubes 1.2 kg,Party Ice,1.65 EUR,1.38 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/froz...,894580,2026-03-13T21:04:22.682Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8667,alphamega,Limassol,Food Cupboard,Twix Xtra Chocolate 75 g,Twix,1.09 EUR,14.53 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,794425,2026-03-13T21:04:22.682Z
8668,alphamega,Limassol,Food Cupboard,Pop A Corn Unsalted Plain Popcorn 75 g,Pop A Corn,1.59 EUR,21.20 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,789680,2026-03-13T21:04:22.682Z
8669,alphamega,Limassol,Food Cupboard,Lawry's Garlic Salt 85 g,Lawrys,3.60 EUR,42.35 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,786721,2026-03-13T21:04:22.682Z
8670,alphamega,Limassol,Food Cupboard,Agia Skepi Organic Lentils 500 g,Agia Skepi,2.95 EUR,5.90 EUR/1kg,NaN,NaN,in_stock,/Admin/Public/GetImage.ashx?width=640&crop=5&F...,https://www.alphamega.com.cy/en/groceries/food...,785240,2026-03-13T21:04:22.682Z


In [4]:
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,image_url,product_url,sku,scraped_at
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS Produc...,Spaghetti,2.90 EUR,6.04 EUR/kilogram,NaN,NaN,unknown,https://wolt-menu-images-cdn.wolt.com/menu-ima...,https://wolt.com/el/cyp/limassol/venue/sklaven...,0c007653052dfddbe3fa8b9a,2026-03-13T22:07:58.656Z
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Fre...,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,NaN,NaN,unknown,https://imageproxy.wolt.com/assets/697364789b9...,https://wolt.com/el/cyp/limassol/venue/sklaven...,1a2f60292a699144d9dc5a8f,2026-03-13T22:07:58.656Z
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten F...",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,https://imageproxy.wolt.com/assets/697364789b9...,https://wolt.com/el/cyp/limassol/venue/sklaven...,1eb4903edc4615c66a912397,2026-03-13T22:07:58.656Z
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 40...,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,NaN,NaN,unknown,https://imageproxy.wolt.com/assets/697364769b9...,https://wolt.com/el/cyp/limassol/venue/sklaven...,48f15a58632b152a11126d91,2026-03-13T22:07:58.656Z
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 4...,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,https://imageproxy.wolt.com/assets/6973718b4af...,https://wolt.com/el/cyp/limassol/venue/sklaven...,57d9ef7bd483a42d6a545d04,2026-03-13T22:07:58.656Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spr...,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,NaN,NaN,unknown,https://wolt-menu-images-cdn.wolt.com/menu-ima...,https://wolt.com/el/cyp/limassol/venue/sklaven...,6894915763d649380a188d53,2026-03-13T22:15:24.030Z
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spr...,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,NaN,NaN,unknown,https://wolt-menu-images-cdn.wolt.com/menu-ima...,https://wolt.com/el/cyp/limassol/venue/sklaven...,6894915763d649380a188d54,2026-03-13T22:15:24.030Z
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF5...,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,NaN,NaN,unknown,https://wolt-menu-images-cdn.wolt.com/menu-ima...,https://wolt.com/el/cyp/limassol/venue/sklaven...,6894915763d649380a188d55,2026-03-13T22:15:24.030Z
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,NaN,NaN,unknown,https://wolt-menu-images-cdn.wolt.com/menu-ima...,https://wolt.com/el/cyp/limassol/venue/sklaven...,6894915763d649380a188d56,2026-03-13T22:15:24.030Z


In [5]:
sklavenitis.promo_label.unique()

array([nan])

In [6]:
sklavenitis.promo_label.notna().sum()

np.int64(0)

In [7]:
sklavenitis.category_path.unique()

array(['Lenten Food', 'New',
       'Fresh Fruits & Vegetables > Fresh Nut Selection',
       'Fresh Fruits & Vegetables > Fruits',
       'Fresh Fruits & Vegetables > Vegetables',
       'Fresh Fruits & Vegetables > Vegetable in Bunches',
       'Fresh Fruits & Vegetables > Prepacked Salads -Fruits & Herbs',
       'Fresh Fruits & Vegetables > Bio Fruits & Vegs',
       'Bakery, Cakes & Treats > Treats',
       'Bakery, Cakes & Treats > Bread & Bread Products',
       'Bakery, Cakes & Treats > Sliced Toast Bread',
       'Bakery, Cakes & Treats > Pitta Bread',
       'Bakery, Cakes & Treats > Cakes - Pancakes & More ',
       'Bakery, Cakes & Treats > Croissants',
       'Bakery, Cakes & Treats > Gluten Free Bread',
       'Bakery, Cakes & Treats >  Breadsticks, Rusks & Toasts',
       'Ready Meals > Cooked Food',
       'Ready Meals > Salads-Sandwiches -Fruit Snacks & ',
       'Ready Meals > Sushi', 'Isothermal Bags ',
       'Fresh Meat -Fish & Seafood > Premium Meat',
       'Fres

In [8]:
sklavenitis.query("brand == 'Barilla'")

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,image_url,product_url,sku,scraped_at


In [9]:
sklavenitis.brand.unique()

array(['Spaghetti', 'MACEDONIAN Halva', 'MAKEDONIKOS Halva', ...,
       'GARNIER Ambre', 'NIVEA SUN', 'NIVEA Sun'],
      shape=(1917,), dtype=object)

In [10]:
# list(sklavenitis.brand.drop_duplicates().sort_values())

In [11]:
# sklavenitis[['brand_clean', 'product_type']] = sklavenitis['brand'].str.split(' ', n=1, expand=True)

In [12]:
# sklavenitis

In [13]:
sklavenitis = sklavenitis.drop(columns=['image_url', 'product_url', 'sku'])

In [14]:
sklavenitis= sklavenitis.drop(columns=['availability'])

In [15]:
# sklavenitis['brand_clean'] = sklavenitis['brand_clean'].str.capitalize()
# sklavenitis['product_type'] = sklavenitis['product_type'].str.capitalize()
# sklavenitis

# чтобы удалить строки, где в конкретных столбцах есть пропуски:
# sklavenitis = sklavenitis.dropna(subset=['product_name', 'brand', 'price_unit'])
sklavenitis
sklavenitis = sklavenitis.dropna(subset=['product_name', 'brand'], how='all')
sklavenitis = sklavenitis.dropna(subset=['price', 'price_unit'], how='all')
sklavenitis.price.count()

np.int64(3342)

In [16]:
# sklavenitis

## 1. Очистка и нормализация названия товара

Здесь мы приводим product_name к более чистому и единообразному виду, чтобы потом проще было извлекать размеры, grade и собирать product_core.


In [17]:
# нормализуем название продукта:
# приведем все к строчным буквам, 
# уберем пробелы перед началом и после конца,
# заменим все повторяющиеся пробелы, переносы строк, табы на один обычный пробел \s - это все виды пробелов, + - больше одного подряд 
# NEW: убираем невидимые zero-width символы, которые ломают слова вроде b\u200bread

sklavenitis['product_name_clean'] = (
    sklavenitis['product_name']
    .str.lower()
    .str.replace(r'[\u200b\u200c\u200d\ufeff]', '', regex=True)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

In [18]:
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,scraped_at,product_name_clean
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS Produc...,Spaghetti,2.90 EUR,6.04 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis produc...
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Fre...,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-fre...
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten F...",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,"makedonikos halva with cocoa – vegan, gluten f..."
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 40...,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 40...
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 4...,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 4...
...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spr...,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spr...
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spr...,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spr...
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF5...,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf5...
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml 770881


In [19]:
# приводим к одному виду умножения и запятые с точками
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .fillna('')
    .str.replace('×', 'x', regex=False)
    .str.replace(',', '.', regex=False)
)

In [20]:
sklavenitis[sklavenitis['product_name'].str.startswith('KL', na=False)].sort_values(by='product_name')
# sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,scraped_at,product_name_clean
3129,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Napkins & Tissues,KLEENEX Facial Tissues 3Ply 3x80 Pcs 648978,NaN,3.98 EUR,1.33 EUR/piece,NaN,NaN,2026-03-13T22:14:45.508Z,kleenex facial tissues 3ply 3x80 pcs 648978
3078,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Toilet Paper,KLEENEX Premium Toilet Paper Cream 12 Pcs 801855,NaN,10.50 EUR,0.88 EUR/piece,NaN,NaN,2026-03-13T22:14:39.507Z,kleenex premium toilet paper cream 12 pcs 801855
3079,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Toilet Paper,KLEENEX Premium Toilet Paper White 12 Pcs 801854,NaN,10.50 EUR,0.88 EUR/piece,NaN,NaN,2026-03-13T22:14:39.507Z,kleenex premium toilet paper white 12 pcs 801854
3069,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Toilet Paper,KLEENEX Scottonellepure Clean Mega Toilet Pape...,NaN,11.25 EUR,0.63 EUR/piece,NaN,NaN,2026-03-13T22:14:39.507Z,kleenex scottonellepure clean mega toilet pape...
3134,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Napkins & Tissues,KLEENEX Tissues 48 Pcs 751732,NaN,1.88 EUR,NaN,NaN,NaN,2026-03-13T22:14:45.508Z,kleenex tissues 48 pcs 751732
3124,sklavenitis-ypsonas,Limassol,Everyday Paper Needs > Kitchen Roll,KLEENEX Travel Tissues White 3Ply x 40Pcs 622352,NaN,1.60 EUR,0.53 EUR/piece,NaN,NaN,2026-03-13T22:14:43.412Z,kleenex travel tissues white 3ply x 40pcs 622352
184,sklavenitis-ypsonas,Limassol,Fresh Fruits & Vegetables > Vegetables,KLEOPAS Cherry Red Tomato 500g Pre-Packed 8...,KLEOPAS Cherry,2.45 EUR,4.90 EUR/kilogram,NaN,NaN,2026-03-13T22:08:16.898Z,kleopas cherry red tomato 500g pre-packed 851752
166,sklavenitis-ypsonas,Limassol,Fresh Fruits & Vegetables > Vegetables,KLEOPAS Cherry Black Tomato 500g Pre-Packed 8...,KLEOPAS Cherry,3.65 EUR,7.30 EUR/kilogram,NaN,NaN,2026-03-13T22:08:16.898Z,kleopas cherry black tomato 500g pre-packed 85...
225,sklavenitis-ypsonas,Limassol,Fresh Fruits & Vegetables > Prepacked Salads -...,KLEOPAS Kiwi – Fresh Cut 380g 844742,KLEOPAS Kiwi,4.65 EUR,12.24 EUR/kilogram,NaN,NaN,2026-03-13T22:08:21.725Z,kleopas kiwi – fresh cut 380g 844742
226,sklavenitis-ypsonas,Limassol,Fresh Fruits & Vegetables > Prepacked Salads -...,KLEOPAS Mango – Fresh Cut Mango 350g 844741,KLEOPAS Mango,6.45 EUR,18.43 EUR/kilogram,NaN,NaN,2026-03-13T22:08:21.725Z,kleopas mango – fresh cut mango 350g 844741


In [21]:
# удаляем мусорные цифры после единиц измерения в названии

# только для обычных названий заканчивающихся на типа 250g
# sklavenitis['product_name_clean'] = (
#     sklavenitis['product_name_clean']
#     .str.replace(
#         r'(\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s+\d+\s*$',
#         r'\1',
#         regex=True
#     )
# )

# для обычных названий и названий типа 9x250ml
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(
        r'(\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s+\d+\s*$',
        r'\1',
        regex=True
    )
)

In [22]:
# чтобы пандас все показывал, а не ...
pd.set_option('display.max_colwidth', None)
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,scraped_at,product_name_clean
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis production 480g
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-free 400g
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,makedonikos halva with cocoa – vegan. gluten free 400g
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 400g
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 400g
...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 374831,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 751207,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF50+ Children's & Infant Sunscreen Face & Body 200ml 770880,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body 200ml
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,NaN,NaN,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml


In [23]:
# посмотреть строки, которые НЕ заканчиваются на единицы измерения

# только для обычных названий заканчивающихся на типа 250g
# sklavenitis[
#     ~sklavenitis['product_name_clean'].str.contains(r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$', regex=True, na=False)
# ][['product_name_clean']].head(30)

# для обычных названий и названий типа 9x250ml
sklavenitis[
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']].head(40)

,product_name_clean
58,wettex sponges towel no. 3 – pink 141141
67,agia skepi organic eggs size m x6 653367
91,agroktima fragkou organic eggs x6 652666
94,fresh whole walnuts 4086
95,fresh pecan walnuts local produce 4106
97,zagorin apples imported 4435
98,bajela bananas 4200
99,bananas local produce 4323
101,conference pears 4303
102,pink lady apples 4455


In [24]:
# убрать этот хвост кода в названии

# \s — перед кодом должен быть пробел
# \d{4,} — потом 4 или больше цифр подряд
# \s*$ — и это всё должно быть в конце строки ($)
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s\d{4,}\s*$', '', regex=True)
)

In [25]:
# посмотреть снова строки, которые НЕ заканчиваются на единицы измерения
sklavenitis[
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']].head(40)

,product_name_clean
58,wettex sponges towel no. 3 – pink
67,agia skepi organic eggs size m x6
91,agroktima fragkou organic eggs x6
94,fresh whole walnuts
95,fresh pecan walnuts local produce
97,zagorin apples imported
98,bajela bananas
99,bananas local produce
101,conference pears
102,pink lady apples


In [26]:
# посмотреть те, что заканчиваются на цифру, но не на нормальный размер
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\d\s*$', regex=True, na=False) &
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']]

,product_name_clean
67,agia skepi organic eggs size m x6
91,agroktima fragkou organic eggs x6
527,imported octopus τ6
528,imported octopus t4
530,imported praws 10/20
539,imported prawns 30/40
541,imported praws 06/08
570,meggle uht whole milk 3.5% fat 1lt 8585002504432
766,kri kri pepa x3
1320,imported whole octopus no. 4


In [27]:
repr(sklavenitis.loc[570, 'product_name_clean'])

"'meggle uht whole milk 3.5% fat 1lt 8585002504432'"

In [28]:
# убрать этот хвост кода в названии ЕЩЕ РАЗ

# \s — перед кодом должен быть пробел
# \d{4,} — потом 4 или больше цифр подряд
# \s*$ — и это всё должно быть в конце строки ($)
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s\d{4,}\s*$', '', regex=True)
)

In [29]:
# повторно посмотреть те, что заканчиваются на цифру, но не на нормальный размер и не на х и число и pack of и число
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\d\s*$', regex=True, na=False) &
    ~sklavenitis['product_name_clean'].str.contains(
        r'(?:'
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$'
        r'|'
        r'\b[x×]\s*\d+\s*$'
        r'|'
        r'\bpack\s+of\s+\d+\s*$'
        r'|'
        r'\bno\.?\s+\d+\s*$'
        r'|'
        r'\b(?:\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s*[x×]\s*\d+\s*$'
        r')',
        regex=True,
        na=False
    )
][['product_name_clean']]

,product_name_clean
527,imported octopus τ6
528,imported octopus t4
530,imported praws 10/20
539,imported prawns 30/40
541,imported praws 06/08
2547,babycare sensitive plus triple pack baby wipes (3 x 54 pcs). 162
3220,off beat birthday candles ob32
3221,off beat birthday candles ob32
3222,off beat birthday candles ob31
3274,sanitas easy pack dustbin bags with cord for large kitchen bins 52x75 cm 45 l 10


In [30]:
# чтобы убрать хвост для строки 2547, потому что она выглядит мутно в отличие от остальных
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\.\s*162\s*$', '', regex=True)
)

In [31]:
# опять схлопываем пробелы в конце, если они образовались 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.strip()
)

In [32]:
# посмотреть, на что заканчиваются строки 
sklavenitis['product_name_clean'].str[-1].value_counts()

product_name_clean
g    1831
l     742
s     284
t     177
)      76
e      58
r      46
d      26
m      15
k      11
n       8
%       8
2       8
y       7
6       5
o       5
3       4
c       4
h       3
.       3
p       3
0       3
4       3
8       2
-       2
b       2
i       1
a       1
ς       1
(       1
1       1
v       1
Name: count, dtype: int64

In [33]:
# посмотреть все странные варианты с символами
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-\)\(]\s*$', regex=True, na=False)
][['product_name_clean']]

,product_name_clean
196,beetroot leaves (bunch)
197,spearmint (bunch)
198,parsley (bunch)
199,spring onion (bunch)
200,spinach (bunch)
...,...
3121,dove purely pampering beauty bar – 4x90 g (shea butter & warm vanilla)
3122,eureka classic whitening powder – 4x60 g value pack (
3224,baking paper sanitas 18 sheets (42cmx38cm)
3283,adaptor 13a poka plug 2 round pin to uk type socket (ea1510)


In [34]:
# точечно убираем открывающуюся скобку 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\(\s*$', '', regex=True)
    .str.strip()
)

In [35]:
# снова посмотреть все странные варианты с символами
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-\)\(]\s*$', regex=True, na=False)
][['product_name_clean']]

,product_name_clean
196,beetroot leaves (bunch)
197,spearmint (bunch)
198,parsley (bunch)
199,spring onion (bunch)
200,spinach (bunch)
...,...
3120,septona antibacterial wet wipes multipack (4x15 pcs)
3121,dove purely pampering beauty bar – 4x90 g (shea butter & warm vanilla)
3224,baking paper sanitas 18 sheets (42cmx38cm)
3283,adaptor 13a poka plug 2 round pin to uk type socket (ea1510)


In [36]:
# посмотреть без скобок
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

,product_name_clean
243,alion crunchy salad 125g+30%
244,alion salad of the week 125g+30%
246,alion salad carrot & cabbage 250g+30%
247,alion cypriot salad 125g+30%
248,alion spinach salad 250g+30%
249,alion mix lettuces 125g+30%
496,paradisiotis chicken thighs boneless approx 700g.
498,frangou free range whole chicken 2.2kg.
845,charalambides christis kefalotyri cheese 250g -20%
990,margarine flora light 450g 100% plant based -


In [37]:
# убрать точку и тире в конце одиночные
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'[.\-]\s*$', '', regex=True)
    .str.strip()
)

In [38]:
# снова посмотреть без скобок
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

,product_name_clean
243,alion crunchy salad 125g+30%
244,alion salad of the week 125g+30%
246,alion salad carrot & cabbage 250g+30%
247,alion cypriot salad 125g+30%
248,alion spinach salad 250g+30%
249,alion mix lettuces 125g+30%
845,charalambides christis kefalotyri cheese 250g -20%
1546,tulip pork luncheon meat 3x200g -20%


In [39]:
# убираем -х%, чтобы не мешалась 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s*-\d+%\s*$', '', regex=True)
    .str.strip()
)

In [40]:
# снова посмотреть варианты с символами на конце
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

,product_name_clean
243,alion crunchy salad 125g+30%
244,alion salad of the week 125g+30%
246,alion salad carrot & cabbage 250g+30%
247,alion cypriot salad 125g+30%
248,alion spinach salad 250g+30%
249,alion mix lettuces 125g+30%


In [41]:
# ищем все варианты с % в составе 
# regex=False, потому что % тут мы ищем как обычный символ, не как шаблон
sklavenitis[
    sklavenitis['product_name_clean'].str.contains('%', regex=False, na=False)
][['product_name_clean']]

,product_name_clean
70,mevgal authentic greek yogurt 0% lemon 150g
85,vitam soft margarine 60% fat 200g
88,ryazhenka – 33 korovi 3.5% yoghurt drink 500 ml
243,alion crunchy salad 125g+30%
244,alion salad of the week 125g+30%
...,...
2119,johnnie walkerred label blended scotch whisky 40% 1lt
2391,ion 72% dark chocolate with almonds 100g
2546,🧻 baby care 99% water baby wipes – 216 pcs (3x72 wipes)
2834,greco drug specialties ethyl alcohol 70% antiseptic hand spray 500ml


In [42]:
# убираем в итоге варианты с +% на конце тоже 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s*[+-]\d+%\s*$', '', regex=True)
    .str.strip()
)

## 2. Извлечение size, pack_count и объёма

В этом блоке регулярками достаём вес, объём, количество упаковок и базовый нормализованный размер.


In [43]:
# достаем вес и количество упаковок
# учитывает такие типы записей:
# Вес / объём одной единицы
# 250g
# 300 g
# 270ml
# 1.2 l
# 1lt
# 1ltr
# 2.2kg
# 750+250ml     NEW
# 85gr     NEW

# Мультиупаковка
# 4x125g
# 4 x 125 g
# 9x250ml
# 3x200g
# 4*250ml     NEW
# 330ml x 6
# 3 sachets x 8g     NEW
# 90g x 3pcs      NEW

# Количество упаковок без веса
# x6
# x 6
# x15pcs     NEW
# pack of 2

# Штуки
# 4pcs
# 4 pcs
# 3 x 54 pcs

# БЫЛО БЕЗ НОВЫХ
# ______________________________________________________________________________________________________________
# # 1. Паттерны
# multipack_before_pattern = r'(\d+)\s*x\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
# multipack_after_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*(\d+)\b'
# single_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
# pack_only_x_pattern = r'\bx\s*(\d+)\b'
# pack_of_pattern = r'\bpack\s+of\s+(\d+)\b'

# # 2. Ищем разные форматы
# multi_before = sklavenitis['product_name_clean'].str.extract(multipack_before_pattern)
# multi_after = sklavenitis['product_name_clean'].str.extract(multipack_after_pattern)
# single = sklavenitis['product_name_clean'].str.extract(single_pattern)
# pack_only_x = sklavenitis['product_name_clean'].str.extract(pack_only_x_pattern)
# pack_of = sklavenitis['product_name_clean'].str.extract(pack_of_pattern)

# # 3. Размер как сырой текст
# # np.select как if elif else
# # np.select(
# #     [условие1, условие2, условие3],
# #     [значение1, значение2, значение3],
# #     default=что_поставить_если_ничего_не_подошло
# # )
# sklavenitis['size_raw'] = np.select(
#     [
#         multi_before[0].notna(),
#         multi_after[0].notna(),
#         single[0].notna()
#     ],
#     [
#         sklavenitis['product_name_clean'].str.extract(r'((?:\d+\s*x\s*)\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)')[0],
#         sklavenitis['product_name_clean'].str.extract(r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*\d+\b)')[0],
#         sklavenitis['product_name_clean'].str.extract(r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)')[0]
#     ],
#     default=np.nan
# )

# # 4. Количество упаковок
# sklavenitis['pack_count'] = np.select(
#     [
#         multi_before[0].notna(),
#         multi_after[0].notna(),
#         pack_only_x[0].notna(),
#         pack_of[0].notna()
#     ],
#     [
#         multi_before[0],
#         multi_after[2],
#         pack_only_x[0],
#         pack_of[0]
#     ],
#     default=1
# )

# # 5. Размер одной единицы
# sklavenitis['unit_size_value'] = np.select(
#     [
#         multi_before[1].notna(),
#         multi_after[0].notna(),
#         single[0].notna()
#     ],
#     [
#         multi_before[1],
#         multi_after[0],
#         single[0]
#     ],
#     default=np.nan
# )

# # единицы измерения
# sklavenitis['unit_size_unit'] = np.select(
#     [
#         multi_before[2].notna(),
#         multi_after[1].notna(),
#         single[1].notna()
#     ],
#     [
#         multi_before[2],
#         multi_after[1],
#         single[1]
#     ],
#     default=np.nan
# )

# # 6. Переводим в числа
# sklavenitis['pack_count'] = pd.to_numeric(sklavenitis['pack_count'], errors='coerce')
# sklavenitis['unit_size_value'] = pd.to_numeric(sklavenitis['unit_size_value'], errors='coerce')

# # 7. Общий размер до нормализации
# sklavenitis['size_total_raw'] = sklavenitis['pack_count'] * sklavenitis['unit_size_value']

# # 8. Нормализация в g / ml / pcs
# def normalize_size(row):
#     value = row['size_total_raw']
#     unit = row['unit_size_unit']
    
#     if pd.isna(value) or pd.isna(unit):
#         return pd.Series([np.nan, np.nan])
    
#     if unit == 'g':
#         return pd.Series([value, 'g'])
#     elif unit == 'kg':
#         return pd.Series([value * 1000, 'g'])
#     elif unit == 'ml':
#         return pd.Series([value, 'ml'])
#     elif unit in ['l', 'lt', 'ltr']:
#         return pd.Series([value * 1000, 'ml'])
#     elif unit == 'pcs':
#         return pd.Series([value, 'pcs'])
#     else:
#         return pd.Series([np.nan, np.nan])

# sklavenitis[['size_total_base', 'size_base_unit']] = sklavenitis.apply(normalize_size, axis=1)

# # 9. Красивый nullable int для количества упаковок
# # делает колонку pack_count целочисленной, но так, чтобы она умела хранить пропуски
# sklavenitis['pack_count'] = sklavenitis['pack_count'].astype('Int64')

# СТАЛО С НОВЫМИ
# ______________________________________________________________________________________________________________
# достаем вес и количество упаковок

# 1. Паттерны
# NEW: здесь уже учитываем короткую запись граммов через gr, например 85gr или 85 gr
multipack_before_pattern = r'(\d+)\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'




# CHANGED: добавили optional pcs после количества упаковок
# было: ...\s*x\s*(\d+)\b
# стало: ...\s*x\s*(\d+)\s*(?:pcs)?\b
multipack_after_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*(\d+)\s*(?:pcs)?\b'

single_pattern = r'(?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
pack_only_x_pattern = r'\b[x\u00d7]\s*(\d+)\s*(?:pcs)?\b'
pack_of_pattern = r'\bpack\s+of\s+(\d+)\b'

additive_pattern = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr)\b'

# 2. Ищем разные форматы
# NEW: single_pattern теперь тоже ловит короткую запись граммов через gr, например 85gr / 85 gr
multi_before = sklavenitis['product_name_clean'].str.extract(multipack_before_pattern)
multi_after = sklavenitis['product_name_clean'].str.extract(multipack_after_pattern)
single = sklavenitis['product_name_clean'].str.extract(single_pattern)
pack_only_x = sklavenitis['product_name_clean'].str.extract(pack_only_x_pattern)
pack_of = sklavenitis['product_name_clean'].str.extract(pack_of_pattern)

# NEW
additive = sklavenitis['product_name_clean'].str.extract(additive_pattern)

# 3. Размер как сырой текст
sklavenitis['size_raw'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        additive[0].notna(),   # NEW
        single[0].notna()
    ],
    [
        sklavenitis['product_name_clean'].str.extract(
            r'((?:\d+\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*)\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)'
        )[0],
        sklavenitis['product_name_clean'].str.extract(
            r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*\d+\s*(?:pcs)?\b)'
        )[0],
        # NEW
        sklavenitis['product_name_clean'].str.extract(
            r'(\d+(?:\.\d+)?\s*\+\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr)\b)'
        )[0],
        sklavenitis['product_name_clean'].str.extract(
            r'((?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)'
        )[0]
    ],
    default=np.nan
)

# 4. Количество упаковок
sklavenitis['pack_count'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        pack_only_x[0].notna(),
        pack_of[0].notna(),
        additive[0].notna()    # NEW
    ],
    [
        multi_before[0],
        multi_after[2],
        pack_only_x[0],
        pack_of[0],
        1                      # NEW: 750+250ml = одна упаковка
    ],
    default=1
)

# 5. Размер одной единицы
sklavenitis['unit_size_value'] = np.select(
    [
        multi_before[1].notna(),
        multi_after[0].notna(),
        additive[0].notna(),   # NEW
        single[0].notna()
    ],
    [
        multi_before[1],
        multi_after[0],
        # NEW: складываем две части 750 + 250
        pd.to_numeric(additive[0], errors='coerce') + pd.to_numeric(additive[1], errors='coerce'),
        single[0]
    ],
    default=np.nan
)

# единицы измерения
# NEW: gr сохраняем как отдельную найденную единицу, а ниже при нормализации приводим ее к g
sklavenitis['unit_size_unit'] = np.select(
    [
        multi_before[2].notna(),
        multi_after[1].notna(),
        additive[2].notna(),   # NEW
        single[1].notna()
    ],
    [
        multi_before[2],
        multi_after[1],
        additive[2],           # NEW
        single[1]
    ],
    default=np.nan
)

# 6. Переводим в числа
sklavenitis['pack_count'] = pd.to_numeric(sklavenitis['pack_count'], errors='coerce')
sklavenitis['unit_size_value'] = pd.to_numeric(sklavenitis['unit_size_value'], errors='coerce')

# 7. Общий размер до нормализации
# NEW: умножаем только явную мультиупаковку (4x250ml / 330ml x 6 / 3 sachets x 8g)
# NEW: если штуки стоят отдельно от общего веса, как в x15pcs 216g, total остается 216g, а не 15 * 216g
sklavenitis['size_total_raw'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        additive[0].notna(),
        single[0].notna()
    ],
    [
        sklavenitis['pack_count'] * sklavenitis['unit_size_value'],
        sklavenitis['pack_count'] * sklavenitis['unit_size_value'],
        sklavenitis['unit_size_value'],
        sklavenitis['unit_size_value']
    ],
    default=np.nan
)

# NEW: отдельная корректировка для случаев типа x15pcs 216g
# NEW: если количество штук написано отдельно, а потом идет общий вес, не умножаем total на pack_count
pack_only_with_total_size_mask = (
    sklavenitis['product_name_clean'].str.contains(pack_only_x_pattern, regex=True, na=False)
    & sklavenitis['product_name_clean'].str.contains(single_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(multipack_before_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(multipack_after_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(additive_pattern, regex=True, na=False)
)

sklavenitis.loc[pack_only_with_total_size_mask, 'size_total_raw'] = (
    sklavenitis.loc[pack_only_with_total_size_mask, 'unit_size_value']
)

# 8. Нормализация в g / ml / pcs
def normalize_size(row):
    value = row['size_total_raw']
    unit = row['unit_size_unit']
    
    if pd.isna(value) or pd.isna(unit):
        return pd.Series([np.nan, np.nan])
    
    if unit == 'g':
        return pd.Series([value, 'g'])
    # NEW: короткую запись граммов gr приводим к базовой единице g
    elif unit == 'gr':
        return pd.Series([value, 'g'])
    elif unit == 'kg':
        return pd.Series([value * 1000, 'g'])
    elif unit == 'ml':
        return pd.Series([value, 'ml'])
    elif unit in ['l', 'lt', 'ltr']:
        return pd.Series([value * 1000, 'ml'])
    elif unit == 'pcs':
        return pd.Series([value, 'pcs'])
    else:
        return pd.Series([np.nan, np.nan])

sklavenitis[['size_total_base', 'size_base_unit']] = sklavenitis.apply(normalize_size, axis=1)

# 9. Красивый nullable int для количества упаковок
sklavenitis['pack_count'] = sklavenitis['pack_count'].astype('Int64')

C:\Users\valeriya.brovchuk\AppData\Local\Temp\ipykernel_20892\891301541.py:293: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sklavenitis['product_name_clean'].str.contains(pack_only_x_pattern, regex=True, na=False)
C:\Users\valeriya.brovchuk\AppData\Local\Temp\ipykernel_20892\891301541.py:294: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & sklavenitis['product_name_clean'].str.contains(single_pattern, regex=True, na=False)
C:\Users\valeriya.brovchuk\AppData\Local\Temp\ipykernel_20892\891301541.py:295: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & ~sklavenitis['product_name_clean'].str.contains(multipack_before_pattern, regex=True, na=False)
C:\Users\valeriya.brovchuk\AppData\Local\Temp\ipykernel_20892\891301541.py:296: 

In [44]:
# удалю еще 2 колонки чтобы не мешались
sklavenitis = sklavenitis.drop(columns=['promo_price', 'promo_label'])
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,scraped_at,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480.0,g
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400.0,g
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400.0,g
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400.0,g
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400.0,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 374831,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250.0,ml
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 751207,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250.0,ml
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF50+ Children's & Infant Sunscreen Face & Body 200ml 770880,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body 200ml,200ml,1,200.0,ml,200.0,200.0,ml
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml,270ml,1,270.0,ml,270.0,270.0,ml


In [45]:
# чтобы колонки с числами другие тоже были integer и лучше читались 
# sklavenitis['unit_size_value'] = sklavenitis['unit_size_value'].round().astype('Int64') #не стоит раундить до целого, иначе будет неточным для 1.5 кг
# sklavenitis['size_total_raw'] = sklavenitis['size_total_raw'].round().astype('Int64')
sklavenitis['size_total_base'] = sklavenitis['size_total_base'].round().astype('Int64')

In [46]:
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,scraped_at,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480,g
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400,g
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400,g
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400,g
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 374831,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 751207,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF50+ Children's & Infant Sunscreen Face & Body 200ml 770880,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body 200ml,200ml,1,200.0,ml,200.0,200,ml
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml,270ml,1,270.0,ml,270.0,270,ml


## 3. Извлечение grade и сборка product_core

Здесь отдельно достаём grade, а потом убираем из названия служебные части, чтобы получить более чистый product_core.


In [47]:
# БЫЛО
# ____________________________________________________________________________________________
# # теперь чистим название, чтобы там не было лишних значений, веса и тд 
# # и еще параллельно выделяем отдельной колонкой grade (это было на креветках и осьминогах):
# # 10/20
# # 30/40
# # 06/08
# # t4
# # t6

# # 1. Вытащить grade / size range
# grade_pattern = r'\b(?:\d{1,2}/\d{1,2}|t\d+)\b'

# sklavenitis['grade'] = sklavenitis['product_name_clean'].str.extract(f'({grade_pattern})', expand=False)

# # 2. Создать колонку без размеров/объёмов/pack count/grade
# sklavenitis['product_core'] = sklavenitis['product_name_clean'].copy()

# # убрать multipack before: 4x125g / 4 x 125 g / 3 x 54 pcs
# # общий вид функции: df['col'].str.replace(что_ищем, на_что_заменяем, regex=...)
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+\s*x\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
#     '',
#     regex=True
# )

# # убрать multipack after: 330ml x 6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*\d+\b',
#     '',
#     regex=True
# )

# # убрать обычный размер/объём: 250g / 1lt / 2.2kg / 4pcs
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
#     '',
#     regex=True
# )

# # убрать pack count: x6 / x 6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\bx\s*\d+\b',
#     '',
#     regex=True
# )

# # убрать pack of 2
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\bpack\s+of\s+\d+\b',
#     '',
#     regex=True
# )

# # убрать grade: 10/20, 30/40, 06/08, t4, t6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     grade_pattern,
#     '',
#     regex=True
# )

# # подчистить пробелы
# sklavenitis['product_core'] = (
#     sklavenitis['product_core']
#     .str.replace(r'\s+', ' ', regex=True)
#     .str.strip()
# )


# СТАЛО
# ____________________________________________________________________________________________
# теперь чистим название, чтобы там не было лишних значений, веса и тд 
# и еще параллельно выделяем отдельной колонкой grade (это было на креветках и осьминогах):
# 10/20
# 30/40
# 06/08
# t4
# t6

# 1. Вытащить grade / size range
# percent / percent fat: 0%, 1.5%, 17% fat
# washes: 36 washes
# age / stage / range: 4+ months, 6-12 months, 12 years old
# child range: 4-10 years, 20-35kg
# NEW: range с граммами тоже учитываем в короткой записи через gr, например 85gr или 10-20gr
# dimensions / measures: 42cm x 38cm, 190 meters, 20m, 8 servings, 9volt
# size codes: 6/10, t4, size 2, a4, 5*
# комбинация sheets: 225 sheets, 2ply
# standalone 3ply пока специально не добавляем
grade_percent_pattern = r'\b\d+(?:[\.,]\d+)?%(?:\s*fat)?(?=\b|$)'
grade_washes_pattern = r'\b\d+\s*washes\b'
grade_age_pattern = r'\b(?:age\s*)?\d+\+\s*(?:months?|years?)\b|\b\d+\s*[-–—]\s*\d+\s*(?:months?|years?)\b|\b\d+\s*years?\s*old\b'
grade_measure_pattern = r'\b\d+(?:[\.,]\d+)?\s*(?:meters|m|servings|volt|min)\b'
grade_dimensions_pattern = r'\b\d+(?:[\.,]\d+)?\s*(?:cm|m)?\s*[x\u00d7*]\s*\d+(?:[\.,]\d+)?\s*(?:cm|m)?\b'
grade_range_pattern = r'\b\d+\s*[-–—]\s*\d+\s*(?:g|gr|kg)\b'
grade_slash_pattern = r'\b\d{1,2}/\d{1,2}\b'
grade_t_pattern = r'\bt\d+\b'
grade_size_pattern = r'\bsize\s*\d+\b'
grade_a_pattern = r'\ba\d+\b'
grade_star_pattern = r'\b\d+\*+(?=\s|$)'
grade_sheets_pattern = r'\b\d+\s*sheets(?:\s*,\s*\d+ply)?\b'
grade_intensity_pattern = r'\bintensity\s*\d+\b'
# NEW: отдельно перечисляем все варианты тире, чтобы regex одинаково видел их в названии
grade_dash_pattern = r'[-\u2013\u2014]'

# диапазоны для grade достаём по частям через str.extract
# так нам не важно, стоят ли между ними скобки, запятые или просто пробелы
grade_age_pattern = rf'\b(?:age\s*)?\d+\+\s*(?:months?|years?)\b|\b\d+\s*{grade_dash_pattern}\s*\d+\s*(?:months?|years?)\b|\b\d+\s*years?\s*old\b'
grade_range_pattern = rf'\b\d+\s*{grade_dash_pattern}\s*\d+\s*(?:g|gr|kg)\b'

# здесь лежат остальные устойчивые форматы grade, которые не нужно собирать из двух частей
grade_general_pattern = (
    f'(?:{grade_percent_pattern}|{grade_washes_pattern}|{grade_measure_pattern}|{grade_dimensions_pattern}|'
    f'{grade_slash_pattern}|{grade_t_pattern}|{grade_size_pattern}|{grade_a_pattern}|{grade_star_pattern}|'
    f'{grade_sheets_pattern}|{grade_intensity_pattern})'
)

# этот общий grade_pattern потом используем ниже, когда чистим product_core
grade_pattern = f'(?:{grade_age_pattern}|{grade_range_pattern}|{grade_general_pattern})'

# Шаг 1. Отдельно достаём age range и range с unit
grade_age = sklavenitis['product_name_clean'].str.extract(f'({grade_age_pattern})', expand=False)
grade_range = sklavenitis['product_name_clean'].str.extract(f'({grade_range_pattern})', expand=False)

# Шаг 2. Если нашли обе части, собираем их в один grade
grade_combined = pd.Series(pd.NA, index=sklavenitis.index, dtype='object')
grade_both_mask = grade_age.notna() & grade_range.notna()
grade_combined.loc[grade_both_mask] = grade_age.loc[grade_both_mask].str.strip() + ', ' + grade_range.loc[grade_both_mask].str.strip()
grade_combined = grade_combined.combine_first(grade_age)
grade_combined = grade_combined.combine_first(grade_range)

# Шаг 3. Потом достаём остальные варианты grade
grade_general = sklavenitis['product_name_clean'].str.extract(f'({grade_general_pattern})', expand=False)

# Шаг 4. Сначала берём составной диапазон, а если его нет — остальные форматы
sklavenitis['grade'] = grade_combined.combine_first(grade_general)

# Дальше собираем product_core: берём product_name_clean и постепенно убираем из него служебные части
sklavenitis['product_core'] = sklavenitis['product_name_clean'].copy()

# Сначала убираем multipack вида 4x125g или 3 x 54 pcs
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
    '',
    regex=True
)

# Потом убираем запись, где size стоит перед количеством
# было: ...\s*x\s*\d+\b
# стало: ...\s*x\s*\d+\s*(?:pcs)?\b
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*\d+\s*(?:pcs)?\b',
    '',
    regex=True
)

# NEW: отдельно убираем additive size, например 750+250ml
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+(?:\.\d+)?\s*\+\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr)\b',
    '',
    regex=True
)

# Потом убираем обычный size или объём
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'(?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
    '',
    regex=True
)

# Потом убираем отдельный pack count
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b[x\u00d7]\s*\d+\s*(?:pcs)?\b',
    '',
    regex=True
)

# NEW: если число уже удалилось, убираем оставшийся pcs
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bpcs\b',
    '',
    regex=True
)

# Потом убираем конструкции вида pack of 2
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bpack\s+of\s+\d+\b',
    '',
    regex=True
)

# После этого убираем из product_core всё, что уже записали в grade
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    grade_pattern,
    '',
    regex=True
)

# NEW: если осталось одинокое approx, тоже убираем его
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bapprox\.?\b',
    '',
    regex=True
)

# В самом конце подчищаем хвосты после всех удалений
# NEW: сначала убираем пустые и одинокие скобки
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\(\s*\)|\s*\)\s*$|\(\s*$',
    '',
    regex=True
)

# NEW: потом убираем хвостовые тире
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\s*[-\u2013\u2014]\s*$',
    '',
    regex=True
)

# NEW: и ещё раз проверяем скобки после тире
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\(\s*\)|\s*\)\s*$|\(\s*$',
    '',
    regex=True
)

sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [48]:
sklavenitis

,store,store_location,category_path,product_name,brand,price,price_unit,scraped_at,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit,grade,product_core
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480,g,NaN,spaghetti with tomato sauce sklavenitis production
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400,g,NaN,macedonian halva with almonds vegan gluten-free
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,makedonikos halva with cocoa – vegan. gluten free
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,olympos halva with pistachios – gluten free
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,makedonikos halva with peanuts – gluten free
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 374831,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml,NaN,nivea sun kids protect & care 5 in 1colour spray spf
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 751207,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml,NaN,nivea sun kids protect & care 5 in 1colour spray spf
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF50+ Children's & Infant Sunscreen Face & Body 200ml 770880,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body 200ml,200ml,1,200.0,ml,200.0,200,ml,NaN,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml,270ml,1,270.0,ml,270.0,270,ml,NaN,nivea sun kids trigger spray spf 50


In [49]:
# посмотреть как выглядят строки с кг теперь 
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\b\d+(?:\.\d+)?\s*kg\b', regex=True, na=False)
][['product_name_clean', 'pack_count', 'unit_size_value', 'unit_size_unit', 'size_total_base', 'size_base_unit']]

,product_name_clean,pack_count,unit_size_value,unit_size_unit,size_total_base,size_base_unit
53,triumph plain flour 1kg,1,1.0,kg,1000,g
61,triumph village flour 1kg,1,1.0,kg,1000,g
96,zagorin apples imported pre-packed 2kg,1,2.0,kg,2000,g
295,katsellis club sandwich wheat bread large slices sliced 1kg,1,1.0,kg,1000,g
482,nefeli whole chicken 2.5kg,1,2.5,kg,2500,g
...,...,...,...,...,...,...
3198,whiskas dry cat food 1+ years chicken 3.8kg,1,3.8,kg,3800,g
3199,brekkies almon tuna vegetables & cereals dry food 1.5kg,1,1.5,kg,1500,g
3218,kaouris mixture for canaries 1kg,1,1.0,kg,1000,g
3219,kaouris mixture for parrots 1kg,1,1.0,kg,1000,g


## 4. Подготовка выборки для ручной разметки

Этот блок нужен для создания файлов, которые потом размечаются вручную и используются для словарей.


In [50]:
# для разметки в будущем под NLP мини-часть датасета
label_df = sklavenitis[['product_core']].sample(30, random_state=42).copy()
# label_df.to_excel(r'C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\label_df.xlsx', index=False)

### Правила разметки:

### Правило 1. brand
Только сам бренд, без типа товара.

Если:
macedonikos halva
то brand = macedonikos, а не macedonikos halva

### Правило 2. product_type
Главный тип товара.

Если:
halva with almonds
то product_type = halva

### Правило 3. variant
То, что отличает товар внутри типа, но не является общим атрибутом.

Например:
- cocoa
- almonds
- banana

### Правило 4. attributes
Общие свойства.

Например:
- organic
- vegan
- gluten free
- smoked

### Правило 5. grade

Размерный класс / диапазон:
- 10/20
- t6

### Правило 6. model_code

Технический код товара:
- ob32
- dl2016b2

In [51]:
# для разметки в будущем под NLP мини-часть датасета С ИНДЕКСАМИ 
label_df_with_index = sklavenitis[['product_core']].sample(100, random_state=42).copy()
# label_df_with_index.to_excel(r'C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\label_df_with_index.xlsx', index=True)

In [52]:
sklavenitis_raw = pd.read_csv(base_path / "sklavenitis-ypsonas.csv", encoding="utf-8")

In [53]:
label_df_with_index = label_df_with_index.join(sklavenitis_raw[['product_name', 'image_url', 'category_path']], how='left') 

In [54]:
# label_df_with_index.to_excel(r'C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\label_df_with_index.xlsx', index=True)

In [55]:
# посмотреть примеры с пустыми скобками в конце
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\s+\(\)\s*$', regex=True, na=False)
][['product_core']]

# посмотреть примеры с точкой в конце
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\.\s*$', regex=True, na=False)
][['product_core']]

# убрать пустые скобки в конце и точки
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
    .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
    .str.strip()
)

In [56]:
# видим, что после strip появились снова, убираем по кругу 
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
    .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
    .str.strip()
)

In [57]:
# посмотреть в конце дефисы лишние
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\s+[-–—]\s*$', regex=True, na=False)
][['product_core']]

,product_core
2548,proderm liquid laundry detergent for baby clothes –
2549,eureka massalias liquid detergent for baby clothes –


In [58]:
# удалить в конце разного вида дефисы
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+[-–—]\s*$', '', regex=True)
    .str.strip()
)

In [59]:
# # импортировать обновленный датасет с 100 размеченными вариантами обратно
# label_updated = pd.read_excel(
#     r'C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\label_df_with_index.xlsx',
#     index_col=0
# )

In [60]:
# label_updated 

In [61]:
# # посмотреть примеры с пустыми скобками в конце
# label_updated[
#     label_updated['product_core'].str.contains(r'\s+\(\)\s*$', regex=True, na=False)
# ][['product_core']]

# # посмотреть примеры с точкой в конце
# label_updated[
#     label_updated['product_core'].str.contains(r'\.\s*$', regex=True, na=False)
# ][['product_core']]

# # убрать пустые скобки в конце и точки
# label_updated['product_core'] = (
#     label_updated['product_core']
#     .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
#     .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
#     .str.strip()
# )

In [62]:
# # посмотреть в конце дефисы лишние
# label_updated[
#     label_updated['product_core'].str.contains(r'\s+[-–—]\s*$', regex=True, na=False)
# ][['product_core']]

# # удалить в конце разного вида дефисы
# label_updated['product_core'] = (
#     label_updated['product_core']
#     .str.replace(r'\s+[-–—]\s*$', '', regex=True)
#     .str.strip()
# )

In [63]:
sklavenitis_for_annotation_100_500=sklavenitis.join(sklavenitis_raw[['image_url']], how='left') 

In [64]:
sklavenitis_for_annotation_100_500

,store,store_location,category_path,product_name,brand,price,price_unit,scraped_at,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit,grade,product_core,image_url
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,2026-03-13T22:07:58.656Z,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480,g,NaN,spaghetti with tomato sauce sklavenitis production,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/f45d06fa-6c44-11f0-a339-a27005a1ab0f_22174529.jpg
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,2026-03-13T22:07:58.656Z,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400,g,NaN,macedonian halva with almonds vegan gluten-free,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,makedonikos halva with cocoa – vegan. gluten free,https://imageproxy.wolt.com/assets/697364789b938fe622d8d157
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,2026-03-13T22:07:58.656Z,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,olympos halva with pistachios – gluten free,https://imageproxy.wolt.com/assets/697364769b938fe622d8d14a
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,2026-03-13T22:07:58.656Z,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400,g,NaN,makedonikos halva with peanuts – gluten free,https://imageproxy.wolt.com/assets/6973718b4af2fcbd36bd6cc1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3337,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 374831,NIVEA SUN,20.10 EUR,10.05 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml,NaN,nivea sun kids protect & care 5 in 1colour spray spf,https://wolt-menu-images-cdn.wolt.com/menu-images/682304f3e5808953754787ab/49f8858c-3a20-11f0-a479-8e89d0a1d977_4005808286867.jpg
3338,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Protect & Care 5 in 1Colour Spray SPF 50+ 200ml 751207,NIVEA SUN,15.68 EUR,7.84 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids protect & care 5 in 1colour spray spf 50+ 200ml,50+ 200ml,1,250.0,ml,250.0,250,ml,NaN,nivea sun kids protect & care 5 in 1colour spray spf,https://wolt-menu-images-cdn.wolt.com/menu-images/682304f3e5808953754787ab/37e3d4fa-3a25-11f0-82d8-46297e107e25_4005900915559.jpg
3339,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA Sun Babies & Kids Sensitive Protect SPF50+ Children's & Infant Sunscreen Face & Body 200ml 770880,NIVEA Sun,15.26 EUR,7.63 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body 200ml,200ml,1,200.0,ml,200.0,200,ml,NaN,nivea sun babies & kids sensitive protect spf50+ children's & infant sunscreen face & body,https://wolt-menu-images-cdn.wolt.com/menu-images/682304f3e5808953754787ab/4f0a864c-3a25-11f0-b3d6-02f9e0079d00_4005900915580.jpg
3340,sklavenitis-ypsonas,Limassol,Sun Care > Kids Sun Care,NIVEA SUN Kids Trigger Spray SPF 50 270ml 770881,NIVEA SUN,21.30 EUR,7.89 EUR/millilitre,2026-03-13T22:15:24.030Z,nivea sun kids trigger spray spf 50 270ml,270ml,1,270.0,ml,270.0,270,ml,NaN,nivea sun kids trigger spray 

In [65]:
sklavenitis_for_annotation_100_500.loc[2536]

store                                                            sklavenitis-ypsonas
store_location                                                              Limassol
category_path                                     Child & Infant Care > Baby Diapers
product_name          SAFE NIGHTS Kids Pants Boy 4–10 Years (20–35kg) – 14pcs 847625
brand                                                                    SAFE NIGHTS
price                                                                       7.86 EUR
price_unit                                                            0.56 EUR/piece
scraped_at                                                  2026-03-13T22:13:29.276Z
product_name_clean           safe nights kids pants boy 4–10 years (20–35kg) – 14pcs
size_raw                                                                       14pcs
pack_count                                                                         1
unit_size_value                                                  

In [66]:
sklavenitis_for_annotation_100_500.loc[2405]

store                                                          sklavenitis-ypsonas
store_location                                                            Limassol
category_path         Cookies, Chocolates & Sweets > Chocolates-Chewing Gums-Candy
product_name                                     Smarties Minis X15pcs 216g 309322
brand                                                               Smarties Minis
price                                                                     4.20 EUR
price_unit                                                                     NaN
scraped_at                                                2026-03-13T22:13:08.817Z
product_name_clean                                      smarties minis x15pcs 216g
size_raw                                                                      216g
pack_count                                                                      15
unit_size_value                                                              216.0
unit

In [67]:
# строки, где в product_core остался отдельный pcs
sklavenitis_for_annotation_100_500.loc[905, ['product_name', 'product_name_clean', 'product_core']]

product_name          La Vache Qui X8 Pcs 128g 768752
product_name_clean           la vache qui x8 pcs 128g
product_core                             la vache qui
Name: 905, dtype: object

In [68]:
# строки, где в product_core остался approx или approx.
approx_rows = sklavenitis_for_annotation_100_500[
    sklavenitis_for_annotation_100_500['product_core'].str.contains(r'\bapprox\.?\b', regex=True, na=False)
][['product_name', 'product_name_clean', 'product_core', 'size_raw', 'pack_count', 'grade']]

approx_rows

,product_name,product_name_clean,product_core,size_raw,pack_count,grade


In [69]:
sklavenitis_for_annotation_100_500.loc[2536]

store                                                            sklavenitis-ypsonas
store_location                                                              Limassol
category_path                                     Child & Infant Care > Baby Diapers
product_name          SAFE NIGHTS Kids Pants Boy 4–10 Years (20–35kg) – 14pcs 847625
brand                                                                    SAFE NIGHTS
price                                                                       7.86 EUR
price_unit                                                            0.56 EUR/piece
scraped_at                                                  2026-03-13T22:13:29.276Z
product_name_clean           safe nights kids pants boy 4–10 years (20–35kg) – 14pcs
size_raw                                                                       14pcs
pack_count                                                                         1
unit_size_value                                                  

## 5. Проверка полного обработанного датасета

Здесь начинаем смотреть на полный sklavenitis, уже обработанный после regex-очистки.


In [70]:
# Это финальный очищенный файл по полному датасету Sklavenitis.
# В нем уже собран конечный очищенный product_core и вынесены то, что удалось стабильно достать в отдельные колонки.
# Дальше в ноутбуке ориентируемся именно на этот файл как на основной источник правды по очищенному product_core.
processed_sklavenitis=pd.read_csv("output/sklavenitis_ypsonas_processed.csv")
processed_sklavenitis.head()

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,...,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit,grade,product_core
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,NaN,NaN,unknown,...,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480.0,g,NaN,spaghetti with tomato sauce sklavenitis production
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,NaN,NaN,unknown,...,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,macedonian halva with almonds vegan gluten-free
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,makedonikos halva with cocoa – vegan. gluten free
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,NaN,NaN,unknown,...,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,olympos halva with pistachios – gluten free
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,makedonikos halva with peanuts – gluten free


In [71]:
processed_sklavenitis.loc[2536]

store                                                            sklavenitis-ypsonas
store_location                                                              Limassol
category_path                                     Child & Infant Care > Baby Diapers
product_name          SAFE NIGHTS Kids Pants Boy 4–10 Years (20–35kg) – 14pcs 847625
brand                                                                    SAFE NIGHTS
price                                                                       7.86 EUR
price_unit                                                            0.56 EUR/piece
promo_price                                                                      NaN
promo_label                                                                      NaN
availability                                                                 unknown
image_url                https://imageproxy.wolt.com/assets/690352341c0a709b4414e6fc
product_url               https://wolt.com/el/cyp/limassol/venue/

In [72]:
processed_sklavenitis.head(50)

,store,store_location,category_path,product_name,brand,price,price_unit,promo_price,promo_label,availability,...,product_name_clean,size_raw,pack_count,unit_size_value,unit_size_unit,size_total_raw,size_total_base,size_base_unit,grade,product_core
0,sklavenitis-ypsonas,Limassol,Lenten Food,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Spaghetti,2.90 EUR,6.04 EUR/kilogram,NaN,NaN,unknown,...,spaghetti with tomato sauce sklavenitis production 480g,480g,1,480.0,g,480.0,480.0,g,NaN,spaghetti with tomato sauce sklavenitis production
1,sklavenitis-ypsonas,Limassol,Lenten Food,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,MACEDONIAN Halva,5.75 EUR,14.38 EUR/kilogram,NaN,NaN,unknown,...,macedonian halva with almonds vegan gluten-free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,macedonian halva with almonds vegan gluten-free
2,sklavenitis-ypsonas,Limassol,Lenten Food,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with cocoa – vegan. gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,makedonikos halva with cocoa – vegan. gluten free
3,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Pistachios – Gluten Free 400g 634763,OLYMPOS Halva,5.78 EUR,14.45 EUR/kilogram,NaN,NaN,unknown,...,olympos halva with pistachios – gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,olympos halva with pistachios – gluten free
4,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Peanuts – Gluten Free 400g 119460,MAKEDONIKOS Halva,5.28 EUR,13.20 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with peanuts – gluten free 400g,400g,1,400.0,g,400.0,400.0,g,NaN,makedonikos halva with peanuts – gluten free
5,sklavenitis-ypsonas,Limassol,Lenten Food,OLYMPOS Halva with Dark Chocolate & Almonds – Gluten Free 250g 677094,OLYMPOS Halva,4.10 EUR,16.40 EUR/kilogram,NaN,NaN,unknown,...,olympos halva with dark chocolate & almonds – gluten free 250g,250g,1,250.0,g,250.0,250.0,g,NaN,olympos halva with dark chocolate & almonds – gluten free
6,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Chocolate – Gluten Free 40g 767518,MAKEDONIKOS Halva,0.98 EUR,24.50 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with chocolate – gluten free 40g,40g,1,40.0,g,40.0,40.0,g,NaN,makedonikos halva with chocolate – gluten free
7,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva Espresso – Gluten Free 40g 767517,MAKEDONIKOS Halva,0.98 EUR,24.50 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva espresso – gluten free 40g,40g,1,40.0,g,40.0,40.0,g,NaN,makedonikos halva espresso – gluten free
8,sklavenitis-ypsonas,Limassol,Lenten Food,MAKEDONIKOS Halva with Vanilla Flavour – Gluten Free 40g 472644,MAKEDONIKOS Halva,0.98 EUR,24.50 EUR/kilogram,NaN,NaN,unknown,...,makedonikos halva with vanilla flavour – gluten free 40g,40g,1,40.0,g,40.0,40.0,g,NaN,makedonikos halva with vanilla flavour – gluten free
9,sklavenitis-ypsonas,Limassol,Lenten Food,Orzo with Shrimp SKLAVENITIS ProductION 400g 795271,Orzo,5.70 EUR,14.25 EUR/kilogram,NaN,NaN,unknown,...,orzo with shrimp sklavenitis production 400g,400g,1,400.0,g,400.0,400.0,g,NaN,orzo with shrimp sklavenitis production


## 6. Базовый словарный проход по brand и product_type

Здесь возвращаем тот базовый шаг, на котором мы уже остановились раньше:
- собираем словари из reviewed-разметки
- ищем `brand` по всему `product_core`, не только в начале
- ищем `product_type` в тексте после удаления найденного бренда
- считаем coverage

Здесь **нет** дополнительной логики с `match_mode`, `ambiguous` и спец-правилами для отдельных брендов.
Это просто базовый словарный проход, который мы пока фиксируем как есть.


## 7. Импорт и пути

Здесь подключаем библиотеки и задаём пути к входным и выходным файлам.


In [73]:
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

# Финальный reviewed-файл для словарей и coverage.
# Это canonical-версия reviewed-разметки, уже привязанная к конечному очищенному product_core.
REVIEWED_PATH = OUTPUT_DIR / 'label_df_random42_labeled_500_reviewed_v5_canonical.csv'

# Финальный очищенный файл по полному датасету Sklavenitis.
# Именно он считается главным источником правды для product_core после очистки.
PROCESSED_PATH = OUTPUT_DIR / 'sklavenitis_ypsonas_processed.csv'

BRAND_VOCAB_PATH = OUTPUT_DIR / 'brand_vocab.csv'
PRODUCT_TYPE_VOCAB_PATH = OUTPUT_DIR / 'product_type_vocab.csv'
FIRST_PASS_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_first_pass.csv'
COVERAGE_PATH = OUTPUT_DIR / 'brand_product_type_coverage_summary.csv'
UNKNOWN_BRANDS_PATH = OUTPUT_DIR / 'unknown_brand_product_core_top.csv'
UNKNOWN_PRODUCT_TYPES_PATH = OUTPUT_DIR / 'unknown_product_type_product_core_top.csv'
BRAND_NOT_FIRST_WORD_PATH = OUTPUT_DIR / 'brand_match_not_first_word_examples.csv'

PRODUCT_TYPE_NON_LITERAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_cases.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.csv'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.xlsx'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.xlsx'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWED_PATH, PROCESSED_PATH


(WindowsPath('C:/Users/valeriya.brovchuk/Documents/projects/price_comparison_grocery/output/label_df_random42_labeled_500_reviewed_v5_canonical.csv'),
 WindowsPath('C:/Users/valeriya.brovchuk/Documents/projects/price_comparison_grocery/output/sklavenitis_ypsonas_processed.csv'))

## 8. Загружаем данные

Здесь используем уже финальные файлы, а не старые промежуточные версии.

- `sklavenitis_ypsonas_processed.csv` — это конечный очищенный файл по полному датасету Sklavenitis.
  В нем уже собран очищенный `product_core` и вынесены устойчиво найденные `size`, `grade` и другие служебные поля.
- `label_df_random42_labeled_500_reviewed_v5_canonical.csv` — это reviewed-разметка на 500 строк,
  уже джойнутая к финальному очищенному файлу, чтобы словари и coverage считались от той же версии `product_core`.


In [74]:
# Загружаем уже канонические файлы: reviewed после join к финальному processed и сам полный cleaned processed.
reviewed = pd.read_csv(REVIEWED_PATH)
processed = pd.read_csv(PROCESSED_PATH)

print('reviewed rows:', len(reviewed))
print('processed rows:', len(processed))

reviewed[['product_core', 'brand', 'product_type', 'variant', 'attributes', 'grade', 'pieces']].head()


reviewed rows: 500
processed rows: 3342


,product_core,brand,product_type,variant,attributes,grade,pieces
0,pork neck steaks,papantoniou,pork neck steaks,NaN,NaN,NaN,NaN
1,ainos potatoes classic,ainos,potatoes,classic,NaN,NaN,NaN
2,actimel dessert yogurt multifruit,actimel,yogurt,multifruit,dessert,NaN,NaN
3,eureka newborn baby clothes liquid detergent,eureka,liquid detergent,newborn baby clothes,NaN,36 washes,NaN
4,carefree cotton aloe sanitary napkins,carefree,sanitary napkins,aloe,cotton,NaN,NaN


## 9. Нормализация текста

Приводим текст к более единообразному виду: убираем лишние пробелы,
делаем lowercase и заменяем пропуски на пустую строку.


In [75]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


reviewed = reviewed.copy()
processed = processed.copy()

reviewed['product_core'] = reviewed['product_core'].fillna('')
reviewed['brand'] = reviewed['brand'].map(normalize_text)
reviewed['product_type'] = reviewed['product_type'].map(normalize_text)
reviewed['product_core_norm'] = reviewed['product_core'].map(normalize_text)

processed['product_core'] = processed['product_core'].fillna('')
processed['product_core_norm'] = processed['product_core'].map(normalize_text)

reviewed[['product_core', 'product_core_norm', 'brand', 'product_type']].head()


,product_core,product_core_norm,brand,product_type
0,pork neck steaks,pork neck steaks,papantoniou,pork neck steaks
1,ainos potatoes classic,ainos potatoes classic,ainos,potatoes
2,actimel dessert yogurt multifruit,actimel dessert yogurt multifruit,actimel,yogurt
3,eureka newborn baby clothes liquid detergent,eureka newborn baby clothes liquid detergent,eureka,liquid detergent
4,carefree cotton aloe sanitary napkins,carefree cotton aloe sanitary napkins,carefree,sanitary napkins


## 10. Собираем словари brand и product_type

Это обзорные словари из reviewed-разметки:
- само значение
- сколько раз оно встретилось
- один пример `product_core`


In [76]:
def build_vocab_table(df, column):
    non_empty = df.loc[df[column].ne(''), ['product_core', column]].copy()
    summary = (
        non_empty.groupby(column, as_index=False)
        .agg(
            count=('product_core', 'size'),
            example_product_core=('product_core', 'first')
        )
        .sort_values(['count', column], ascending=[False, True], ignore_index=True)
    )
    return summary


brand_vocab = build_vocab_table(reviewed, 'brand')
product_type_vocab = build_vocab_table(reviewed, 'product_type')

print('distinct brands:', len(brand_vocab))
print('distinct product_types:', len(product_type_vocab))
brand_vocab.head(10)


distinct brands: 289
distinct product_types: 280


,brand,count,example_product_core
0,marata,10,marata rigatoni
1,bonora,8,bonora corn balls with pistachio
2,sklavenitis,7,sklavenitis black currunts
3,alion,6,alion arugula salad
4,fresh land,6,fresh land caramel pineapple cheesecake
5,l'oreal paris,6,l’oreal paris men expert hydra energetic extreme sport shower gel – cooling effect
6,grigoriou,5,grigoriou pizza special
7,pedigree,5,pedigree vital adult dry dog food with veal & vegetables
8,sette elements,5,sette elements absorbent cotton wool
9,charalambides christis,4,charalambides christis staggato yogurt


In [77]:
product_type_vocab.head(10)


,product_type,count,example_product_core
0,cheese,24,bliss point – aged graviera with thyme
1,bread,17,katselis bread for toast wheat
2,yogurt,12,actimel dessert yogurt multifruit
3,salad,10,chef salad
4,baby food,9,nestle nan optipro baby food in powder
5,beer,8,erdinger weiss beer
6,coffee,8,douwe egberts instant espresso coffee
7,potato chips,8,lays wholegrain snacks salt
8,water,8,zaros natural mineral water
9,cat food,7,whiskas salmon


## 11. Первый автоматический проход по полному sklavenitis

Базовая логика:
- `brand` ищем по всей строке, не только в начале
- многословные бренды тоже ищем
- после найденного бренда убираем его из текста один раз
- `product_type` ищем в оставшемся тексте
- если `product_type` состоит из нескольких слов, разрешаем небольшие вставки между ними


In [78]:
def build_phrase_pattern(term):
    escaped = re.escape(term)
    return re.compile(rf'(?<![a-z0-9]){escaped}(?![a-z0-9])')


def build_flexible_multiword_pattern(term, max_gap_words=3):
    words = term.split()
    if len(words) <= 1:
        return build_phrase_pattern(term)

    gap = rf"(?:[\s\-/,()]+[a-z0-9%'.&+]+){{0,{max_gap_words}}}[\s\-/,()]+"
    pattern = rf'(?<![a-z0-9]){re.escape(words[0])}'

    for word in words[1:]:
        pattern += gap + re.escape(word)

    pattern += r'(?![a-z0-9])'
    return re.compile(pattern)


def match_brand_from_vocab(product_core_norm, brand_terms):
    if not product_core_norm:
        return '', None

    best_brand = ''
    best_start = None

    for brand in brand_terms:
        match = build_phrase_pattern(brand).search(product_core_norm)
        if not match:
            continue

        if not best_brand:
            best_brand = brand
            best_start = match.start()
            continue

        is_longer = len(brand) > len(best_brand)
        same_length_earlier = (
            len(brand) == len(best_brand)
            and best_start is not None
            and match.start() < best_start
        )

        if is_longer or same_length_earlier:
            best_brand = brand
            best_start = match.start()

    return best_brand, best_start


def remove_brand_once(product_core_norm, brand):
    if not product_core_norm or not brand:
        return product_core_norm
    cleaned = build_phrase_pattern(brand).sub(' ', product_core_norm, count=1)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip(' -')


def match_product_type_from_vocab(core_without_brand, product_type_terms):
    if not core_without_brand:
        return '', ''

    best_match = ''
    best_start = None
    best_method = ''

    for product_type in product_type_terms:
        exact_match = build_phrase_pattern(product_type).search(core_without_brand)
        flexible_match = None

        if len(product_type.split()) > 1:
            flexible_match = build_flexible_multiword_pattern(product_type).search(core_without_brand)

        match = exact_match or flexible_match
        if not match:
            continue

        current_method = (
            'product_type_vocab_phrase'
            if exact_match is not None
            else 'product_type_vocab_flexible'
        )

        if not best_match:
            best_match = product_type
            best_start = match.start()
            best_method = current_method
            continue

        is_longer = len(product_type) > len(best_match)
        same_length_earlier = (
            len(product_type) == len(best_match)
            and best_start is not None
            and match.start() < best_start
        )

        if is_longer or same_length_earlier:
            best_match = product_type
            best_start = match.start()
            best_method = current_method

    return best_match, best_method


brand_terms = brand_vocab['brand'].tolist()
product_type_terms = product_type_vocab['product_type'].tolist()

brand_predictions = []
brand_methods = []
brand_match_starts = []
core_without_brand_values = []
product_type_predictions = []
product_type_methods = []

for row in processed.itertuples(index=False):
    product_core_norm = row.product_core_norm

    brand_pred, brand_start = match_brand_from_vocab(product_core_norm, brand_terms)
    if brand_pred:
        brand_method = (
            'brand_vocab_first_word'
            if brand_start == 0
            else 'brand_vocab_not_first_word'
        )
    else:
        brand_method = ''

    core_without_brand = remove_brand_once(product_core_norm, brand_pred)
    product_type_pred, product_type_method = match_product_type_from_vocab(
        core_without_brand, product_type_terms
    )

    brand_predictions.append(brand_pred)
    brand_methods.append(brand_method)
    brand_match_starts.append(brand_start)
    core_without_brand_values.append(core_without_brand)
    product_type_predictions.append(product_type_pred)
    product_type_methods.append(product_type_method)

processed['brand_pred'] = brand_predictions
processed['brand_match_method'] = brand_methods
processed['brand_match_start'] = brand_match_starts
processed['product_core_without_brand'] = core_without_brand_values
processed['product_type_pred'] = product_type_predictions
processed['product_type_match_method'] = product_type_methods

processed[[
    'product_core', 'brand_pred', 'brand_match_method', 'brand_match_start',
    'product_core_without_brand', 'product_type_pred', 'product_type_match_method'
]].head(15)


,product_core,brand_pred,brand_match_method,brand_match_start,product_core_without_brand,product_type_pred,product_type_match_method
0,spaghetti with tomato sauce sklavenitis production,sklavenitis,brand_vocab_not_first_word,28.0,spaghetti with tomato sauce production,sauce,product_type_vocab_phrase
1,macedonian halva with almonds vegan gluten-free,,,NaN,macedonian halva with almonds vegan gluten-free,halva,product_type_vocab_phrase
2,makedonikos halva with cocoa – vegan. gluten free,,,NaN,makedonikos halva with cocoa – vegan. gluten free,halva,product_type_vocab_phrase
3,olympos halva with pistachios – gluten free,olympos,brand_vocab_first_word,0.0,halva with pistachios – gluten free,halva,product_type_vocab_phrase
4,makedonikos halva with peanuts – gluten free,,,NaN,makedonikos halva with peanuts – gluten free,peanuts,product_type_vocab_phrase
5,olympos halva with dark chocolate & almonds – gluten free,olympos,brand_vocab_first_word,0.0,halva with dark chocolate & almonds – gluten free,chocolate,product_type_vocab_phrase
6,makedonikos halva with chocolate – gluten free,,,NaN,makedonikos halva with chocolate – gluten free,chocolate,product_type_vocab_phrase
7,makedonikos halva espresso – gluten free,,,NaN,makedonikos halva espresso – gluten free,halva,product_type_vocab_phrase
8,makedonikos halva with vanilla flavour – gluten free,,,NaN,makedonikos halva with vanilla flavour – gluten free,halva,product_type_vocab_phrase
9,orzo with shrimp sklavenitis production,sklavenitis,brand_vocab_not_first_word,17.0,orzo with shrimp production,,


## 12. Coverage

Считаем, сколько строк удалось покрыть на базовой логике.


In [79]:
coverage = pd.DataFrame([
    {
        'field': 'brand',
        'total_rows': len(processed),
        'matched_rows': int(processed['brand_pred'].ne('').sum()),
        'matched_share': round(processed['brand_pred'].ne('').mean(), 4),
        'vocab_match': int(processed['brand_pred'].ne('').sum()),
        'brand_match_first_word': int((processed['brand_match_method'] == 'brand_vocab_first_word').sum()),
        'brand_match_not_first_word': int((processed['brand_match_method'] == 'brand_vocab_not_first_word').sum()),
        'unknown_rows': int(processed['brand_pred'].eq('').sum())
    },
    {
        'field': 'product_type',
        'total_rows': len(processed),
        'matched_rows': int(processed['product_type_pred'].ne('').sum()),
        'matched_share': round(processed['product_type_pred'].ne('').mean(), 4),
        'vocab_match': int(processed['product_type_pred'].ne('').sum()),
        'flexible_multiword_match': int((processed['product_type_match_method'] == 'product_type_vocab_flexible').sum()),
        'unknown_rows': int(processed['product_type_pred'].eq('').sum())
    }
])

coverage


,field,total_rows,matched_rows,matched_share,vocab_match,brand_match_first_word,brand_match_not_first_word,unknown_rows,flexible_multiword_match
0,brand,3342,2064,0.6176,2064,1909.0,155.0,1278,NaN
1,product_type,3342,2165,0.6478,2165,NaN,NaN,1177,46.0


## 13. Unknown cases

Эти таблицы полезны для ручной проверки:
- где не поймался `brand`
- где не поймался `product_type`


In [80]:
unknown_brand = (
    processed.loc[processed['brand_pred'].eq(''), ['product_core', 'product_name']]
    .groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

unknown_product_type = (
    processed.loc[processed['product_type_pred'].eq(''), ['product_core', 'product_name']]
    .groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

print('unknown brand rows:', int(processed['brand_pred'].eq('').sum()))
print('unknown product_type rows:', int(processed['product_type_pred'].eq('').sum()))


unknown brand rows: 1278
unknown product_type rows: 1177


In [81]:
unknown_brand.head(20)


,product_core,count,example_product_name
0,nan optipro baby formula,3,NAN 1 Optipro Baby Formula 400g 139787
1,san pellegrino sparkling natural mineral water,3,SAN PELLEGRINO Sparkling Natural Mineral Water 6×1L 774780
2,soumaki eggs 53,3,SOUMAKI Eggs 12Pcs 53-63g 146657
3,yogurt char. christis straggato light,3,Yogurt CHAR. CHRISTIS Straggato Light 1kg 145352
4,absol ut vodka,2,ABSOL UT Vodka 1lt 118845
5,agrokipos halvatzi greek beetroot greek steamed,2,AGROKIPOS HALVATZI Greek beetroot Greek Steamed 500g 660296
6,agrokipos halvatzi greek steamed corn,2,AGROKIPOS HALVATZI Greek steamed corn 400g 660297
7,agrokipos halvatzi prepacked greek cooked chickpeas,2,AGROKIPOS HALVATZI Prepacked Greek Cooked Chickpeas 300g 709586
8,agrokipos halvatzi prepacked greek pumpkin ready to eat,2,AGROKIPOS HALVATZI Prepacked Greek Pumpkin Ready to Eat 400g 676035
9,arla protein pudding with salted caramel flavor lactose free no added sugar,2,ARLA Protein Pudding with Salted Caramel flavor Lactose free No added sugar 200g 775728


In [82]:
unknown_product_type.head(20)


,product_core,count,example_product_name
0,glaros health paper,4,GLAROS Health Paper 2 sheets 2pcs 230g 665940
1,nan optipro baby formula,3,NAN 1 Optipro Baby Formula 400g 139787
2,vileda super mop head standard,3,VILEDA Super Mop Head Standard 2 Pcs 150541
3,agrokipos halvatzi greek beetroot greek steamed,2,AGROKIPOS HALVATZI Greek beetroot Greek Steamed 500g 660296
4,agrokipos halvatzi greek steamed corn,2,AGROKIPOS HALVATZI Greek steamed corn 400g 660297
5,agrokipos halvatzi prepacked greek cooked chickpeas,2,AGROKIPOS HALVATZI Prepacked Greek Cooked Chickpeas 300g 709586
6,agrokipos halvatzi prepacked greek pumpkin ready to eat,2,AGROKIPOS HALVATZI Prepacked Greek Pumpkin Ready to Eat 400g 676035
7,alion mixed fruits pre cut,2,ALION Mixed Fruits Pre Cut 250g 720479
8,aroxol pure & strong insecticide liquid,2,AROXOL Pure & Strong Insecticide Liquid 2X25ml 801861
9,barilla penne rigate gluten free,2,BARILLA Penne Rigate Gluten Free 400g 641547


## 14. Кейсы, где бренд поймался не первым словом

Это просто проверочная таблица, чтобы можно было глазами посмотреть,
насколько адекватно работает поиск `brand` по всей строке.


In [83]:
brand_not_first_word = (
    processed.loc[
        (processed['brand_pred'].ne(''))
        & (processed['brand_match_method'] == 'brand_vocab_not_first_word'),
        ['brand_pred', 'product_core', 'product_name', 'brand_match_start']
    ]
    .groupby(['brand_pred', 'product_core'], as_index=False)
    .agg(
        count=('product_name', 'size'),
        first_brand_position=('brand_match_start', 'min'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'brand_pred', 'product_core'], ascending=[False, True, True], ignore_index=True)
)

print('brand matched not as first word:', len(processed.loc[processed['brand_match_method'] == 'brand_vocab_not_first_word']))
brand_not_first_word.head(30)


brand matched not as first word: 155


,brand_pred,product_core,count,first_brand_position,example_product_name
0,aptamil,milupa aptamil baby milk,3,7.0,MILUPA Aptamil 2 Baby Milk 6-12m 400g 133041
1,ambrosia,salad dip ambrosia fish roe salad (taramas,2,10.0,Salad Dip AMBROSIA Fish Roe Salad (Taramas) 250g 223720
2,ambrosia,salad dip ambrosia houmous,2,10.0,Salad Dip AMBROSIA Houmous 250g 232605
3,ambrosia,salad dip ambrosia tahini salad,2,10.0,Salad Dip AMBROSIA Tahini Salad 250g 223721
4,bonora,orange pie bonora,2,11.0,Orange Pie BONORA 240g 770366
5,ifantis,salad dip ifantis fish roe salad (taramas,2,10.0,Salad Dip Ifantis Fish Roe Salad (Taramas) 400g
6,ifantis,salad dip ifantis hummus salad,2,10.0,Salad Dip IFANTIS Hummus Salad 400g 738844
7,laikon,cyprus coffee laikon gold,2,14.0,Cyprus Coffee LAIKON Gold 200g 134707
8,sklavenitis,giant beans sklavenitis production,2,12.0,Giant Beans SKLAVENITIS ProductION 400g 775283
9,sklavenitis,orzo with shrimp sklavenitis production,2,17.0,Orzo with Shrimp SKLAVENITIS ProductION 400g 795271


In [84]:
processed.loc[
    processed['brand_match_method'] == 'brand_vocab_not_first_word',
    ['product_core', 'brand_pred', 'brand_match_start', 'product_type_pred', 'product_name']
].head(30)


,product_core,brand_pred,brand_match_start,product_type_pred,product_name
0,spaghetti with tomato sauce sklavenitis production,sklavenitis,28.0,sauce,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297
9,orzo with shrimp sklavenitis production,sklavenitis,17.0,,Orzo with Shrimp SKLAVENITIS ProductION 400g 795271
11,giant beans sklavenitis production,sklavenitis,12.0,beans,Giant Beans SKLAVENITIS ProductION 400g 775283
17,salad dip ifantis fish roe salad (taramas,ifantis,10.0,fish roe salad,Salad Dip Ifantis Fish Roe Salad (Taramas) 400g
18,salad dip ifantis hummus salad,ifantis,10.0,salad dip,Salad Dip IFANTIS Hummus Salad 400g 738844
22,salad dip ambrosia tahini salad,ambrosia,10.0,salad dip,Salad Dip AMBROSIA Tahini Salad 250g 223721
24,salad dip ambrosia fish roe salad (taramas,ambrosia,10.0,fish roe salad,Salad Dip AMBROSIA Fish Roe Salad (Taramas) 250g 223720
29,salad dip ambrosia houmous,ambrosia,10.0,salad dip,Salad Dip AMBROSIA Houmous 250g 232605
88,ryazhenka – 33 korovi 3.5% yoghurt drink,33 korovi,12.0,ryazhenka,RYAZHENKA – 33 KOROVI 3.5% Yoghurt Drink 500 ml 690030
183,cucumbers snack,snack,10.0,cucumbers,Cucumbers Snack 500g 788823


## 15. Сохраняем базовые результаты

После выполнения этой ячейки в папке `output` появятся базовые CSV по brand/product_type coverage.


In [85]:
brand_vocab.to_csv(BRAND_VOCAB_PATH, index=False, encoding='utf-8-sig')
product_type_vocab.to_csv(PRODUCT_TYPE_VOCAB_PATH, index=False, encoding='utf-8-sig')
processed.to_csv(FIRST_PASS_PATH, index=False, encoding='utf-8-sig')
coverage.to_csv(COVERAGE_PATH, index=False, encoding='utf-8-sig')
unknown_brand.to_csv(UNKNOWN_BRANDS_PATH, index=False, encoding='utf-8-sig')
unknown_product_type.to_csv(UNKNOWN_PRODUCT_TYPES_PATH, index=False, encoding='utf-8-sig')
brand_not_first_word.to_csv(BRAND_NOT_FIRST_WORD_PATH, index=False, encoding='utf-8-sig')

print('saved:', BRAND_VOCAB_PATH)
print('saved:', PRODUCT_TYPE_VOCAB_PATH)
print('saved:', FIRST_PASS_PATH)
print('saved:', COVERAGE_PATH)
print('saved:', UNKNOWN_BRANDS_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPES_PATH)
print('saved:', BRAND_NOT_FIRST_WORD_PATH)


saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\brand_vocab.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_vocab.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_brand_product_type_first_pass.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\brand_product_type_coverage_summary.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\unknown_brand_product_core_top.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\unknown_product_type_product_core_top.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\brand_match_not_first_word_examples.csv


## 16. Углубляемся в product_type триггеры

Теперь отдельным следующим шагом идём глубже именно в `product_type`.
Здесь `brand` используется только как вспомогательная подсказка внутри reviewed-500,
потому что мы хотим понять, какие части названия уже размечены,
а какие куски текста ещё могут нести смысл `product_type`.

Новая логика здесь такая:
- сначала проверяем, есть ли `product_type` прямо в `product_core`
- если есть, это обычный literal-кейс, и он не так интересен для триггеров
- если нет, только тогда вычитаем из `product_core` уже размеченные части
- смотрим, что осталось после вычитания
- если остаток не пустой, используем его как источник кандидатов на триггеры
- если остаток пустой, сохраняем такие кейсы отдельно


## 17. Сначала делим кейсы на literal и non-literal

`literal` здесь значит: нужный `product_type` прямо написан в `product_core`.

Пример literal-кейса:
- `product_core = greek yogurt strawberry`
- `product_type = yogurt`
- слово `yogurt` в тексте есть

`non-literal` значит: правильный `product_type` известен из ручной разметки,
но в самом `product_core` он не написан теми же словами.

Пример non-literal-кейса:
- `product_core = aged graviera with thyme`
- `product_type = cheese`
- слова `cheese` в тексте нет

Только для non-literal кейсов мы дальше вычитаем уже размеченные части:
- `brand`
- `variant`
- `attributes`

Важно: `No.`, `No5`, `No1` и похожие обозначения теперь не считаем `grade` на этом шаге.
Они остаются внутри `product_core` и будут разруливаться уже следующим алгоритмом отдельно.

После этого смотрим, что осталось.
Если ничего не осталось, это тоже важный отдельный тип кейсов.


In [86]:
# Разбиваем текст на токены, чтобы потом безопасно вычитать целые фразы.
TOKEN_PATTERN = re.compile(r"[a-z]+|\d+")

# Это только лёгкая уборка служебных слов-связок.
# Это уже не основная логика блока.
LIGHT_CONNECTOR_WORDS = {'and', 'with', 'for', 'from', 'the', 'a', 'an', 'to', 'of', 'in', 'on', 'by', 'or', 'at'}


def tokenize(text):
    # Приводим текст к нижнему регистру, убираем zero-width символы
    # и разбиваем на чистый список токенов.
    cleaned = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', str(text).lower())
    return TOKEN_PATTERN.findall(cleaned)


def normalize_phrase_list(value):
    # В variant / attributes может быть несколько фраз через запятую.
    # Разбиваем их, чтобы потом вычитать каждую фразу отдельно.
    if not value:
        return []
    parts = [normalize_text(part) for part in str(value).split(',')]
    return [part for part in parts if part]


def build_grade_removal_candidates(value):
    # Для grade иногда нужен не только исходный текст, но и его слитная форма.
    # Пример: grade = '12-24m', а в product_core осталось '1224m'.
    candidates = []
    for part in normalize_phrase_list(value):
        candidates.append(part)
        compact = re.sub(r'[\s\-–—/]+', '', part)
        if compact and compact != part:
            candidates.append(compact)
    # Сохраняем порядок и убираем дубликаты.
    return list(dict.fromkeys(candidates))


def remove_phrase_if_present(text, phrase):
    # Вспомогательная функция: если точная фраза есть в тексте, вычитаем её один раз.
    if not phrase:
        return text
    if not literal_phrase_present(text, phrase):
        return text
    remaining_tokens = remove_phrase_once_by_tokens(text, phrase)
    return ' '.join(remaining_tokens)


def build_product_core_for_trigger(row):
    # reviewed product_core местами всё ещё содержит grade из старой версии разметки.
    # Здесь делаем локально очищенную версию product_core именно для триггерного шага.
    # Важно: убираем только то, что реально лежит в колонке grade.
    # Обозначения вида No.5 / No1 / No 6 теперь не считаем grade и не вычитаем отсюда автоматически.
    text = row.product_core_norm
    for grade_part in build_grade_removal_candidates(getattr(row, 'grade', '')):
        text = remove_phrase_if_present(text, grade_part)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def literal_phrase_present(text, phrase):
    # Точное вхождение по токенам: слова должны идти подряд и в том же порядке.
    text_joined = ' '.join(tokenize(text))
    phrase_joined = ' '.join(tokenize(phrase))
    return bool(phrase_joined) and phrase_joined in text_joined


def product_type_present_in_core(core_text, product_type):
    # Для product_type используем более мягкую проверку.
    # Сначала ищем точную фразу.
    # Если product_type состоит из нескольких слов,
    # считаем совпадение и тогда, когда все слова есть в product_core,
    # даже если порядок другой или между ними есть другие слова.
    core_tokens = tokenize(core_text)
    product_type_tokens = tokenize(product_type)

    if not product_type_tokens:
        return False

    core_joined = ' '.join(core_tokens)
    product_type_joined = ' '.join(product_type_tokens)

    if product_type_joined in core_joined:
        return True

    if len(product_type_tokens) == 1:
        return False

    core_counts = Counter(core_tokens)
    product_type_counts = Counter(product_type_tokens)
    return all(core_counts[token] >= count for token, count in product_type_counts.items())


def remove_phrase_once_by_tokens(text, phrase):
    # Убираем из текста одну целую фразу, а не отдельные буквы или символы.
    text_tokens = tokenize(text)
    phrase_tokens = tokenize(phrase)

    if not phrase_tokens:
        return text_tokens

    result = []
    i = 0
    n = len(phrase_tokens)

    while i < len(text_tokens):
        if text_tokens[i:i + n] == phrase_tokens:
            i += n
        else:
            result.append(text_tokens[i])
            i += 1

    return result


def subtract_labeled_parts(row, product_core_for_trigger):
    # Для non-literal кейсов вычитаем из product_core уже размеченные части:
    # brand, variant и attributes.
    # Важно: сюда передаём уже локально очищенный product_core,
    # а не пытаемся читать несуществующее поле из row.
    remaining_text = product_core_for_trigger
    removed_parts = []

    for column in ['brand', 'variant', 'attributes']:
        for phrase in normalize_phrase_list(getattr(row, column, '')):
            if literal_phrase_present(remaining_text, phrase):
                remaining_tokens = remove_phrase_once_by_tokens(remaining_text, phrase)
                remaining_text = ' '.join(remaining_tokens)
                removed_parts.append(f'{column}: {phrase}')

    # После вычитания делаем только лёгкую уборку служебных слов и пробелов.
    remaining_tokens = [token for token in tokenize(remaining_text) if token not in LIGHT_CONNECTOR_WORDS]
    residual_text = ' '.join(remaining_tokens).strip()

    return residual_text, ' | '.join(removed_parts)


reviewed_product_type_check_rows = []

for row in reviewed.itertuples(index=False):
    if not row.product_type:
        continue

    product_core_for_trigger = build_product_core_for_trigger(row)

    # Шаг 1: проверяем, найден ли product_type в очищенном product_core.
    # Для многословного product_type разрешаем и случай,
    # когда все слова есть в product_core не подряд.
    literal_in_product_core = product_type_present_in_core(product_core_for_trigger, row.product_type)

    # Шаг 2: только для non-literal кейсов строим остаток после вычитания.
    if literal_in_product_core:
        residual_after_subtraction = ''
        removed_parts = ''
    else:
        residual_after_subtraction, removed_parts = subtract_labeled_parts(row, product_core_for_trigger)

    reviewed_product_type_check_rows.append({
        'product_core_original': row.product_core,
        'product_core': product_core_for_trigger,
        'product_type': row.product_type,
        'brand': row.brand,
        'variant': getattr(row, 'variant', ''),
        'attributes': getattr(row, 'attributes', ''),
        'grade': getattr(row, 'grade', ''),
        'category_path': getattr(row, 'category_path', ''),
        'literal_in_product_core': literal_in_product_core,
        'removed_parts': removed_parts,
        'residual_after_subtraction': residual_after_subtraction,
        'empty_after_subtraction': (not literal_in_product_core) and (residual_after_subtraction == ''),
    })

reviewed_product_type_check = pd.DataFrame(reviewed_product_type_check_rows)

non_literal_cases = reviewed_product_type_check.loc[
    ~reviewed_product_type_check['literal_in_product_core']
].copy()

non_literal_residual_cases = non_literal_cases.loc[
    non_literal_cases['residual_after_subtraction'].ne('')
].copy()

empty_after_subtraction_cases = non_literal_cases.loc[
    non_literal_cases['empty_after_subtraction']
].copy()

print('rows with product_type:', len(reviewed_product_type_check))
print('literal product_type in product_core:', int(reviewed_product_type_check['literal_in_product_core'].sum()))
print('non-literal cases:', len(non_literal_cases))
print('non-literal with residual text:', len(non_literal_residual_cases))
print('non-literal empty after subtraction:', len(empty_after_subtraction_cases))

non_literal_cases.head(15)


rows with product_type: 500
literal product_type in product_core: 405
non-literal cases: 95
non-literal with residual text: 46
non-literal empty after subtraction: 49


,product_core_original,product_core,product_type,brand,variant,attributes,grade,category_path,literal_in_product_core,removed_parts,residual_after_subtraction,empty_after_subtraction
5,bliss point – aged graviera with thyme,bliss point – aged graviera with thyme,cheese,bliss point,graviera with thyme,aged,NaN,Cheese & Vegetable Alternatives > Cheese Block,False,brand: bliss point | variant: graviera with thyme | attributes: aged,,True
27,coca cola,coca cola,drink,coca cola,NaN,NaN,NaN,"Soft Drinks, Water & Juices > Soft Drinks, Sodas & Energy Drinks",False,brand: coca cola,,True
30,lays wholegrain snacks salt,lays wholegrain snacks salt,potato chips,lays,salt,wholegrain snacks,NaN,"Nuts & Snacks > Potato Chips, Cheese Puff & Other Snacks",False,brand: lays | variant: salt | attributes: wholegrain snacks,,True
34,whiskas salmon,whiskas salmon,cat food,whiskas,salmon,NaN,NaN,Pet Food & Supplies > Cat Food & Supplies,False,brand: whiskas | variant: salmon,,True
45,clementine's,clementine's,clementines,,NaN,NaN,NaN,Fresh Fruits & Vegetables > Fruits,False,,clementine s,False
46,zoflora linen fresh multi-purpose disinfectant cleaning sol ution,zoflora linen fresh multi-purpose disinfectant cleaning sol ution,disinfectant cleaning solution,zoflora,"linen fresh, multi-purpose",NaN,NaN,Detergents & Cleaning Supplies > Multipurpose,False,brand: zoflora | variant: linen fresh | variant: multi-purpose,disinfectant cleaning sol ution,False
59,aroxol ants and cockroaches killer,aroxol ants and cockroaches killer,insect repellent,aroxol,ants and cockroaches killer,NaN,NaN,Detergents & Cleaning Supplies > Mosquito Repellents - Insect Repellents,False,brand: aroxol | variant: ants and cockroaches killer,,True
68,emmental dirollo light in slices,emmental dirollo light in slices,cheese,dirollo,"emmental, light",in slices,16% fat,Cheese & Vegetable Alternatives > Sliced Cheese,False,brand: dirollo | variant: emmental | variant: light | attributes: in slices,,True
71,🐾 rispet coconut & keratin shampoo & condition er for pets,🐾 rispet coconut & keratin shampoo & condition er for pets,shampoo & conditioner for pets,rispet,coconut & keratin,NaN,NaN,Pet Food & Supplies > Dog Food & Supplies,False,brand: rispet | variant: coconut & keratin,shampoo condition er pets,False
76,la molisana gluten free spaghetti,la molisana gluten free spaghetti,pasta,la molisana,spaghetti,gluten free,NaN,Gluten Free,False,brand: la molisana | variant: spaghetti | attributes: gluten free,,True


## 18. Из остатка собираем черновые trigger-кандидаты

Основная идея такая:
- сначала вычесть из `product_core` уже размеченные части
- потом посмотреть именно на остаток
- и уже из этого остатка собирать кандидатов на `trigger -> product_type`

То есть теперь триггеры ищутся не в почти сыром названии,
а в том, что осталось неразобранным после вычитания `brand`, `variant` и `attributes`.

Важно: это всё ещё не финальный словарь,
а аккуратный черновик для ручного просмотра.


In [87]:
def build_trigger_candidates(non_literal_df):
    # ngram_total: сколько раз триггер встретился вообще среди residual-кейсов.
    ngram_total = Counter()
    # ngram_type: сколько раз этот же триггер встретился именно с этим product_type.
    ngram_type = Counter()
    # examples: сохраняем несколько примеров очищенного остатка для ручной проверки.
    examples = defaultdict(list)
    # product_core_examples: сохраняем несколько примеров исходного product_core для того же trigger -> product_type.
    product_core_examples = defaultdict(list)

    for row in non_literal_df.itertuples(index=False):
        row_tokens = tokenize(row.residual_after_subtraction)
        # seen не даёт считать один и тот же триггер несколько раз внутри одной строки.
        seen = set()

        # Смотрим одиночные слова, пары слов и тройки слов.
        # Это помогает ловить и короткие триггеры вроде 'gouda', и фразы вроде 'baby milk'.
        for n in (1, 2, 3):
            for i in range(len(row_tokens) - n + 1):
                gram_tokens = row_tokens[i:i + n]

                # Пропускаем слишком слабые куски: чистые цифры и очень короткие токены.
                if any(token.isdigit() for token in gram_tokens):
                    continue
                if any(len(token) < 3 for token in gram_tokens):
                    continue

                gram = ' '.join(gram_tokens)
                if gram in seen:
                    continue
                seen.add(gram)

                ngram_total[gram] += 1
                ngram_type[(gram, row.product_type)] += 1

                if len(examples[(gram, row.product_type)]) < 3:
                    examples[(gram, row.product_type)].append(row.residual_after_subtraction)
                if len(product_core_examples[(gram, row.product_type)]) < 3:
                    product_core_examples[(gram, row.product_type)].append(row.product_core)

    rows = []
    for (gram, product_type), support in ngram_type.items():
        total = ngram_total[gram]
        precision = support / total
        rows.append({
            'product_type': product_type,
            'trigger': gram,
            # support_in_reviewed = сколько reviewed-строк подтверждают связь trigger -> product_type.
            'support_in_reviewed': support,
            # total_trigger_count = сколько раз этот trigger встретился вообще среди residual-кейсов.
            'total_trigger_count': total,
            # precision_in_residual_subset = support / total.
            'precision_in_residual_subset': round(precision, 3),
            # Если support == 1, триггер может быть полезным, но пока слабо подтверждён.
            'needs_manual_review': support == 1,
            'product_core': ' | '.join(product_core_examples[(gram, product_type)]),
            'example_residual_text': ' | '.join(examples[(gram, product_type)]),
        })

    result = pd.DataFrame(rows).sort_values(
        ['support_in_reviewed', 'precision_in_residual_subset', 'product_type', 'trigger'],
        ascending=[False, False, True, True],
        ignore_index=True,
    )
    return result


product_type_trigger_candidates = build_trigger_candidates(non_literal_residual_cases)
repeated_trigger_candidates = product_type_trigger_candidates.loc[
    (product_type_trigger_candidates['support_in_reviewed'] >= 2)
    & (product_type_trigger_candidates['precision_in_residual_subset'] >= 0.9)
].copy()

print('all trigger candidates from residual text:', len(product_type_trigger_candidates))
print('repeated trigger candidates:', len(repeated_trigger_candidates))
repeated_trigger_candidates.head(20)


all trigger candidates from residual text: 100
repeated trigger candidates: 9


,product_type,trigger,support_in_reviewed,total_trigger_count,precision_in_residual_subset,needs_manual_review,product_core,example_residual_text
0,potato chips,chips,3,3,1.0,False,corina – lights salt & vinegar chips | bonora wavy chips with salt | bonora barbeque wavy chips,lights chips | chips salt | chips
2,baby food,baby,2,2,1.0,False,noυnoυ baby farin lactee powder wheat flour and milk | s-26 gold ι baby milk powder,no no baby | baby
3,conditioner,hair,2,2,1.0,False,syoss color hair condition er for colored hair | syoss repair hair condition er,color hair condition er | hair condition er
4,conditioner,hair condition,2,2,1.0,False,syoss color hair condition er for colored hair | syoss repair hair condition er,color hair condition er | hair condition er
5,iced coffee,coffee,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
6,iced coffee,ice,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
7,iced coffee,ice coffee,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
8,tomatoes,tomato,2,2,1.0,False,kleopas cherry black tomato pre-packed | kleopas cherry red tomato pre-packed,tomato | tomato
9,yogurt,yoghurt,2,2,1.0,False,keses tradition al sheep's yoghurt | zita super protein strawberries yoghurt,tradition al yoghurt | yoghurt


## 19. Что смотреть глазами

Ниже самые полезные таблицы:
- `non_literal_cases` — все кейсы, где `product_type` не нашёлся прямо в очищенном `product_core`
- `empty_after_subtraction_cases` — non-literal кейсы, где после вычитания уже размеченных частей ничего не осталось
- `repeated_trigger_candidates` — более устойчивые кандидаты на триггеры

Как это читать:
- `literal_in_product_core = True` значит, что `product_type` нашёлся в очищенном `product_core`: либо точной фразой, либо по всем словам многословного product_type
- `product_core_original` — исходный reviewed `product_core` до локальной дочистки только тех частей, которые лежат в `grade`
- `residual_after_subtraction` — это текстовый остаток после вычитания `brand`, `variant` и `attributes`
- `empty_after_subtraction = True` значит, что non-literal кейс есть, но после вычитания остаток пустой
- высокий `support_in_reviewed` = триггер уже несколько раз подтверждался в reviewed-разметке
- высокий `precision_in_residual_subset` = триггер выглядит более чистым и менее двусмысленным в residual-кейсах


In [88]:
non_literal_cases[[
    'product_core', 'product_type', 'brand', 'variant', 'attributes',
    'removed_parts', 'residual_after_subtraction', 'empty_after_subtraction'
]].head(20)


,product_core,product_type,brand,variant,attributes,removed_parts,residual_after_subtraction,empty_after_subtraction
5,bliss point – aged graviera with thyme,cheese,bliss point,graviera with thyme,aged,brand: bliss point | variant: graviera with thyme | attributes: aged,,True
27,coca cola,drink,coca cola,NaN,NaN,brand: coca cola,,True
30,lays wholegrain snacks salt,potato chips,lays,salt,wholegrain snacks,brand: lays | variant: salt | attributes: wholegrain snacks,,True
34,whiskas salmon,cat food,whiskas,salmon,NaN,brand: whiskas | variant: salmon,,True
45,clementine's,clementines,,NaN,NaN,,clementine s,False
46,zoflora linen fresh multi-purpose disinfectant cleaning sol ution,disinfectant cleaning solution,zoflora,"linen fresh, multi-purpose",NaN,brand: zoflora | variant: linen fresh | variant: multi-purpose,disinfectant cleaning sol ution,False
59,aroxol ants and cockroaches killer,insect repellent,aroxol,ants and cockroaches killer,NaN,brand: aroxol | variant: ants and cockroaches killer,,True
68,emmental dirollo light in slices,cheese,dirollo,"emmental, light",in slices,brand: dirollo | variant: emmental | variant: light | attributes: in slices,,True
71,🐾 rispet coconut & keratin shampoo & condition er for pets,shampoo & conditioner for pets,rispet,coconut & keratin,NaN,brand: rispet | variant: coconut & keratin,shampoo condition er pets,False
76,la molisana gluten free spaghetti,pasta,la molisana,spaghetti,gluten free,brand: la molisana | variant: spaghetti | attributes: gluten free,,True


In [89]:
empty_after_subtraction_cases[[
    'product_core', 'product_type', 'brand', 'variant', 'attributes', 'removed_parts'
]].head(20)


,product_core,product_type,brand,variant,attributes,removed_parts
5,bliss point – aged graviera with thyme,cheese,bliss point,graviera with thyme,aged,brand: bliss point | variant: graviera with thyme | attributes: aged
27,coca cola,drink,coca cola,NaN,NaN,brand: coca cola
30,lays wholegrain snacks salt,potato chips,lays,salt,wholegrain snacks,brand: lays | variant: salt | attributes: wholegrain snacks
34,whiskas salmon,cat food,whiskas,salmon,NaN,brand: whiskas | variant: salmon
59,aroxol ants and cockroaches killer,insect repellent,aroxol,ants and cockroaches killer,NaN,brand: aroxol | variant: ants and cockroaches killer
68,emmental dirollo light in slices,cheese,dirollo,"emmental, light",in slices,brand: dirollo | variant: emmental | variant: light | attributes: in slices
76,la molisana gluten free spaghetti,pasta,la molisana,spaghetti,gluten free,brand: la molisana | variant: spaghetti | attributes: gluten free
77,hipp apple,baby food,hipp,apple,NaN,brand: hipp | variant: apple
81,makaronia melissa tricolore,pasta,melissa,makaronia tricolore,NaN,brand: melissa | variant: makaronia tricolore
86,armitage good boy snack with pork,dog food,armitage good boy,snack with pork,NaN,brand: armitage good boy | variant: snack with pork


In [90]:
repeated_trigger_candidates.head(30)


,product_type,trigger,support_in_reviewed,total_trigger_count,precision_in_residual_subset,needs_manual_review,product_core,example_residual_text
0,potato chips,chips,3,3,1.0,False,corina – lights salt & vinegar chips | bonora wavy chips with salt | bonora barbeque wavy chips,lights chips | chips salt | chips
2,baby food,baby,2,2,1.0,False,noυnoυ baby farin lactee powder wheat flour and milk | s-26 gold ι baby milk powder,no no baby | baby
3,conditioner,hair,2,2,1.0,False,syoss color hair condition er for colored hair | syoss repair hair condition er,color hair condition er | hair condition er
4,conditioner,hair condition,2,2,1.0,False,syoss color hair condition er for colored hair | syoss repair hair condition er,color hair condition er | hair condition er
5,iced coffee,coffee,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
6,iced coffee,ice,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
7,iced coffee,ice coffee,2,2,1.0,False,el cafe ice coffee with milk & sugar | bonora ice coffee with caramel macchiato flavor,ice coffee | ice coffee flavor
8,tomatoes,tomato,2,2,1.0,False,kleopas cherry black tomato pre-packed | kleopas cherry red tomato pre-packed,tomato | tomato
9,yogurt,yoghurt,2,2,1.0,False,keses tradition al sheep's yoghurt | zita super protein strawberries yoghurt,tradition al yoghurt | yoghurt


## 20. Сохраняем таблицы по product_type триггерам

Эти CSV нужны для следующего ручного шага.
Теперь отдельно сохраняем:
- полную таблицу literal/non-literal проверки
- non-literal кейсы с остатком
- non-literal кейсы с пустым остатком
- все trigger-кандидаты
- более устойчивые repeated trigger-кандидаты


In [91]:
PRODUCT_TYPE_LITERAL_CHECK_PATH = OUTPUT_DIR / 'product_type_literal_check.csv'
PRODUCT_TYPE_NON_LITERAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_cases.csv'
PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_residual_cases.csv'
PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH = OUTPUT_DIR / 'product_type_empty_after_subtraction_cases.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.csv'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.xlsx'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.xlsx'

reviewed_product_type_check.to_csv(PRODUCT_TYPE_LITERAL_CHECK_PATH, index=False, encoding='utf-8-sig')
non_literal_cases.to_csv(PRODUCT_TYPE_NON_LITERAL_CASES_PATH, index=False, encoding='utf-8-sig')
non_literal_residual_cases.to_csv(PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH, index=False, encoding='utf-8-sig')
empty_after_subtraction_cases.to_csv(PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH, index=False, encoding='utf-8-sig')
product_type_trigger_candidates.to_csv(PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
repeated_trigger_candidates.to_csv(PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
# NEW: для ручной работы удобно сохраняем те же trigger-таблицы ещё и в Excel.
product_type_trigger_candidates.to_excel(PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH, index=False)
repeated_trigger_candidates.to_excel(PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH, index=False)

print('saved:', PRODUCT_TYPE_LITERAL_CHECK_PATH)
print('saved:', PRODUCT_TYPE_NON_LITERAL_CASES_PATH)
print('saved:', PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH)
print('saved:', PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH)
print('saved:', PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH)
print('saved:', PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH)
print('saved:', PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH)
print('saved:', PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH)


saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_literal_check.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_non_literal_cases.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_non_literal_residual_cases.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_empty_after_subtraction_cases.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_trigger_candidates.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_trigger_candidates_repeated.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_trigger_candidates.xlsx
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_trigger_candidates_repeated.xlsx

In [92]:
import pandas as pd

trigger_candidates = pd.read_csv('output/product_type_trigger_candidates.csv')
trigger_candidates.head(30).sort_values('product_type')

,product_type,trigger,support_in_reviewed,total_trigger_count,precision_in_residual_subset,needs_manual_review,product_core,example_residual_text
11,adult diapers,adult,1,1,1.000,True,sani sensitive extra large (no. 4) adult open briefs,adult
14,baby diapers,extra care,1,1,1.000,True,huggeis extra care,extra care
13,baby diapers,extra,1,1,1.000,True,huggeis extra care,extra care
12,baby diapers,care,1,1,1.000,True,huggeis extra care,extra care
15,baby food,formula,1,1,1.000,True,almiron nutricia pronutra follow on formula no2,formula
16,baby food,milupa,1,1,1.000,True,milupa aptamil baby milk,milupa
17,baby food,snack,1,1,1.000,True,kiddylicious cheese flavour snack s,snack s
2,baby food,baby,2,2,1.000,False,noυnoυ baby farin lactee powder wheat flour and milk | s-26 gold ι baby milk powder,no no baby | baby
22,cat food,complete food cats,1,1,1.000,True,whiskas complete wet food for cats poultry selection in jelly,complete food cats
24,cat food,food cats,1,1,1.000,True,whiskas complete wet food for cats poultry selection in jelly,complete food cats


## 21. Подключаем отдельный manual trigger-словарь product_type

Теперь ручной словарь лежит в отдельном файле, а не только во вкладке Excel.

Логика шага такая:
- сначала оставляем базовый `product_type_pred`, который уже нашёлся словарями
- потом только для пустых `product_type` пробуем ручной trigger-словарь
- если у правила стоит `category_check_required = yes`, дополнительно проверяем `category_path_keywords`
- сохраняем полный результат, coverage и отдельную таблицу manual-hit кейсов


In [93]:
import pandas as pd

# Здесь не пересобираем финальные Sklavenitis-артефакты заново,
# а загружаем уже подтверждённые итоговые файлы с корректным coverage.
MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH = OUTPUT_DIR / 'product_type_manual_trigger_dictionary_current.csv'
FIRST_PASS_WITH_MANUAL_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_with_manual_triggers.csv'
COVERAGE_WITH_MANUAL_PATH = OUTPUT_DIR / 'brand_product_type_coverage_with_manual_triggers.csv'
MANUAL_TRIGGER_HITS_PATH = OUTPUT_DIR / 'product_type_manual_trigger_hits.csv'
MANUAL_TRIGGER_HITS_XLSX_PATH = OUTPUT_DIR / 'product_type_manual_trigger_hits.xlsx'

manual_trigger_dict = pd.read_csv(MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH, encoding='utf-8-sig').fillna('')
first_pass_with_manual = pd.read_csv(FIRST_PASS_WITH_MANUAL_PATH, encoding='utf-8-sig').fillna('')
coverage_with_manual = pd.read_csv(COVERAGE_WITH_MANUAL_PATH, encoding='utf-8-sig').fillna('')

if MANUAL_TRIGGER_HITS_PATH.exists():
    manual_trigger_hits = pd.read_csv(MANUAL_TRIGGER_HITS_PATH, encoding='utf-8-sig').fillna('')
else:
    manual_trigger_hits = pd.DataFrame()

print('loaded:', MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH)
print('loaded:', FIRST_PASS_WITH_MANUAL_PATH)
print('loaded:', COVERAGE_WITH_MANUAL_PATH)
if MANUAL_TRIGGER_HITS_PATH.exists():
    print('loaded:', MANUAL_TRIGGER_HITS_PATH)
if MANUAL_TRIGGER_HITS_XLSX_PATH.exists():
    print('loaded:', MANUAL_TRIGGER_HITS_XLSX_PATH)
print('manual dictionary rows:', len(manual_trigger_dict))
print('manual trigger hit rows:', len(manual_trigger_hits))
coverage_with_manual


loaded: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_manual_trigger_dictionary_current.csv
loaded: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_brand_product_type_with_manual_triggers.csv
loaded: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\brand_product_type_coverage_with_manual_triggers.csv
loaded: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_manual_trigger_hits.csv
loaded: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\product_type_manual_trigger_hits.xlsx
manual dictionary rows: 682
manual trigger hit rows: 921


,field,total_rows,matched_rows,matched_share,unknown_rows,manual_trigger_added_rows
0,brand_before_manual_triggers,3342,2064,0.6176,1278,
1,brand_after_manual_triggers,3342,2853,0.8537,489,789.0
2,product_type_before_manual_triggers,3342,2168,0.6487,1174,
3,product_type_after_manual_triggers,3342,3089,0.9243,253,921.0


In [94]:
manual_trigger_hits.head(20)


,product_core,product_core_without_brand,category_path,product_name,image_url,product_url,product_type_pred_with_manual,product_type_match_method_with_manual,manual_trigger_used,manual_category_keywords_used,manual_trigger_notes_used
0,orzo with shrimp sklavenitis production,orzo with shrimp production,Lenten Food,Orzo with Shrimp SKLAVENITIS ProductION 400g 795271,https://imageproxy.wolt.com/assets/69984fb83eb3341f036e6b8f,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,pasta,product_type_manual_trigger,orzo,,
1,vici cold smoked atlantic herring fillet,vici cold smoked atlantic herring fillet,Lenten Food,VICI Cold Smoked Atlantic Herring Fillet 200g 687000,https://imageproxy.wolt.com/assets/697372df4af2fcbd36bd6d4f,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,smoked fish,product_type_manual_trigger,herring fillet,,
2,amazing island salted and dried sea bream,amazing island salted and dried sea bream,Lenten Food,AMAZING ISLAND Salted and Dried Sea Bream 350g 645328,https://imageproxy.wolt.com/assets/697372289b938fe622d8d714,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,dried fish,product_type_manual_trigger_with_category,dried sea bream,lenten,
3,ifantis caramelized onion hummus,caramelized onion hummus,Lenten Food,IFANTIS Caramelized Onion Hummus 250g 755730,https://imageproxy.wolt.com/assets/697364769b938fe622d8d14c,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,hummus,product_type_manual_product_type,hummus,,
4,magoulitsa hummus,hummus,Lenten Food,MAGOULITSA Hummus 250g 747889,https://imageproxy.wolt.com/assets/697372945790090b1e1cc26b,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,hummus,product_type_manual_product_type,hummus,,
5,alfa gefsi garlic spread (skordalia,alfa gefsi garlic spread (skordalia,Lenten Food,ALFA GEFSI Garlic Spread (Skordalia) 250g 757292,https://imageproxy.wolt.com/assets/697364709b938fe622d8d136,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,garlic spread,product_type_manual_product_type,garlic spread,,
6,violife cheddar flavour vegan slices,cheddar flavour vegan slices,Lenten Food,VIOLIFE Cheddar Flavour Vegan Slices 140g 780619,https://imageproxy.wolt.com/assets/697364739b938fe622d8d13f,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,cheese,product_type_manual_trigger,cheddar,,
7,df pikla,pikla,Lenten Food,DF Pikla 470g 136626,https://imageproxy.wolt.com/assets/669695fc83329c5657597168,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,pickled vegetables,product_type_manual_trigger,pikla,,
8,hengstenb knax gherkins,hengstenb knax gherkins,Lenten Food,HENGSTENB Knax Gherkins 185g 138461,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/b99bfed0-0ade-11f0-bb5e-7a72dc818c2a_40081427.jpg,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,pickled vegetables,product_type_manual_trigger,gherkins,,
9,agrokipos halvatzi prepacked greek cooked chickpeas,agrokipos halvatzi prepacked greek cooked chickpeas,Lenten Food,AGROKIPOS HALVATZI Prepacked Greek Cooked Chickpeas 300g 709586,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/21911c50-6930-11f0-9f02-ce025473cae5_5202658509361.jpg,https://wolt.com/el/cyp/limassol/venue/sklavenitis-ypsonas,cooked chickpeas,product_type_manual_product_type,cooked chickpeas,,


## 22. Смотрим unknown product_type после manual trigger-словаря

Этот блок показывает, какие `product_type` всё ещё не нашлись даже после ручного trigger-словаря.

Здесь сохраняем две таблицы:
- полные `unknown`-строки
- сводную top-таблицу по `product_core`, чтобы быстро видеть, что осталось добивать


In [95]:
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_rows.csv'
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_top.csv'
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_top.xlsx'

first_pass_with_manual = pd.read_csv(FIRST_PASS_WITH_MANUAL_PATH, encoding='utf-8-sig').fillna('')

unknown_after_manual = first_pass_with_manual.loc[
    first_pass_with_manual['product_type_pred_with_manual'].astype(str).str.strip().eq(''),
    ['product_core', 'product_name', 'category_path', 'product_core_without_brand']
].copy()

unknown_after_manual_top = (
    unknown_after_manual.groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first'),
        example_category_path=('category_path', 'first'),
        example_product_core_without_brand=('product_core_without_brand', 'first'),
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

unknown_after_manual.to_csv(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH, index=False, encoding='utf-8-sig')
unknown_after_manual_top.to_csv(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH, index=False, encoding='utf-8-sig')
unknown_after_manual_top.to_excel(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH, index=False)

print('unknown rows after manual triggers:', len(unknown_after_manual))
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH)
unknown_after_manual_top.head(30)


unknown rows after manual triggers: 253
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\unknown_product_type_after_manual_triggers_rows.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\unknown_product_type_after_manual_triggers_top.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\unknown_product_type_after_manual_triggers_top.xlsx


,product_core,count,example_product_name,example_category_path,example_product_core_without_brand
0,glaros health paper,4,GLAROS Health Paper 2 sheets 2pcs 230g 665940,Everyday Paper Needs > Toilet Paper,health paper
1,forza blu cubes for cistern blue,2,FORZA BLU Cubes for Cistern Blue 2x50g 756642,Detergents & Cleaning Supplies > Bathroom Cleaners,forza blu cubes for cistern blue
2,imported praws,2,Imported Praws 10/20 802962,Fresh Meat -Fish & Seafood > Fish & Seafood,imported praws
3,moving mountains vegan burgers,2,MOVING MOUNTAINS Vegan Burgers 227g 717230,Frozen > Frozen Vegan Products,moving mountains vegan burgers
4,orange pie bonora,2,Orange Pie BONORA 240g 770366,"Bakery, Cakes & Treats > Cakes - Pancakes & More",orange pie
5,pet pet disposable pet training pads,2,PET PET Disposable Pet Training Pads 60×60 cm 15Pcs 668353,Pet Food & Supplies > Dog Food & Supplies,pet pet disposable pet training pads
6,snickers bar,2,SNICKERS Bar 2pcs 80g 121721,"Cookies, Chocolates & Sweets > Chocolates-Chewing Gums-Candy",snickers bar
7,weetabix protein,2,WEETABIX Protein 440g 656780,Protein Diet,protein
8,3a canadian green lentils,1,3A Canadian Green Lentils 1Kg 139873,Food Cupboard > Legumes,canadian green lentils
9,5 oceans marinated salmon trout,1,5 OCEANS Marinated Salmon Trout 100g 649447,Appetizers & Delicatessen > Smoked Fish,5 oceans marinated salmon trout


## 23. Итоги coverage по brand и product_type

Для product_type и brand добавила ручные варианты и trigger-слова.

Ниже конечный coverage после всех манипуляций и итерационных доработок в файлах. Считаю итоговые доли покрытия по формуле:

coverage = matched_rows / total_rows

Где:
- total_rows — это общее число строк в полном очищенном датасете Sklavenitis
- matched_rows — это сколько строк алгоритм смог автоматически разметить
- matched_share — это доля таких строк от всего датасета


In [96]:
COVERAGE_WITH_MANUAL_PATH = OUTPUT_DIR / 'brand_product_type_coverage_with_manual_triggers.csv'

coverage_with_manual = pd.read_csv(COVERAGE_WITH_MANUAL_PATH, encoding='utf-8-sig')
coverage_with_manual


,field,total_rows,matched_rows,matched_share,unknown_rows,manual_trigger_added_rows
0,brand_before_manual_triggers,3342,2064,0.6176,1278,NaN
1,brand_after_manual_triggers,3342,2853,0.8537,489,789.0
2,product_type_before_manual_triggers,3342,2168,0.6487,1174,NaN
3,product_type_after_manual_triggers,3342,3089,0.9243,253,921.0


### Как читать эти числа

Для product_type итоговое покрытие после ручного trigger-словаря сейчас высокое, потому что мы уже добавили:
- базовый словарь из reviewed-разметки
- ручные trigger-слова для нестандартных названий
- дополнительные product_type, которые встретились в unknown-остатках
- проверки по категории там, где без нее были бы ложные срабатывания

Для brand покрытие ниже, чем для product_type: бренды чаще бывают редкими, длинными, нестандартно написанными или встречаются только в части ассортимента. Поэтому брендовый слой обычно приходится доразмечать дольше.

Текущие числа уже после применения ручных brand-правил из Excel-файла с unknown brand: из него берутся заполненные brand и trigger_word, после чего эти правила применяются ко всему датасету Sklavenitis.


In [97]:
def get_coverage_row(field_name):
    matched_rows = coverage_with_manual.loc[coverage_with_manual['field'] == field_name]
    if matched_rows.empty:
        return None
    return matched_rows.iloc[0]

product_type_row = get_coverage_row('product_type_after_manual_triggers')

# После product_type manual triggers в coverage_with_manual может не остаться строки
# про brand. Тогда честно считаем brand coverage прямо из финального first-pass файла.
brand_row = get_coverage_row('brand_after_manual_triggers')
if brand_row is None:
    first_pass_with_manual = pd.read_csv(FIRST_PASS_WITH_MANUAL_PATH, encoding='utf-8-sig').fillna('')
    brand_matched_rows = int(first_pass_with_manual['brand_pred_with_manual'].astype(str).str.strip().ne('').sum())
    brand_row = {
        'matched_rows': brand_matched_rows,
        'total_rows': int(len(first_pass_with_manual)),
        'matched_share': brand_matched_rows / len(first_pass_with_manual),
    }

print('Формула: coverage = matched_rows / total_rows')
print('')
print('Product type coverage after manual triggers:')
print(f"{int(product_type_row['matched_rows'])} / {int(product_type_row['total_rows'])} = {product_type_row['matched_share'] * 100:.2f}%")
print('')
print('Brand coverage after manual triggers:')
print(f"{int(brand_row['matched_rows'])} / {int(brand_row['total_rows'])} = {brand_row['matched_share'] * 100:.2f}%")


Формула: coverage = matched_rows / total_rows

Product type coverage after manual triggers:
3089 / 3342 = 92.43%

Brand coverage after manual triggers:
2853 / 3342 = 85.37%


### Короткая интерпретация результата

product_type сейчас покрыт на 92%, brand сейчас покрыт на 85%, спасибо Господи. Можно переходить к следующему этапу - прогонять на втором супермаркете Alpha Omega


## 24. Alpha Omega transfer

This block reuses the finished Sklavenitis dictionaries and applies the same cleanup idea to `Alpha Omega`.
It saves a cleaned processed file, a first-pass dictionary file, a final file with manual triggers, and a coverage summary in `output/`.


In [98]:
from scripts.alphaomega_pipeline import run_alphaomega_pipeline

alphaomega_results = run_alphaomega_pipeline()
alphaomega_results['coverage']


saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_processed.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_brand_product_type_first_pass.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_brand_product_type_with_manual_triggers.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_brand_product_type_coverage_with_manual_triggers.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_unknown_brand_product_core_top_after_manual.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\alphamega_unknown_product_type_top_after_manual.csv

                              field  total_rows  matched_rows  matched_share  unknown_rows  manual_trigger_added_rows
       brand_before_manual_triggers        8672          2575         0.2969

,field,total_rows,matched_rows,matched_share,unknown_rows,manual_trigger_added_rows
0,brand_before_manual_triggers,8672,2575,0.2969,6097,NaN
1,brand_after_manual_triggers,8672,4213,0.4858,4459,1638.0
2,product_type_before_manual_triggers,8672,5155,0.5944,3517,NaN
3,product_type_after_manual_triggers,8672,6127,0.7065,2545,972.0


In [99]:
alphaomega_results['final'][[
    'product_name', 'product_core', 'size_raw', 'grade',
    'brand_pred_with_manual', 'product_type_pred_with_manual'
]].head(20)


,product_name,product_core,size_raw,grade,brand_pred_with_manual,product_type_pred_with_manual
0,Qualifood Salmon 4x125 g,qualifood salmon,4x125 g,NaN,,
1,Creta Farms En Elladi Chicken Nuggets 400 g,creta farms en elladi chicken nuggets,400 g,NaN,creta farms,nuggets
2,Green Isle Frozen Blueberries 300 g,green isle frozen blueberries,300 g,NaN,,
3,Qualifood Salmon Fillet With Skin 4x125 g,qualifood salmon fillet with skin,4x125 g,NaN,,salmon fillet
4,Party Ice Ice Cubes 1.2 kg,party ice ice cubes,1.2 kg,NaN,,
5,Alphamega Puff Pastry Leaves 850 g,alphamega puff pastry leaves,850 g,NaN,private label,
6,Grigoriou Pizza Special 490 g,grigoriou pizza special,490 g,NaN,grigoriou,pizza
7,Mitsides Traditional Village Ravioli 375 g,mitsides traditional village ravioli,375 g,NaN,mitsides,ravioli
8,Party Ice Ice Cubes 3 kg,party ice ice cubes,3 kg,NaN,,
9,Edesma Chicken Nuggets 900 g,edesma chicken nuggets,900 g,NaN,edesma,nuggets


## 25. Candidate blocking для cross-store matching

Теперь переходим к следующему этапу: хотим матчить товары между `Sklavenitis` и `Alpha Omega`.

Но перед тем как строить сам matcher, нам нужен `candidate blocking`.
Смысл шага такой:

- мы не сравниваем каждый товар с каждым, потому что это даёт слишком много шумных пар
- сначала строим только разумные кандидатные пары
- потом уже на этих кандидатах можно будет запускать более точную логику matching

В этом блоке делаем не одно жёсткое правило, а `union` из нескольких правил.
Это важно, потому что одно правило почти всегда либо:

- слишком слабое и даёт слишком много лишних пар
- либо слишком жёсткое и теряет правильные пары

Поэтому здесь берём сразу несколько блоков:

1. `product_type + exact size`
Смотрим одинаковый `product_type` и одинаковый нормализованный размер, потому что это сейчас самый стабильный структурный якорь.

2. `brand + product_type`
Смотрим одинаковые `brand` и `product_type`, потому что когда оба поля уже нашлись, это даёт очень сильную и чистую подгруппу.

3. `brand + exact size`
Смотрим одинаковый `brand` и одинаковый размер, потому что иногда `product_type` ещё пустой, но бренд и фасовка уже достаточно хорошо режут пространство сравнения.

4. `product_type + informative token`
Это rescue-блок: смотрим одинаковый `product_type` и хотя бы один общий смысловой токен в очищенном тексте, потому что так можно вернуть часть хороших кандидатов там, где brand или size не сработали.

5. `product_type + similar brand fragment`
Это ещё один rescue-блок: смотрим одинаковый `product_type` и похожий brand-like фрагмент строки, потому что бренд может отличаться пробелом, точкой, апострофом или небольшой опечаткой.
При этом не считаем, что весь остаток названия — это бренд.
Сравниваем только вероятные куски строки: бренд, если он уже найден, префикс до `product_type` и остаток текста после вычитания найденного бренда.

6. `similar brand fragment + exact size`
Это более строгий rescue-блок: если похожий брендовый фрагмент сочетается с одинаковым общим размером, такая пара становится заметно правдоподобнее даже без exact `product_type`.

Важно: blocking ещё не говорит, что товары одинаковые.
Он только отвечает на вопрос:
`с кем вообще стоит сравнивать этот товар дальше`.


In [100]:
from pathlib import Path
from difflib import SequenceMatcher
import re

import numpy as np
import pandas as pd

# Здесь задаём пути к уже готовым финальным таблицам двух магазинов.
# Берём именно финальные файлы после ручных brand/product_type правил,
# потому что хотим строить blocking уже на максимально сильных полях.
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

SKL_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_with_manual_triggers.csv'
ALPHA_PATH = OUTPUT_DIR / 'alphamega_brand_product_type_with_manual_triggers.csv'

CANDIDATE_PAIRS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_v1.csv'
CANDIDATE_SUMMARY_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_blocking_summary_v1.csv'

sk_raw = pd.read_csv(SKL_PATH, encoding='utf-8-sig').fillna('')
alpha_raw = pd.read_csv(ALPHA_PATH, encoding='utf-8-sig').fillna('')

print('Sklavenitis rows:', len(sk_raw))
print('Alpha Omega rows:', len(alpha_raw))


Sklavenitis rows: 3342
Alpha Omega rows: 8672


### Подготовка полей для blocking

Сначала приводим поля к единому виду.

Смотрим:

- `brand_pred_with_manual`, потому что это наш лучший доступный бренд после ручных правил
- `product_type_pred_with_manual`, потому что это основной смысловой якорь товара
- `size_total_base + size_base_unit`, потому что общий размер уже нормализован в общую систему
- `product_core_without_brand_with_manual`, потому что это хороший текстовый остаток для rescue-блоков
- `product_core`, потому что в нём можно отдельно посмотреть кусок текста до `product_type`, где бренд часто живёт особенно часто

Ниже код делает три вещи:

1. собирает аккуратные blocking-поля для обоих магазинов
2. готовит `informative tokens` — смысловые токены из очищенного названия для rescue-блока по тексту
3. готовит `brand candidate phrases` — короткие куски строки, которые потом используем для rescue-блока по похожему бренду


In [101]:
# Здесь перечисляем короткий список частых служебных слов,
# которые не помогают отличать один товар от другого.
BLOCKING_STOP_WORDS = {
    'and', 'for', 'from', 'the', 'a', 'an', 'of', 'in', 'on', 'to'
}

# Для rescue-блоков нам удобны два способа разбиения текста:
# 1) regex-токены для обычных смысловых слов и чисел
# 2) split по пробелам для брендовых фрагментов, где важно не терять точки, апострофы и пробелы внутри куска
TOKEN_PATTERN = re.compile(r"[a-z]+|\d+")

# Здесь задаём высокий порог похожести для брендовых rescue-блоков,
# чтобы они ловили мягкие различия вроде пробела или точки, но не расползались в шум.
BRAND_SIMILARITY_THRESHOLD = 0.85


def normalize_text(value):
    # Приводим текст к нижнему регистру и убираем скрытые символы,
    # чтобы одинаковые слова сравнивались одинаково.
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def strip_edge_punctuation(text):
    # Убираем только мусор по краям фразы.
    # Внутренние точки, апострофы и пробелы не трогаем,
    # потому что для брендов они могут быть полезны.
    return normalize_text(text).strip(" ,.;:!?/\\|-_()[]{}\"'")


def normalize_numeric_key(value):
    # Приводим числовое значение к стабильному текстовому ключу.
    # Так 6 и 6.0 дальше считаются одним и тем же значением.
    numeric = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(numeric):
        return ''

    if float(numeric).is_integer():
        return str(int(numeric))

    return str(round(float(numeric), 3)).rstrip('0').rstrip('.')


def choose_best_brand(row):
    # Берём финальный brand после manual-правил.
    return normalize_text(row.get('brand_pred_with_manual', ''))


def choose_best_product_type(row):
    # Берём финальный product_type после manual-правил.
    return normalize_text(
        row.get('product_type_pred_with_manual', '')
    )


def choose_residual_text(row):
    # Берём финальный текст после вычитания бренда.
    return normalize_text(
        row.get('product_core_without_brand_with_manual', '')
    )


def tokenize(text):
    # Возвращаем список токенов в чистом виде.
    return TOKEN_PATTERN.findall(normalize_text(text))


def split_preserving_punctuation(text):
    # Для брендовых кандидатов разбиваем только по пробелам,
    # чтобы не ломать записи вроде dr.oetker или mcvitie's.
    return [part for part in normalize_text(text).split(' ') if part]


def normalize_size_key(size_value, size_unit):
    # Собираем размер в один текстовый ключ вида 500_g или 750_ml.
    # Это удобнее, чем каждый раз отдельно матчить число и единицу измерения.
    unit = normalize_text(size_unit)
    numeric_text = normalize_numeric_key(size_value)

    if unit == '' or numeric_text == '':
        return ''

    return f'{numeric_text}_{unit}'


def normalize_pack_count_key(pack_count):
    # Количество упаковок тоже приводим к стабильному текстовому ключу.
    return normalize_numeric_key(pack_count)


def normalize_unit_size_key(unit_size_value, unit_size_unit):
    # Для multipack-правил хотим отдельно смотреть размер одной единицы внутри упаковки.
    return normalize_size_key(unit_size_value, unit_size_unit)


def build_informative_tokens(row):
    # Здесь оставляем слова и числа, которые могут помочь rescue-блоку найти похожие товары.
    residual_text = choose_residual_text(row)
    tokens = tokenize(residual_text)

    product_type_tokens = set(tokenize(row.get('product_type_block', '')))
    brand_tokens = set(tokenize(row.get('brand_block', '')))

    informative_tokens = []
    seen = set()

    for token in tokens:
        # Убираем слова, которые уже лежат в brand/product_type,
        # иначе rescue-блок просто будет повторять более сильные правила.
        if token in product_type_tokens:
            continue
        if token in brand_tokens:
            continue

        # Убираем общие служебные слова.
        if token in BLOCKING_STOP_WORDS:
            continue

        if token not in seen:
            informative_tokens.append(token)
            seen.add(token)

    return informative_tokens


def extract_text_before_product_type(product_core_text, product_type_text):
    # Смотрим кусок строки до product_type,
    # потому что бренд очень часто живёт именно там.
    product_core_text = normalize_text(product_core_text)
    product_type_text = normalize_text(product_type_text)

    if product_core_text == '' or product_type_text == '':
        return ''

    position = product_core_text.find(product_type_text)
    if position <= 0:
        return ''

    return strip_edge_punctuation(product_core_text[:position])


def build_brand_search_fragments(row):
    # Здесь собираем несколько зон, где бренд может встретиться:
    # 1) сам найденный brand, если он уже есть
    # 2) префикс product_core до product_type
    # 3) остаток текста после вычитания найденного бренда
    brand_text = normalize_text(row.get('brand_block', ''))
    product_core_text = normalize_text(row.get('product_core_block', ''))
    product_type_text = normalize_text(row.get('product_type_block', ''))
    residual_text = choose_residual_text(row)

    fragments = []
    seen = set()

    def add_fragment(text):
        cleaned = strip_edge_punctuation(text)
        if cleaned and cleaned not in seen:
            fragments.append(cleaned)
            seen.add(cleaned)

    add_fragment(brand_text)
    add_fragment(extract_text_before_product_type(product_core_text, product_type_text))
    add_fragment(residual_text)

    return fragments


def expand_brand_candidate_phrases(fragments):
    # Из каждой зоны строим короткие куски по 1-3 токена.
    # Так мы не считаем весь остаток текста брендом,
    # а проверяем только локальные brand-like фрагменты.
    phrases = []
    seen = set()

    for fragment in fragments:
        parts = split_preserving_punctuation(fragment)

        for window_size in (1, 2, 3):
            if len(parts) < window_size:
                continue

            for start in range(len(parts) - window_size + 1):
                phrase = ' '.join(parts[start:start + window_size]).strip()
                phrase = strip_edge_punctuation(phrase)

                # Для brand rescue хотим убрать совсем пустые и слишком короткие куски,
                # где после выкидывания пробелов и знаков почти не осталось содержимого.
                compact_phrase = re.sub(r'[^a-z0-9]+', '', phrase)
                if len(compact_phrase) < 3:
                    continue

                # Если в куске нет букв вообще, для бренда он нам не помогает.
                if not re.search(r'[a-z]', phrase):
                    continue

                if phrase and phrase not in seen:
                    phrases.append(phrase)
                    seen.add(phrase)

    return phrases


def first_letter_key(text):
    # Для мягких brand-блоков берём первый буквенно-цифровой символ.
    # Это дешёвый способ сильно сократить число сравнений, не убивая recall слишком рано.
    match = re.search(r'[a-z0-9]', normalize_text(text))
    return match.group(0) if match else ''


def brand_similarity_score(left_text, right_text):
    # Сравниваем бренд и брендовый фрагмент посимвольно,
    # не вычищая точки, пробелы и апострофы внутри строки.
    left_text = normalize_text(left_text)
    right_text = normalize_text(right_text)

    if left_text == '' or right_text == '':
        return 0.0

    return SequenceMatcher(None, left_text, right_text).ratio()


def prepare_store(df, store_prefix):
    # Готовим единый набор blocking-полей для каждого магазина.
    prepared = df.copy().reset_index(drop=True)

    prepared[f'{store_prefix}_item_id'] = [f'{store_prefix}_{i}' for i in range(len(prepared))]
    prepared['brand_block'] = prepared.apply(choose_best_brand, axis=1)
    prepared['product_type_block'] = prepared.apply(choose_best_product_type, axis=1)
    prepared['grade_block'] = prepared['grade'].map(normalize_text)
    prepared['product_core_block'] = prepared['product_core'].map(normalize_text)
    prepared['residual_block'] = prepared.apply(choose_residual_text, axis=1)
    prepared['size_key'] = prepared.apply(
        lambda row: normalize_size_key(row.get('size_total_base', ''), row.get('size_base_unit', '')),
        axis=1
    )
    prepared['pack_count_block'] = prepared['pack_count'].apply(normalize_pack_count_key)
    prepared['unit_size_key'] = prepared.apply(
        lambda row: normalize_unit_size_key(row.get('unit_size_value', ''), row.get('unit_size_unit', '')),
        axis=1
    )
    prepared['size_unit_block'] = prepared['size_base_unit'].map(normalize_text)
    prepared['informative_tokens_raw'] = prepared.apply(build_informative_tokens, axis=1)
    prepared['brand_search_fragments_raw'] = prepared.apply(build_brand_search_fragments, axis=1)
    prepared['brand_candidate_phrases'] = prepared['brand_search_fragments_raw'].apply(
        expand_brand_candidate_phrases
    )
    prepared['brand_first_char'] = prepared['brand_block'].apply(first_letter_key)

    return prepared


sk = prepare_store(sk_raw, 'sk')
alpha = prepare_store(alpha_raw, 'am')

# Считаем, насколько часто встречается каждый смысловой токен в двух магазинах вместе.
# Это нужно, чтобы rescue-блок не опирался на слишком общие слова вроде
# очень популярных вкусов или категорий, которые дадут слишком много шума.
token_rows = []

for df, item_col in ((sk, 'sk_item_id'), (alpha, 'am_item_id')):
    for row in df[[item_col, 'informative_tokens_raw']].itertuples(index=False):
        for token in row.informative_tokens_raw:
            token_rows.append({'item_id': getattr(row, item_col), 'token': token})

token_stats = pd.DataFrame(token_rows)

if not token_stats.empty:
    token_stats = (
        token_stats.drop_duplicates()
        .groupby('token', as_index=False)
        .agg(item_count=('item_id', 'nunique'))
    )
    valid_tokens = set(
        token_stats.loc[
            (token_stats['item_count'] >= 2) & (token_stats['item_count'] <= 200),
            'token'
        ]
    )
else:
    valid_tokens = set()


def filter_valid_tokens(tokens):
    # Оставляем только токены, которые не слишком редкие и не слишком частые.
    # Если токен встретился всего один раз, он всё равно не поможет построить пару.
    return [token for token in tokens if token in valid_tokens]


sk['informative_tokens'] = sk['informative_tokens_raw'].apply(filter_valid_tokens)
alpha['informative_tokens'] = alpha['informative_tokens_raw'].apply(filter_valid_tokens)

print('Sklavenitis rows with product_type:', int(sk['product_type_block'].ne('').sum()))
print('Alpha Omega rows with product_type:', int(alpha['product_type_block'].ne('').sum()))
print('Valid informative tokens for rescue block:', len(valid_tokens))


Sklavenitis rows with product_type: 3089
Alpha Omega rows with product_type: 6127
Valid informative tokens for rescue block: 3505


### Строим candidate pairs по нескольким правилам

Теперь по очереди строим шесть наборов кандидатов.

Логика такая:

- каждое правило отвечает за свой тип случаев
- потом мы объединим все пары в один общий кандидатный список
- если одна и та же пара нашлась по нескольким правилам, это даже хорошо: значит, она выглядит более правдоподобной
- для новых правил отдельно считаем вклад в union, чтобы понять, какой реальный прирост они дают, а не просто сколько сырых пар они нагенерировали

Ещё один важный фильтр:

Если у пары с обеих сторон заполнен `grade`, и эти `grade` разные,
такую пару сразу выбрасываем.

Смотрим на `grade` именно как на ограничитель,
потому что он часто несёт очень конкретный смысл:
возраст, диапазон, размер креветок, число sheets и так далее.


In [102]:
PAIR_COLUMNS = [
    'item_id', 'product_name', 'product_core_block', 'residual_block',
    'brand_block', 'product_type_block', 'size_key', 'size_unit_block', 'grade_block'
]


def make_pair_frame(df, id_col):
    # Оставляем только нужные колонки для candidate pairs,
    # чтобы merge был проще и понятнее.
    return (
        df[[
            id_col, 'product_name', 'product_core_block', 'residual_block',
            'brand_block', 'product_type_block', 'size_key', 'size_unit_block', 'grade_block'
        ]]
        .rename(columns={id_col: 'item_id'})
        .copy()
    )


sk_pairs = make_pair_frame(sk, 'sk_item_id')
alpha_pairs = make_pair_frame(alpha, 'am_item_id')


def apply_grade_filter(pairs):
    # Если grade заполнен только с одной стороны, пару пока оставляем.
    # Если grade заполнен с обеих сторон и он разный, пару убираем.
    same_or_missing_grade = (
        pairs['grade_block_sk'].eq('')
        | pairs['grade_block_am'].eq('')
        | pairs['grade_block_sk'].eq(pairs['grade_block_am'])
    )
    return pairs.loc[same_or_missing_grade].copy()


def apply_size_unit_filter(pairs):
    # Для rescue-блоков дополнительно смотрим единицу измерения.
    # Если unit известен у обеих сторон и он разный, такую пару убираем.
    same_or_missing_unit = (
        pairs['size_unit_block_sk'].eq('')
        | pairs['size_unit_block_am'].eq('')
        | pairs['size_unit_block_sk'].eq(pairs['size_unit_block_am'])
    )
    return pairs.loc[same_or_missing_unit].copy()


def finalize_rule_pairs(pairs, rule_name, shared_token_column=''):
    # Приводим пару к общему формату и подписываем, по какому правилу она нашлась.
    # Если часть ключей пришла из merge без суффиксов, дублируем их в обе стороны,
    # чтобы дальше у всех правил был единый формат колонок.
    for column_name in ['brand_block', 'product_type_block', 'size_key']:
        if column_name in pairs.columns:
            if f'{column_name}_sk' not in pairs.columns:
                pairs[f'{column_name}_sk'] = pairs[column_name]
            if f'{column_name}_am' not in pairs.columns:
                pairs[f'{column_name}_am'] = pairs[column_name]

    pairs = apply_grade_filter(pairs)
    pairs['blocking_rule'] = rule_name

    if shared_token_column == '':
        pairs['shared_rescue_tokens'] = ''
    else:
        pairs['shared_rescue_tokens'] = pairs[shared_token_column].astype(str)

    return pairs[[
        'item_id_sk', 'item_id_am', 'blocking_rule', 'shared_rescue_tokens',
        'product_name_sk', 'product_name_am',
        'product_core_block_sk', 'product_core_block_am',
        'brand_block_sk', 'brand_block_am',
        'product_type_block_sk', 'product_type_block_am',
        'size_key_sk', 'size_key_am',
        'grade_block_sk', 'grade_block_am'
    ]].copy()


def build_known_brand_frame(df, item_col):
    # Здесь оставляем строки, где уже найден бренд.
    # Это будет якорная сторона для мягких брендовых rescue-блоков.
    known = df.loc[
        df['brand_block'].ne(''),
        [
            item_col, 'product_name', 'product_core_block', 'residual_block',
            'brand_block', 'product_type_block', 'size_key', 'size_unit_block', 'grade_block'
        ]
    ].rename(columns={item_col: 'item_id'}).copy()

    known['brand_first_char'] = known['brand_block'].apply(first_letter_key)
    return known.loc[known['brand_first_char'].ne('')].copy()


def build_brand_phrase_frame(df, item_col):
    # Здесь разворачиваем короткие brand-like куски строки,
    # которые потом сравниваем с уже найденным брендом второй стороны.
    phrase_frame = df.loc[
        df['brand_candidate_phrases'].map(len).gt(0),
        [
            item_col, 'product_name', 'product_core_block', 'residual_block',
            'brand_block', 'product_type_block', 'size_key', 'size_unit_block', 'grade_block', 'brand_candidate_phrases'
        ]
    ].explode('brand_candidate_phrases').copy()

    phrase_frame = phrase_frame.rename(columns={
        item_col: 'item_id',
        'brand_candidate_phrases': 'brand_candidate_phrase'
    })
    phrase_frame['brand_candidate_phrase'] = phrase_frame['brand_candidate_phrase'].map(normalize_text)
    phrase_frame['brand_candidate_first_char'] = phrase_frame['brand_candidate_phrase'].apply(first_letter_key)

    return phrase_frame.loc[
        phrase_frame['brand_candidate_phrase'].ne('')
        & phrase_frame['brand_candidate_first_char'].ne('')
    ].copy()


def format_brand_similarity_signal(known_brand, candidate_phrase, similarity_score):
    # Сохраняем в явном виде, какой бренд с каким куском строки совпал.
    return f'{known_brand} ~ {candidate_phrase} ({similarity_score:.2f})'


def build_similar_brand_pairs(
    known_brand_frame,
    brand_phrase_frame,
    left_on,
    right_on,
    rule_name,
    known_store_prefix,
    phrase_store_prefix
):
    # Строим пары по похожему бренду, а точный якорь задаём через join-ключи.
    # Здесь отдельно следим, какая сторона на самом деле Sklavenitis, а какая Alpha Omega,
    # чтобы не перепутать store-суффиксы в итоговых candidate pairs.
    pairs = known_brand_frame.merge(
        brand_phrase_frame,
        left_on=left_on,
        right_on=right_on,
        how='inner',
        suffixes=('_left', '_right')
    )

    pairs['brand_similarity_score'] = pairs.apply(
        lambda row: brand_similarity_score(row['brand_block_left'], row['brand_candidate_phrase']),
        axis=1
    )
    pairs = pairs.loc[
        pairs['brand_similarity_score'] >= BRAND_SIMILARITY_THRESHOLD
    ].copy()

    pairs['brand_similarity_signal'] = pairs.apply(
        lambda row: format_brand_similarity_signal(
            row['brand_block_left'],
            row['brand_candidate_phrase'],
            row['brand_similarity_score']
        ),
        axis=1
    )

    rename_bases = [
        'item_id', 'product_name', 'product_core_block', 'residual_block',
        'brand_block', 'product_type_block', 'size_key', 'pack_count_block',
        'unit_size_key', 'size_unit_block', 'grade_block'
    ]
    rename_map = {}

    for base_name in rename_bases:
        left_name = f'{base_name}_left'
        right_name = f'{base_name}_right'

        if left_name in pairs.columns:
            rename_map[left_name] = f'{base_name}_{known_store_prefix}'
        if right_name in pairs.columns:
            rename_map[right_name] = f'{base_name}_{phrase_store_prefix}'

    pairs = pairs.rename(columns=rename_map)

    return finalize_rule_pairs(
        pairs,
        rule_name,
        shared_token_column='brand_similarity_signal'
    )


# Правило 1. Смотрим exact product_type + exact size,
# потому что это самый сильный структурный блок для стартового MVP.
rule_1_left = sk_pairs.loc[
    sk_pairs['product_type_block'].ne('') & sk_pairs['size_key'].ne('')
].copy()
rule_1_right = alpha_pairs.loc[
    alpha_pairs['product_type_block'].ne('') & alpha_pairs['size_key'].ne('')
].copy()

rule_1_pairs = rule_1_left.merge(
    rule_1_right,
    on=['product_type_block', 'size_key'],
    how='inner',
    suffixes=('_sk', '_am')
)
rule_1_pairs = finalize_rule_pairs(rule_1_pairs, 'product_type + exact_size')


# Правило 2. Смотрим brand + product_type,
# потому что если нашли оба поля, это обычно очень чистая подгруппа.
rule_2_left = sk_pairs.loc[
    sk_pairs['brand_block'].ne('') & sk_pairs['product_type_block'].ne('')
].copy()
rule_2_right = alpha_pairs.loc[
    alpha_pairs['brand_block'].ne('') & alpha_pairs['product_type_block'].ne('')
].copy()

rule_2_pairs = rule_2_left.merge(
    rule_2_right,
    on=['brand_block', 'product_type_block'],
    how='inner',
    suffixes=('_sk', '_am')
)
rule_2_pairs = finalize_rule_pairs(rule_2_pairs, 'brand + product_type')


# Правило 3. Смотрим brand + exact size,
# потому что это fallback для случаев, где product_type ещё не нашли,
# но бренд и фасовка уже выглядят многообещающе.
rule_3_left = sk_pairs.loc[
    sk_pairs['brand_block'].ne('') & sk_pairs['size_key'].ne('')
].copy()
rule_3_right = alpha_pairs.loc[
    alpha_pairs['brand_block'].ne('') & alpha_pairs['size_key'].ne('')
].copy()

rule_3_pairs = rule_3_left.merge(
    rule_3_right,
    on=['brand_block', 'size_key'],
    how='inner',
    suffixes=('_sk', '_am')
)
rule_3_pairs = finalize_rule_pairs(rule_3_pairs, 'brand + exact_size')


# Правило 4. Rescue-блок по product_type и общему informative token.
# Он помогает вернуть часть кандидатов там, где brand или size не сработали.
rule_4_left = sk.loc[
    sk['product_type_block'].ne('') & sk['informative_tokens'].map(len).gt(0)
].copy()
rule_4_right = alpha.loc[
    alpha['product_type_block'].ne('') & alpha['informative_tokens'].map(len).gt(0)
].copy()

rule_4_left = rule_4_left[[
    'sk_item_id', 'product_name', 'product_core_block', 'brand_block',
    'product_type_block', 'size_key', 'pack_count_block', 'unit_size_key',
    'size_unit_block', 'grade_block', 'informative_tokens'
]].explode('informative_tokens')

rule_4_right = rule_4_right[[
    'am_item_id', 'product_name', 'product_core_block', 'brand_block',
    'product_type_block', 'size_key', 'pack_count_block', 'unit_size_key',
    'size_unit_block', 'grade_block', 'informative_tokens'
]].explode('informative_tokens')

rule_4_pairs_raw = rule_4_left.merge(
    rule_4_right,
    on=['product_type_block', 'informative_tokens'],
    how='inner',
    suffixes=('_sk', '_am')
)
rule_4_pairs_raw = apply_size_unit_filter(rule_4_pairs_raw)

rule_4_pairs_raw = rule_4_pairs_raw.rename(columns={
    'sk_item_id': 'item_id_sk',
    'am_item_id': 'item_id_am',
    'informative_tokens': 'shared_rescue_token'
})

# После merge ключевой product_type лежит в одной колонке.
# Для удобства дальше дублируем его в обе стороны.
rule_4_pairs_raw['product_type_block_sk'] = rule_4_pairs_raw['product_type_block']
rule_4_pairs_raw['product_type_block_am'] = rule_4_pairs_raw['product_type_block']

rule_4_pairs = finalize_rule_pairs(
    rule_4_pairs_raw,
    'product_type + informative_token',
    shared_token_column='shared_rescue_token'
)


# Правило 5. Rescue-блок по одинаковому product_type и похожему брендовому фрагменту.
# Здесь не считаем, что весь остаток строки должен быть брендом.
# Наоборот: берём только короткие brand-like куски строки и сравниваем их
# с уже найденным брендом второй стороны.
known_brand_sk = build_known_brand_frame(sk, 'sk_item_id')
known_brand_am = build_known_brand_frame(alpha, 'am_item_id')
brand_phrase_sk = build_brand_phrase_frame(sk, 'sk_item_id')
brand_phrase_am = build_brand_phrase_frame(alpha, 'am_item_id')

rule_5_pairs_from_sk = build_similar_brand_pairs(
    known_brand_sk.loc[known_brand_sk['product_type_block'].ne('')].copy(),
    brand_phrase_am.loc[brand_phrase_am['product_type_block'].ne('')].copy(),
    left_on=['product_type_block', 'brand_first_char'],
    right_on=['product_type_block', 'brand_candidate_first_char'],
    rule_name='product_type + similar_brand_fragment',
    known_store_prefix='sk',
    phrase_store_prefix='am'
)

rule_5_pairs_from_am = build_similar_brand_pairs(
    known_brand_am.loc[known_brand_am['product_type_block'].ne('')].copy(),
    brand_phrase_sk.loc[brand_phrase_sk['product_type_block'].ne('')].copy(),
    left_on=['product_type_block', 'brand_first_char'],
    right_on=['product_type_block', 'brand_candidate_first_char'],
    rule_name='product_type + similar_brand_fragment',
    known_store_prefix='am',
    phrase_store_prefix='sk'
)

rule_5_pairs = pd.concat([rule_5_pairs_from_sk, rule_5_pairs_from_am], ignore_index=True)
rule_5_pairs = rule_5_pairs.drop_duplicates(
    subset=['item_id_sk', 'item_id_am', 'blocking_rule', 'shared_rescue_tokens']
).copy()


# Правило 6. Похожий брендовый фрагмент + exact size.
# Это помогает там, где product_type не совпал или ещё не найден,
# но общий размер уже хорошо выровнял пространство сравнения.
rule_6_pairs_from_sk = build_similar_brand_pairs(
    known_brand_sk.loc[known_brand_sk['size_key'].ne('')].copy(),
    brand_phrase_am.loc[brand_phrase_am['size_key'].ne('')].copy(),
    left_on=['size_key', 'brand_first_char'],
    right_on=['size_key', 'brand_candidate_first_char'],
    rule_name='similar_brand_fragment + exact_size',
    known_store_prefix='sk',
    phrase_store_prefix='am'
)

rule_6_pairs_from_am = build_similar_brand_pairs(
    known_brand_am.loc[known_brand_am['size_key'].ne('')].copy(),
    brand_phrase_sk.loc[brand_phrase_sk['size_key'].ne('')].copy(),
    left_on=['size_key', 'brand_first_char'],
    right_on=['size_key', 'brand_candidate_first_char'],
    rule_name='similar_brand_fragment + exact_size',
    known_store_prefix='am',
    phrase_store_prefix='sk'
)

rule_6_pairs = pd.concat([rule_6_pairs_from_sk, rule_6_pairs_from_am], ignore_index=True)
rule_6_pairs = rule_6_pairs.drop_duplicates(
    subset=['item_id_sk', 'item_id_am', 'blocking_rule', 'shared_rescue_tokens']
).copy()



rule_frames = {
    'product_type + exact_size': rule_1_pairs,
    'brand + product_type': rule_2_pairs,
    'brand + exact_size': rule_3_pairs,
    'product_type + informative_token': rule_4_pairs,
    'product_type + similar_brand_fragment': rule_5_pairs,
    'similar_brand_fragment + exact_size': rule_6_pairs
}

for rule_name, frame in rule_frames.items():
    print(rule_name, '-> pairs:', len(frame))


product_type + exact_size -> pairs: 31013
brand + product_type -> pairs: 6766
brand + exact_size -> pairs: 6370
product_type + informative_token -> pairs: 26296
product_type + similar_brand_fragment -> pairs: 8891
similar_brand_fragment + exact_size -> pairs: 7742


### Объединяем все candidate pairs и считаем сводку

Теперь объединяем пары из всех правил.

Здесь уже смотрим:

- сколько уникальных пар получилось всего
- сколько товаров `Sklavenitis` получили хотя бы одного кандидата
- сколько товаров `Alpha Omega` попали хотя бы в одну кандидатную пару
- сколько кандидатов в среднем приходится на один товар `Sklavenitis`

Это и есть первый sanity check для blocking.

Если кандидатов слишком мало, мы рискуем терять настоящие пары.
Если кандидатов слишком много, значит blocking пока режет пространство слишком слабо.


In [103]:
all_rule_pairs = pd.concat(list(rule_frames.values()), ignore_index=True)

# Если одна и та же пара нашлась по нескольким правилам,
# это полезная информация, поэтому не выбрасываем её молча,
# а собираем список правил в одну строку.
candidate_pairs = (
    all_rule_pairs
    .groupby(['item_id_sk', 'item_id_am'], as_index=False)
    .agg(
        blocking_rules=('blocking_rule', lambda s: ' | '.join(sorted(set(s)))),
        shared_rescue_tokens=('shared_rescue_tokens', lambda s: ' | '.join(sorted({token for token in s if token}))),
        product_name_sk=('product_name_sk', 'first'),
        product_name_am=('product_name_am', 'first'),
        product_core_block_sk=('product_core_block_sk', 'first'),
        product_core_block_am=('product_core_block_am', 'first'),
        brand_block_sk=('brand_block_sk', 'first'),
        brand_block_am=('brand_block_am', 'first'),
        product_type_block_sk=('product_type_block_sk', 'first'),
        product_type_block_am=('product_type_block_am', 'first'),
        size_key_sk=('size_key_sk', 'first'),
        size_key_am=('size_key_am', 'first'),
        grade_block_sk=('grade_block_sk', 'first'),
        grade_block_am=('grade_block_am', 'first')
    )
)

# Для сводки считаем coverage блокинга относительно товаров Sklavenitis,
# потому что пока строим кандидатов от одного магазина к другому.
sk_candidate_counts = (
    candidate_pairs.groupby('item_id_sk', as_index=False)
    .agg(candidate_count=('item_id_am', 'size'))
)

alpha_candidate_counts = (
    candidate_pairs.groupby('item_id_am', as_index=False)
    .agg(candidate_count=('item_id_sk', 'size'))
)

summary_rows = []

for rule_name, frame in rule_frames.items():
    summary_rows.append({
        'scope': 'single_rule',
        'rule_name': rule_name,
        'pair_count': int(len(frame)),
        'unique_sklavenitis_items': int(frame['item_id_sk'].nunique()),
        'unique_alphamega_items': int(frame['item_id_am'].nunique())
    })

# Здесь считаем не сырой размер правила, а его реальный вклад в union.
# Так видно, сколько новых пар и новых товаров правило принесло поверх уже добавленных правил.
seen_pairs = set()
seen_sklavenitis = set()
seen_alphamega = set()

for rule_name, frame in rule_frames.items():
    pair_keys = set(zip(frame['item_id_sk'], frame['item_id_am']))
    current_sklavenitis = set(frame['item_id_sk'])
    current_alphamega = set(frame['item_id_am'])

    new_pair_keys = pair_keys - seen_pairs
    new_sklavenitis = current_sklavenitis - seen_sklavenitis
    new_alphamega = current_alphamega - seen_alphamega

    summary_rows.append({
        'scope': 'incremental_union',
        'rule_name': rule_name,
        'pair_count': int(len(new_pair_keys)),
        'unique_sklavenitis_items': int(len(new_sklavenitis)),
        'unique_alphamega_items': int(len(new_alphamega))
    })

    seen_pairs |= pair_keys
    seen_sklavenitis |= current_sklavenitis
    seen_alphamega |= current_alphamega

summary_rows.extend([
    {
        'scope': 'union',
        'rule_name': 'all_rules_union',
        'pair_count': int(len(candidate_pairs)),
        'unique_sklavenitis_items': int(candidate_pairs['item_id_sk'].nunique()),
        'unique_alphamega_items': int(candidate_pairs['item_id_am'].nunique())
    },
    {
        'scope': 'union_metrics',
        'rule_name': 'avg_candidates_per_sklavenitis_item',
        'pair_count': float(sk_candidate_counts['candidate_count'].mean()) if len(sk_candidate_counts) else 0.0,
        'unique_sklavenitis_items': int(sk_candidate_counts['item_id_sk'].nunique()),
        'unique_alphamega_items': np.nan
    },
    {
        'scope': 'union_metrics',
        'rule_name': 'median_candidates_per_sklavenitis_item',
        'pair_count': float(sk_candidate_counts['candidate_count'].median()) if len(sk_candidate_counts) else 0.0,
        'unique_sklavenitis_items': int(sk_candidate_counts['item_id_sk'].nunique()),
        'unique_alphamega_items': np.nan
    }
])

blocking_summary = pd.DataFrame(summary_rows)

candidate_pairs.to_csv(CANDIDATE_PAIRS_PATH, index=False, encoding='utf-8-sig')
blocking_summary.to_csv(CANDIDATE_SUMMARY_PATH, index=False, encoding='utf-8-sig')

print('saved:', CANDIDATE_PAIRS_PATH)
print('saved:', CANDIDATE_SUMMARY_PATH)
print('')
print('Уникальных candidate pairs:', len(candidate_pairs))
print('Sklavenitis rows with at least one candidate:', candidate_pairs['item_id_sk'].nunique())
print('Alpha Omega rows with at least one candidate:', candidate_pairs['item_id_am'].nunique())
print('')
print(blocking_summary)


saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_v1.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_blocking_summary_v1.csv

Уникальных candidate pairs: 53317
Sklavenitis rows with at least one candidate: 2484
Alpha Omega rows with at least one candidate: 5314

                scope                               rule_name    pair_count  \
0         single_rule               product_type + exact_size  31013.000000   
1         single_rule                    brand + product_type   6766.000000   
2         single_rule                      brand + exact_size   6370.000000   
3         single_rule        product_type + informative_token  26296.000000   
4         single_rule   product_type + similar_brand_fragment   8891.000000   
5         single_rule     similar_brand_fragment + exact_size   7742.000000   
6   incremental_union               pr

### 25.2 Candidate blocking v2: TF-IDF rescue rules

Теперь добавляем `candidate blocking v2`.

Идея такая: v1 оставляем как базовый, более структурный blocking. Но сверху добавляем два rescue-правила по TF-IDF, чтобы вернуть пары, которые могли потеряться из-за пустого/неидеального `brand` или `product_type`.

Добавляем два правила:

1. `tfidf_0_60 + size_similarity_0_70`
Берем пару, если хотя бы один TF-IDF score высокий:

```text
tfidf_word_1_2_similarity >= 0.60
или
tfidf_char_3_5_similarity >= 0.60
```

и при этом размер достаточно близкий:

```text
size_similarity >= 0.70
```

2. `tfidf_0_85_any_size`
Берем пару, если хотя бы один TF-IDF score очень высокий. Размер здесь не ограничиваем: правило срабатывает всегда, если текстовая похожесть достаточно сильная.

```text
tfidf_word_1_2_similarity >= 0.85
или
tfidf_char_3_5_similarity >= 0.85
```

После этого объединяем v1 и новые TF-IDF rescue-пары в один `candidate_pairs_v2`.

Важно: v1-файлы не перезаписываем. v2 сохраняем отдельно:

- `sklavenitis_alphamega_candidate_pairs_v2.csv`
- `sklavenitis_alphamega_candidate_blocking_summary_v2.csv`

После этой ячейки переменная `candidate_pairs` указывает уже на v2, поэтому следующий scoring-блок будет считать score по расширенному blocking.


In [104]:
from sklearn.feature_extraction.text import TfidfVectorizer

CANDIDATE_PAIRS_V2_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_v2.csv'
CANDIDATE_SUMMARY_V2_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_blocking_summary_v2.csv'

TFIDF_SIZE_THRESHOLD = 0.60
TFIDF_MISSING_SIZE_THRESHOLD = 0.85
SIZE_SIMILARITY_RESCUE_THRESHOLD = 0.70
TFIDF_BATCH_SIZE = 100


def parse_blocking_size_key(size_key):
    text = normalize_text(size_key)
    if text == '' or '_' not in text:
        return np.nan, ''

    number_text, unit = text.split('_', 1)
    number = pd.to_numeric(pd.Series([number_text]), errors='coerce').iloc[0]
    if pd.isna(number) or unit == '':
        return np.nan, ''

    return float(number), unit


def blocking_size_similarity(left_size_key, right_size_key):
    left_size, left_unit = parse_blocking_size_key(left_size_key)
    right_size, right_unit = parse_blocking_size_key(right_size_key)

    if pd.isna(left_size) or pd.isna(right_size):
        return 0.5

    if left_unit == '' or right_unit == '':
        return 0.5

    if left_unit != right_unit:
        return 0.05

    max_size = max(left_size, right_size)
    if max_size == 0:
        return 0.5

    diff = abs(left_size - right_size) / max_size

    if diff == 0:
        return 1.0
    if diff <= 0.03:
        return 0.9
    if diff <= 0.10:
        return 0.7
    if diff <= 0.25:
        return 0.4
    return 0.05


def has_missing_size(size_key):
    return normalize_text(size_key) == ''


TFIDF_RESCUE_PAIR_COLUMNS = [
    'item_id_sk', 'item_id_am', 'blocking_rule', 'shared_rescue_tokens',
    'product_name_sk', 'product_name_am',
    'product_core_block_sk', 'product_core_block_am',
    'brand_block_sk', 'brand_block_am',
    'product_type_block_sk', 'product_type_block_am',
    'size_key_sk', 'size_key_am',
    'grade_block_sk', 'grade_block_am'
]


def make_empty_tfidf_rescue_frame():
    return pd.DataFrame(columns=TFIDF_RESCUE_PAIR_COLUMNS)


def should_keep_by_grade(sk_row, alpha_row):
    grade_sk = normalize_text(sk_row.get('grade_block', ''))
    grade_am = normalize_text(alpha_row.get('grade_block', ''))
    return grade_sk == '' or grade_am == '' or grade_sk == grade_am


def make_tfidf_rescue_row(sk_row, alpha_row, rule_name, word_similarity, char_similarity, rescue_similarity, size_similarity_value):
    return {
        'item_id_sk': sk_row['sk_item_id'],
        'item_id_am': alpha_row['am_item_id'],
        'blocking_rule': rule_name,
        'shared_rescue_tokens': (
            f"tfidf={rescue_similarity:.3f}; "
            f"word={word_similarity:.3f}; "
            f"char={char_similarity:.3f}; "
            f"size={size_similarity_value:.2f}"
        ),
        'product_name_sk': sk_row.get('product_name', ''),
        'product_name_am': alpha_row.get('product_name', ''),
        'product_core_block_sk': sk_row.get('product_core_block', ''),
        'product_core_block_am': alpha_row.get('product_core_block', ''),
        'brand_block_sk': sk_row.get('brand_block', ''),
        'brand_block_am': alpha_row.get('brand_block', ''),
        'product_type_block_sk': sk_row.get('product_type_block', ''),
        'product_type_block_am': alpha_row.get('product_type_block', ''),
        'size_key_sk': sk_row.get('size_key', ''),
        'size_key_am': alpha_row.get('size_key', ''),
        'grade_block_sk': sk_row.get('grade_block', ''),
        'grade_block_am': alpha_row.get('grade_block', ''),
    }


def threshold_sparse_similarity(matrix, threshold):
    matrix = matrix.tocsr(copy=True)
    matrix.data[matrix.data < threshold] = 0
    matrix.eliminate_zeros()
    return matrix


def build_tfidf_rescue_pairs():
    sk_texts = sk['product_core_block'].fillna('').astype(str).map(normalize_text)
    alpha_texts = alpha['product_core_block'].fillna('').astype(str).map(normalize_text)
    all_texts = pd.concat([sk_texts, alpha_texts], ignore_index=True)

    word_vectorizer_v2 = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        token_pattern=r'(?u)\b\w+\b',
        lowercase=False
    )
    char_vectorizer_v2 = TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        lowercase=False
    )

    word_vectorizer_v2.fit(all_texts)
    char_vectorizer_v2.fit(all_texts)

    sk_word_vectors = word_vectorizer_v2.transform(sk_texts)
    alpha_word_vectors = word_vectorizer_v2.transform(alpha_texts)
    sk_char_vectors = char_vectorizer_v2.transform(sk_texts)
    alpha_char_vectors = char_vectorizer_v2.transform(alpha_texts)

    size_rule_rows = []
    missing_size_rule_rows = []
    seen_size_rule_pairs = set()
    seen_missing_size_rule_pairs = set()

    for batch_start in range(0, len(sk), TFIDF_BATCH_SIZE):
        batch_end = min(batch_start + TFIDF_BATCH_SIZE, len(sk))

        word_similarity = sk_word_vectors[batch_start:batch_end] @ alpha_word_vectors.T
        char_similarity = sk_char_vectors[batch_start:batch_end] @ alpha_char_vectors.T
        rescue_similarity = word_similarity.maximum(char_similarity)
        rescue_similarity = threshold_sparse_similarity(rescue_similarity, TFIDF_SIZE_THRESHOLD)

        for local_sk_position, alpha_position, rescue_value in zip(
            rescue_similarity.tocoo().row,
            rescue_similarity.tocoo().col,
            rescue_similarity.tocoo().data
        ):
            sk_position = batch_start + local_sk_position
            sk_row = sk.iloc[sk_position]
            alpha_row = alpha.iloc[alpha_position]

            if not should_keep_by_grade(sk_row, alpha_row):
                continue

            pair_key = (sk_row['sk_item_id'], alpha_row['am_item_id'])
            word_value = float(word_similarity[local_sk_position, alpha_position])
            char_value = float(char_similarity[local_sk_position, alpha_position])
            size_similarity_value = blocking_size_similarity(
                sk_row.get('size_key', ''),
                alpha_row.get('size_key', '')
            )

            if size_similarity_value >= SIZE_SIMILARITY_RESCUE_THRESHOLD:
                if pair_key not in seen_size_rule_pairs:
                    size_rule_rows.append(
                        make_tfidf_rescue_row(
                            sk_row,
                            alpha_row,
                            'tfidf_0_60 + size_similarity_0_70',
                            word_value,
                            char_value,
                            float(rescue_value),
                            size_similarity_value
                        )
                    )
                    seen_size_rule_pairs.add(pair_key)

            if rescue_value >= TFIDF_MISSING_SIZE_THRESHOLD:
                if pair_key not in seen_missing_size_rule_pairs:
                    missing_size_rule_rows.append(
                        make_tfidf_rescue_row(
                            sk_row,
                            alpha_row,
                            'tfidf_0_85_any_size',
                            word_value,
                            char_value,
                            float(rescue_value),
                            size_similarity_value
                        )
                    )
                    seen_missing_size_rule_pairs.add(pair_key)

    size_rule_frame = (
        pd.DataFrame(size_rule_rows)
        if size_rule_rows
        else make_empty_tfidf_rescue_frame()
    )
    missing_size_rule_frame = (
        pd.DataFrame(missing_size_rule_rows)
        if missing_size_rule_rows
        else make_empty_tfidf_rescue_frame()
    )

    return size_rule_frame, missing_size_rule_frame


candidate_pairs_v1 = candidate_pairs.copy()
blocking_summary_v1 = blocking_summary.copy()

rule_7_pairs, rule_8_pairs = build_tfidf_rescue_pairs()

rule_frames_v2 = dict(rule_frames)
rule_frames_v2['tfidf_0_60 + size_similarity_0_70'] = rule_7_pairs
rule_frames_v2['tfidf_0_85_any_size'] = rule_8_pairs

all_rule_pairs_v2 = pd.concat(list(rule_frames_v2.values()), ignore_index=True)

candidate_pairs_v2 = (
    all_rule_pairs_v2
    .groupby(['item_id_sk', 'item_id_am'], as_index=False)
    .agg(
        blocking_rules=('blocking_rule', lambda s: ' | '.join(sorted(set(s)))),
        shared_rescue_tokens=('shared_rescue_tokens', lambda s: ' | '.join(sorted({token for token in s if token}))),
        product_name_sk=('product_name_sk', 'first'),
        product_name_am=('product_name_am', 'first'),
        product_core_block_sk=('product_core_block_sk', 'first'),
        product_core_block_am=('product_core_block_am', 'first'),
        brand_block_sk=('brand_block_sk', 'first'),
        brand_block_am=('brand_block_am', 'first'),
        product_type_block_sk=('product_type_block_sk', 'first'),
        product_type_block_am=('product_type_block_am', 'first'),
        size_key_sk=('size_key_sk', 'first'),
        size_key_am=('size_key_am', 'first'),
        grade_block_sk=('grade_block_sk', 'first'),
        grade_block_am=('grade_block_am', 'first')
    )
)

summary_rows_v2 = []
seen_pairs_v2 = set()
seen_sklavenitis_v2 = set()
seen_alphamega_v2 = set()

for rule_name, frame in rule_frames_v2.items():
    pair_keys = set(zip(frame['item_id_sk'], frame['item_id_am']))
    current_sklavenitis = set(frame['item_id_sk'])
    current_alphamega = set(frame['item_id_am'])

    summary_rows_v2.append({
        'scope': 'single_rule',
        'rule_name': rule_name,
        'pair_count': int(len(frame)),
        'unique_sklavenitis_items': int(frame['item_id_sk'].nunique()),
        'unique_alphamega_items': int(frame['item_id_am'].nunique())
    })

    new_pair_keys = pair_keys - seen_pairs_v2
    new_sklavenitis = current_sklavenitis - seen_sklavenitis_v2
    new_alphamega = current_alphamega - seen_alphamega_v2

    summary_rows_v2.append({
        'scope': 'incremental_union',
        'rule_name': rule_name,
        'pair_count': int(len(new_pair_keys)),
        'unique_sklavenitis_items': int(len(new_sklavenitis)),
        'unique_alphamega_items': int(len(new_alphamega))
    })

    seen_pairs_v2 |= pair_keys
    seen_sklavenitis_v2 |= current_sklavenitis
    seen_alphamega_v2 |= current_alphamega

sk_candidate_counts_v2 = (
    candidate_pairs_v2.groupby('item_id_sk', as_index=False)
    .agg(candidate_count=('item_id_am', 'size'))
)

summary_rows_v2.extend([
    {
        'scope': 'union',
        'rule_name': 'all_rules_union_v2',
        'pair_count': int(len(candidate_pairs_v2)),
        'unique_sklavenitis_items': int(candidate_pairs_v2['item_id_sk'].nunique()),
        'unique_alphamega_items': int(candidate_pairs_v2['item_id_am'].nunique())
    },
    {
        'scope': 'delta_vs_v1',
        'rule_name': 'new_pairs_added_by_v2',
        'pair_count': int(
            len(set(zip(candidate_pairs_v2['item_id_sk'], candidate_pairs_v2['item_id_am']))
                - set(zip(candidate_pairs_v1['item_id_sk'], candidate_pairs_v1['item_id_am'])))
        ),
        'unique_sklavenitis_items': int(
            len(set(candidate_pairs_v2['item_id_sk']) - set(candidate_pairs_v1['item_id_sk']))
        ),
        'unique_alphamega_items': int(
            len(set(candidate_pairs_v2['item_id_am']) - set(candidate_pairs_v1['item_id_am']))
        )
    },
    {
        'scope': 'union_metrics',
        'rule_name': 'avg_candidates_per_sklavenitis_item_v2',
        'pair_count': float(sk_candidate_counts_v2['candidate_count'].mean()) if len(sk_candidate_counts_v2) else 0.0,
        'unique_sklavenitis_items': int(sk_candidate_counts_v2['item_id_sk'].nunique()),
        'unique_alphamega_items': np.nan
    },
    {
        'scope': 'union_metrics',
        'rule_name': 'median_candidates_per_sklavenitis_item_v2',
        'pair_count': float(sk_candidate_counts_v2['candidate_count'].median()) if len(sk_candidate_counts_v2) else 0.0,
        'unique_sklavenitis_items': int(sk_candidate_counts_v2['item_id_sk'].nunique()),
        'unique_alphamega_items': np.nan
    }
])

blocking_summary_v2 = pd.DataFrame(summary_rows_v2)

candidate_pairs_v2.to_csv(CANDIDATE_PAIRS_V2_PATH, index=False, encoding='utf-8-sig')
blocking_summary_v2.to_csv(CANDIDATE_SUMMARY_V2_PATH, index=False, encoding='utf-8-sig')

# Дальше scoring-блок будет использовать уже v2.
candidate_pairs = candidate_pairs_v2
blocking_summary = blocking_summary_v2

print('saved:', CANDIDATE_PAIRS_V2_PATH)
print('saved:', CANDIDATE_SUMMARY_V2_PATH)
print('')
print('Candidate pairs v1:', len(candidate_pairs_v1))
print('Candidate pairs v2:', len(candidate_pairs_v2))
print('New pairs added by v2:', len(candidate_pairs_v2) - len(candidate_pairs_v1))
print('')
print('New TF-IDF rescue rule sizes:')
print('tfidf_0_60 + size_similarity_0_70:', len(rule_7_pairs))
print('tfidf_0_85_any_size:', len(rule_8_pairs))
print('')
display(blocking_summary_v2)


saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_v2.csv
saved: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_blocking_summary_v2.csv

Candidate pairs v1: 53317
Candidate pairs v2: 53436
New pairs added by v2: 119

New TF-IDF rescue rule sizes:
tfidf_0_60 + size_similarity_0_70: 1515
tfidf_0_85_any_size: 616



,scope,rule_name,pair_count,unique_sklavenitis_items,unique_alphamega_items
0,single_rule,product_type + exact_size,31013.000000,1680,3330.0
1,incremental_union,product_type + exact_size,31013.000000,1680,3330.0
2,single_rule,brand + product_type,6766.000000,1182,1904.0
3,incremental_union,brand + product_type,4178.000000,209,488.0
4,single_rule,brand + exact_size,6370.000000,1369,2012.0
5,incremental_union,brand + exact_size,3782.000000,295,548.0
6,single_rule,product_type + informative_token,26296.000000,1762,3745.0
7,incremental_union,product_type + informative_token,13344.000000,242,865.0
8,single_rule,product_type + similar_brand_fragment,8891.000000,1307,2096.0
9,incremental_union,product_type + similar_brand_fragment,489.000000,24,28.0


In [105]:
candidate_pairs.head(20)


,item_id_sk,item_id_am,blocking_rules,shared_rescue_tokens,product_name_sk,product_name_am,product_core_block_sk,product_core_block_am,brand_block_sk,brand_block_am,product_type_block_sk,product_type_block_am,size_key_sk,size_key_am,grade_block_sk,grade_block_am
0,sk_0,am_6289,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Barilla Tomato with Basil Sauce 400 g,spaghetti with tomato sauce sklavenitis production,barilla tomato with basil sauce,sklavenitis,barilla,sauce,sauce,480_g,400_g,,
1,sk_0,am_6467,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Barilla Arrabbiata Tomato Sauce With Chilli 400 g,spaghetti with tomato sauce sklavenitis production,barilla arrabbiata tomato sauce with chilli,sklavenitis,barilla,sauce,sauce,480_g,400_g,,
2,sk_0,am_6677,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Heinz Baked Beanz In a Rich Tomato Sauce No Added Sugar 415 g,spaghetti with tomato sauce sklavenitis production,heinz baked beanz in a rich tomato sauce no added sugar,sklavenitis,heinz,sauce,sauce,480_g,415_g,,
3,sk_0,am_7220,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Heinz Hoops Shaped Paste in a Juicy Tomato Sauce 400 g,spaghetti with tomato sauce sklavenitis production,heinz hoops shaped paste in a juicy tomato sauce,sklavenitis,heinz,sauce,sauce,480_g,400_g,,
4,sk_0,am_7353,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Bertolli Tomato Sauce with Basil 400 g,spaghetti with tomato sauce sklavenitis production,bertolli tomato sauce with basil,sklavenitis,bertolli,sauce,sauce,480_g,400_g,,
5,sk_0,am_7610,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Casino Bio Tomato & Basil Sauce 200 g,spaghetti with tomato sauce sklavenitis production,casino bio tomato & basil sauce,sklavenitis,,sauce,sauce,480_g,200_g,,
6,sk_0,am_7878,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Casino Bio Tomato Sauce à la Provençale 200 g,spaghetti with tomato sauce sklavenitis production,casino bio tomato sauce à la provençale,sklavenitis,,sauce,sauce,480_g,200_g,,
7,sk_0,am_7988,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Casino Tomato Sauce Provencal 420 g,spaghetti with tomato sauce sklavenitis production,casino tomato sauce provencal,sklavenitis,,sauce,sauce,480_g,420_g,,
8,sk_0,am_8034,product_type + exact_size,,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Heinz Classic BBQ Sauce 480 g,spaghetti with tomato sauce sklavenitis production,heinz classic bbq sauce,sklavenitis,heinz,sauce,sauce,480_g,480_g,,
9,sk_0,am_8183,product_type + informative_token,tomato,Spaghetti with Tomato Sauce SKLAVENITIS ProductION 480g 775297,Bertolli Bolognese Tomato Sauce with the Beef Meat 400 g,spaghetti with tomato sauce sklavenitis production,bertolli bolognese tomato sauce with the beef meat,sklavenitis,bertolli,sauce,sauce,480_g,400_g,,


In [106]:
# посчитала медианное количество кандидатов на один товар склавенитива и альфы-омеги, чтобы понимать, насколько он раздут и матчинг слабый или наоборот слишком строгий 
candidates_per_sk = candidate_pairs.groupby('item_id_sk')['item_id_am'].nunique()
candidates_per_am = candidate_pairs.groupby('item_id_am')['item_id_sk'].nunique()

median_sk = candidates_per_sk.median()
median_am = candidates_per_am.median()
median_sk, median_am

(np.float64(8.0), np.float64(5.0))

In [107]:
# тут прям подробнее захотелось посмотреть на распределение кол-ва кандидатов на товар по магазинам

# 1. Сколько кандидатов у каждого товара
candidates_per_sk = candidate_pairs.groupby('item_id_sk')['item_id_am'].nunique()
candidates_per_am = candidate_pairs.groupby('item_id_am')['item_id_sk'].nunique()

# 2. Если хочешь учитывать и товары с 0 кандидатами,
# нужно добавить их вручную из полного списка товаров

all_sk_items = sk['sk_item_id'].unique()
all_am_items = alpha['am_item_id'].unique()

candidates_per_sk = candidates_per_sk.reindex(all_sk_items, fill_value=0)
candidates_per_am = candidates_per_am.reindex(all_am_items, fill_value=0)

# 3. Функция для разбиения по корзинам
def bucketize_candidates(series):
    bins = [-0.1, 0, 1, 5, 10, 20, 50, 100, np.inf]
    labels = ['0', '1', '2-5', '6-10', '11-20', '21-50', '51-100', '100+']
    bucketed = pd.cut(series, bins=bins, labels=labels)

    summary = (
        bucketed.value_counts(sort=False)
        .rename_axis('candidate_bucket')
        .reset_index(name='item_count')
    )
    share_pct = (summary['item_count'] / summary['item_count'].sum() * 100).round(1)
    summary['share_pct_label'] = share_pct.astype(str) + '%'
    return summary

# 4. Таблицы по двум магазинам
bucket_table_sk = bucketize_candidates(candidates_per_sk)
bucket_table_am = bucketize_candidates(candidates_per_am)

print("Sklavenitis")
display(bucket_table_sk)

print("Alpha Omega")
display(bucket_table_am)

Sklavenitis


,candidate_bucket,item_count,share_pct_label
0,0,827,24.7%
1,1,291,8.7%
2,2-5,739,22.1%
3,6-10,444,13.3%
4,11-20,410,12.3%
5,21-50,398,11.9%
6,51-100,128,3.8%
7,100+,105,3.1%


Alpha Omega


,candidate_bucket,item_count,share_pct_label
0,0,3306,38.1%
1,1,890,10.3%
2,2-5,1867,21.5%
3,6-10,1010,11.6%
4,11-20,783,9.0%
5,21-50,780,9.0%
6,51-100,36,0.4%
7,100+,0,0.0%


In [108]:
# просто блок чтобы посмотреть что в informative tokens внутри находится, какие чаще всего слова туда попадают

# -----------------------------
# Настройки
# -----------------------------
TOP_N = 50
SUSPICIOUS_PAIR_COUNT = 100
SUSPICIOUS_AVG_PAIRS_PER_ITEM = 5

# -----------------------------
# Вспомогательная функция
# -----------------------------
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        # если вдруг список сохранился строкой вида "['a', 'b']"
        try:
            import ast
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        return [x]
    return list(x)

# -----------------------------
# 1. Подготовим exploded таблицы по токенам для обоих магазинов
# -----------------------------
sk_diag = sk[['sk_item_id', 'product_type_block', 'informative_tokens']].copy()
alpha_diag = alpha[['am_item_id', 'product_type_block', 'informative_tokens']].copy()

sk_diag['informative_tokens'] = sk_diag['informative_tokens'].apply(ensure_list)
alpha_diag['informative_tokens'] = alpha_diag['informative_tokens'].apply(ensure_list)

sk_diag = sk_diag.explode('informative_tokens').rename(columns={
    'sk_item_id': 'item_id',
    'product_type_block': 'product_type',
    'informative_tokens': 'informative_token'
})
alpha_diag = alpha_diag.explode('informative_tokens').rename(columns={
    'am_item_id': 'item_id',
    'product_type_block': 'product_type',
    'informative_tokens': 'informative_token'
})

sk_diag['store'] = 'sk'
alpha_diag['store'] = 'am'

all_token_rows = pd.concat([sk_diag, alpha_diag], ignore_index=True)
all_token_rows['informative_token'] = all_token_rows['informative_token'].fillna('').astype(str).str.strip()
all_token_rows['product_type'] = all_token_rows['product_type'].fillna('').astype(str).str.strip()
all_token_rows = all_token_rows[all_token_rows['informative_token'] != '']

# -----------------------------
# 2. Общая статистика по токенам в датасетах
# -----------------------------
token_global_stats = (
    all_token_rows
    .groupby('informative_token', as_index=False)
    .agg(
        total_rows=('item_id', 'size'),
        unique_items=('item_id', 'nunique'),
        unique_product_types=('product_type', 'nunique'),
        sk_items=('store', lambda s: (s == 'sk').sum()),
        am_items=('store', lambda s: (s == 'am').sum()),
    )
    .sort_values(['unique_items', 'total_rows'], ascending=False)
)

print("TOP informative_tokens по числу уникальных товаров:")
display(token_global_stats.head(TOP_N))

# -----------------------------
# 3. Посмотреть распределение токенов по product_type
# -----------------------------
token_type_stats = (
    all_token_rows
    .groupby(['informative_token', 'product_type'], as_index=False)
    .agg(item_count=('item_id', 'nunique'))
    .sort_values(['informative_token', 'item_count'], ascending=[True, False])
)

print("Пример распределения топ-токенов по product_type:")
top_tokens = token_global_stats.head(20)['informative_token'].tolist()
display(token_type_stats[token_type_stats['informative_token'].isin(top_tokens)].head(200))

# -----------------------------
# 4. Диагностика по candidate_pairs
# -----------------------------
# Ожидается, что в candidate_pairs есть либо:
# - shared_rescue_tokens
# либо
# - shared_token / shared_tokens / informative_token
#
# Ниже попробуем автоматически найти подходящую колонку.

possible_token_cols = [
    'shared_rescue_tokens',
    'shared_token',
    'shared_tokens',
    'informative_token'
]

token_col = None
for col in possible_token_cols:
    if col in candidate_pairs.columns:
        token_col = col
        break

if token_col is None:
    print("Не нашла колонку с токенами в candidate_pairs. Проверь имя колонки вручную.")
else:
    print(f"Использую колонку candidate_pairs['{token_col}'] для анализа.")

    cp = candidate_pairs.copy()
    cp[token_col] = cp[token_col].apply(ensure_list)
    cp = cp.explode(token_col)
    cp = cp.rename(columns={token_col: 'informative_token'})
    cp['informative_token'] = cp['informative_token'].fillna('').astype(str).str.strip()
    cp = cp[cp['informative_token'] != '']

    token_pair_stats = (
        cp
        .groupby('informative_token', as_index=False)
        .agg(
            pair_count=('item_id_sk', 'size'),
            unique_sk_items=('item_id_sk', 'nunique'),
            unique_am_items=('item_id_am', 'nunique')
        )
        .sort_values(['pair_count', 'unique_sk_items'], ascending=False)
    )

    print("TOP informative_tokens по числу candidate pairs:")
    display(token_pair_stats.head(TOP_N))

    # -----------------------------
    # 5. Объединяем pair stats + global stats
    # -----------------------------
    token_diagnostics = token_pair_stats.merge(
        token_global_stats,
        on='informative_token',
        how='left'
    )

    token_diagnostics['pairs_per_sk_item'] = (
        token_diagnostics['pair_count'] / token_diagnostics['unique_sk_items'].replace(0, np.nan)
    )
    token_diagnostics['pairs_per_am_item'] = (
        token_diagnostics['pair_count'] / token_diagnostics['unique_am_items'].replace(0, np.nan)
    )

    print("Диагностическая таблица по токенам:")
    display(token_diagnostics.head(TOP_N))

    # -----------------------------
    # 6. Подозрительные токены
    # -----------------------------
    suspicious_tokens = token_diagnostics[
        (
            (token_diagnostics['pair_count'] >= SUSPICIOUS_PAIR_COUNT) |
            (token_diagnostics['pairs_per_sk_item'] >= SUSPICIOUS_AVG_PAIRS_PER_ITEM) |
            (token_diagnostics['unique_items'] >= 150)
        )
    ].sort_values(
        ['pair_count', 'pairs_per_sk_item', 'unique_items'],
        ascending=False
    )

    print("Подозрительные informative_tokens-кандидаты в stop-list:")
    display(suspicious_tokens.head(100))

TOP informative_tokens по числу уникальных товаров:


,informative_token,total_rows,unique_items,unique_product_types,sk_items,am_items
2268,organic,200,200,55,46,154
1802,light,199,199,37,46,153
2911,slices,197,197,24,42,155
2272,original,195,195,54,52,143
2996,spray,184,184,19,49,135
2160,natural,182,182,29,49,133
52,3,181,181,52,47,134
3317,vanilla,181,181,46,35,146
405,black,179,179,37,40,139
1782,lemon,177,177,48,51,126


Пример распределения топ-токенов по product_type:


,informative_token,product_type,item_count
58,10,,50
64,10,cheese,21
66,10,coffee capsules,16
74,10,hair colour,7
67,10,cream,6
...,...,...,...
2781,chicken,baguette,1
2783,chicken,bread,1
2784,chicken,breadcrumbs,1
2787,chicken,cat treats,1


Использую колонку candidate_pairs['shared_rescue_tokens'] для анализа.
TOP informative_tokens по числу candidate pairs:


,informative_token,pair_count,unique_sk_items,unique_am_items
3279,slices,1330,29,104
1535,dark,1315,24,79
1163,cheddar,642,19,42
2840,natural,635,30,49
3342,spray,558,22,58
2712,melissa ~ melissa (1.00),430,20,28
2876,nivea ~ nivea (1.00),423,31,76
3164,roll,416,19,34
376,alpro ~ alpro (1.00),386,22,30
2774,mitsides ~ mitsides (1.00),352,15,47


Диагностическая таблица по токенам:


,informative_token,pair_count,unique_sk_items,unique_am_items,total_rows,unique_items,unique_product_types,sk_items,am_items,pairs_per_sk_item,pairs_per_am_item
0,slices,1330,29,104,197.0,197.0,24.0,42.0,155.0,45.862069,12.788462
1,dark,1315,24,79,163.0,163.0,18.0,35.0,128.0,54.791667,16.645570
2,cheddar,642,19,42,63.0,63.0,3.0,20.0,43.0,33.789474,15.285714
3,natural,635,30,49,182.0,182.0,29.0,49.0,133.0,21.166667,12.959184
4,spray,558,22,58,184.0,184.0,19.0,49.0,135.0,25.363636,9.620690
5,melissa ~ melissa (1.00),430,20,28,NaN,NaN,NaN,NaN,NaN,21.500000,15.357143
6,nivea ~ nivea (1.00),423,31,76,NaN,NaN,NaN,NaN,NaN,13.645161,5.565789
7,roll,416,19,34,80.0,80.0,12.0,25.0,55.0,21.894737,12.235294
8,alpro ~ alpro (1.00),386,22,30,NaN,NaN,NaN,NaN,NaN,17.545455,12.866667
9,mitsides ~ mitsides (1.00),352,15,47,NaN,NaN,NaN,NaN,NaN,23.466667,7.489362


Подозрительные informative_tokens-кандидаты в stop-list:


,informative_token,pair_count,unique_sk_items,unique_am_items,total_rows,unique_items,unique_product_types,sk_items,am_items,pairs_per_sk_item,pairs_per_am_item
0,slices,1330,29,104,197.0,197.0,24.0,42.0,155.0,45.862069,12.788462
1,dark,1315,24,79,163.0,163.0,18.0,35.0,128.0,54.791667,16.645570
2,cheddar,642,19,42,63.0,63.0,3.0,20.0,43.0,33.789474,15.285714
3,natural,635,30,49,182.0,182.0,29.0,49.0,133.0,21.166667,12.959184
4,spray,558,22,58,184.0,184.0,19.0,49.0,135.0,25.363636,9.620690
...,...,...,...,...,...,...,...,...,...,...,...
102,filling,52,8,31,94.0,94.0,20.0,10.0,84.0,6.500000,1.677419
103,cocoa,51,9,30,111.0,111.0,29.0,21.0,90.0,5.666667,1.700000
104,gold,50,8,12,75.0,75.0,22.0,27.0,48.0,6.250000,4.166667
108,tonic,49,4,17,28.0,28.0,6.0,5.0,23.0,12.250000,2.882353


## 26. Scoring candidate pairs

Теперь переходим от `candidate blocking` к `candidate scoring`.

Blocking уже сделал важную работу: он не сравнивает каждый товар с каждым, а оставляет только разумные пары-кандидаты. Следующий шаг — дать каждой такой паре числовую оценку от `0` до `1`: насколько эта пара похожа на настоящий match.

Важно: мы пока не используем embeddings и нейросети. Делаем прозрачный scoring, где каждую часть можно отдельно посмотреть глазами и понять, почему пара получила высокий или низкий балл.

В этом блоке считаем:

- TF-IDF similarity по символам `3-5` для `product_core`
- TF-IDF similarity по словам для `product_core`
- fuzzy similarity для `product_type`
- fuzzy similarity для `brand`
- `brand_penalty`, если бренд есть у обоих товаров и он явно разный
- ручной `size_similarity`
- небольшой bonus/penalty по `grade`

После каждого метода выводим короткую проверку результата.


### 26.1 TF-IDF char similarity по `product_core`

Сначала сравним `product_core` на уровне коротких кусочков символов длиной `3-5`.

Это помогает ловить похожесть даже тогда, когда строки немного отличаются: есть дефис, пропущен пробел, другое окончание слова или маленькая опечатка.

Здесь используем стандартный `TfidfVectorizer` из `sklearn`: он превращает текст в TF-IDF векторы, а потом мы сравниваем пары через cosine similarity.


In [109]:
from difflib import SequenceMatcher
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances

# Начинаем отдельную scored-таблицу, чтобы не портить исходный candidate_pairs.
candidate_pairs_scored = candidate_pairs.copy()


def normalize_for_similarity(value):
    # Приводим текст к стабильному виду: lowercase, без лишних пробелов.
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def compute_pairwise_tfidf_similarity(left_values, right_values, vectorizer):
    # Строим TF-IDF на общем словаре двух магазинов,
    # а потом считаем cosine similarity только для candidate pairs.
    left_texts = pd.Series(left_values).fillna('').astype(str).map(normalize_for_similarity)
    right_texts = pd.Series(right_values).fillna('').astype(str).map(normalize_for_similarity)

    both_present = left_texts.ne('') & right_texts.ne('')
    similarities = pd.Series(0.5, index=left_values.index, dtype='float64')

    if not both_present.any():
        return similarities

    left_non_empty = left_texts.loc[both_present]
    right_non_empty = right_texts.loc[both_present]
    all_non_empty_texts = pd.concat([left_non_empty, right_non_empty], ignore_index=True)

    vectorizer.fit(all_non_empty_texts)
    left_vectors = vectorizer.transform(left_non_empty)
    right_vectors = vectorizer.transform(right_non_empty)

    # paired_cosine_distances возвращает distance, поэтому similarity = 1 - distance.
    pair_similarities = 1 - paired_cosine_distances(left_vectors, right_vectors)
    similarities.loc[both_present] = pair_similarities

    return similarities.clip(0, 1)


char_tfidf_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    lowercase=False
)

candidate_pairs_scored['tfidf_char_core_similarity'] = compute_pairwise_tfidf_similarity(
    candidate_pairs_scored['product_core_block_sk'],
    candidate_pairs_scored['product_core_block_am'],
    char_tfidf_vectorizer
)

print('TF-IDF char similarity summary:')
display(candidate_pairs_scored['tfidf_char_core_similarity'].describe().to_frame())

print('Примеры пар с высоким TF-IDF char similarity:')
display(
    candidate_pairs_scored
    .sort_values('tfidf_char_core_similarity', ascending=False)
    [[
        'tfidf_char_core_similarity',
        'product_core_block_sk', 'product_core_block_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(15)
)


TF-IDF char similarity summary:


,tfidf_char_core_similarity
count,53436.000000
mean,0.187864
std,0.189580
min,0.000000
25%,0.044137
50%,0.131353
75%,0.271841
max,1.000000


Примеры пар с высоким TF-IDF char similarity:


,tfidf_char_core_similarity,product_core_block_sk,product_core_block_am,product_name_sk,product_name_am
37955,1.0,lux beauty moments soap with almond oil,lux beauty moments soap with almond oil,LUX Beauty Moments Soap With Almond Oil 4X125g 141446,Lux Beauty Moments Soap with Almond Oil 125 g
24158,1.0,frontera merlot red wine,frontera merlot red wine,FRONTERA Merlot Red Wine 750ml 138877,Frontera Merlot Red Wine 750 ml
152,1.0,yumearth organic fruit snacks,yumearth organic fruit snacks,YUMEARTH Organic Fruit Snacks 50g 789675,Yumearth Organic Fruit Snacks 50 g
53430,1.0,farm n.b shikkis 12 quails fresh eggs,farm n.b shikkis 12 quails fresh eggs,Farm N.B SHIKKIS 12 Quails Fresh Eggs 645270,Farm N.B Shikkis 12 Quails Fresh Eggs
52960,1.0,kerrygold spreadable butter with olive oil,kerrygold spreadable butter with olive oil,KERRYGOLD Spreadable Butter with Olive Oil 212 g 783159,Kerrygold Spreadable Butter With Olive Oil 212 g
11366,1.0,coca cola zero sugar,coca cola zero sugar,COCA COLA Zero Sugar 6x150 ml 731434,Coca Cola Zero Sugar 8x330 ml
11367,1.0,coca cola zero sugar,coca cola zero sugar,COCA COLA Zero Sugar 6x150 ml 731434,Coca Cola Zero Sugar 1.5 L
49825,1.0,charalambides christis feta p.o.p,charalambides christis feta p.o.p,CHARALAMBIDES CHRISTIS Feta P.O.P 200g 639240,Charalambides Christis Feta P.O.P 200 g
39168,1.0,tsatsakis breadsticks with carob & sesame,tsatsakis breadsticks with carob & sesame,Tsatsakis Breadsticks With Carob & Sesame 250g,Tsatsakis Breadsticks with Carob & Sesame 250 g
39150,1.0,johnsof carob breadsticks,johnsof carob breadsticks,JOHNSOF Carob Breadsticks 220g,Johnsof Carob Breadsticks 220 g


### 26.2 TF-IDF word similarity по `product_core`

Теперь сравним те же `product_core`, но уже на уровне слов.

Идея проще: если в двух товарах совпадают важные слова, similarity будет выше. Этот метод хуже ловит опечатки, чем char TF-IDF, зато он более понятный: совпали слова — вырос score.

Здесь тоже используем `TfidfVectorizer`, но меняем analyzer на словный режим.


In [110]:
word_tfidf_vectorizer = TfidfVectorizer(
    analyzer='word',
    token_pattern=r'(?u)\b\w+\b',
    lowercase=False
)

candidate_pairs_scored['tfidf_word_core_similarity'] = compute_pairwise_tfidf_similarity(
    candidate_pairs_scored['product_core_block_sk'],
    candidate_pairs_scored['product_core_block_am'],
    word_tfidf_vectorizer
)

print('TF-IDF word similarity summary:')
display(candidate_pairs_scored['tfidf_word_core_similarity'].describe().to_frame())

print('Примеры пар с высоким TF-IDF word similarity:')
display(
    candidate_pairs_scored
    .sort_values('tfidf_word_core_similarity', ascending=False)
    [[
        'tfidf_word_core_similarity',
        'product_core_block_sk', 'product_core_block_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(15)
)


TF-IDF word similarity summary:


,tfidf_word_core_similarity
count,53436.000000
mean,0.228296
std,0.208461
min,0.000000
25%,0.071420
50%,0.173653
75%,0.325859
max,1.000000


Примеры пар с высоким TF-IDF word similarity:


,tfidf_word_core_similarity,product_core_block_sk,product_core_block_am,product_name_sk,product_name_am
36357,1.0,dettol liquid disinfectant for surface disinfection,dettol liquid disinfectant for surface disinfection,DETTOL Liquid Disinfectant for Surface DisinfectION 1lt 400072,Dettol Liquid Disinfectant for Surface Disinfection 750 ml
36356,1.0,dettol liquid disinfectant for surface disinfection,dettol liquid disinfectant for surface disinfection,DETTOL Liquid Disinfectant for Surface DisinfectION 1lt 400072,Dettol Liquid Disinfectant for Surface Disinfection 1 L
53430,1.0,farm n.b shikkis 12 quails fresh eggs,farm n.b shikkis 12 quails fresh eggs,Farm N.B SHIKKIS 12 Quails Fresh Eggs 645270,Farm N.B Shikkis 12 Quails Fresh Eggs
21,1.0,shikkis farm n.b quails eggs in vinegar,farm n.b shikkis quails eggs in vinegar,SHIKKIS Farm N.B Quails Eggs In Vinegar 360g 645271,Farm N.B Shikkis Quails Eggs in Vinegar 360 g
38952,1.0,ritz crackers original,ritz original crackers,Ritz Crackers 200g Original 702056,Ritz Original Crackers 200 g
5074,1.0,karla corned beef,karla corned beef,KARLA Corned Beef 340g 139954,Karla Corned Beef 340 g
26990,1.0,serano roasted pumpkin seeds,serano roasted pumpkin seeds,SERANO Roasted Pumpkin Seeds 136800,Serano Roasted Pumpkin Seeds 125 g
38944,1.0,ritz crackers original,ritz original crackers,Ritz Crackers 200g Original 702056,Ritz Original Crackers 200 g
27001,1.0,serano walnut kernels,serano walnut kernels,SERANO Walnut Kernels 130g 715131,Serano Walnut Kernels 130 g
5001,1.0,mitsides semolina flour,mitsides semolina flour,MITSIDES Semolina Flour 1Kg 116508,Mitsides Semolina flour 1 kg


### 26.3 Fuzzy similarity для `product_type`

Для `product_type` используем fuzzy similarity.

Здесь нам важна не широкая смысловая похожесть, а аккуратное строковое совпадение. Например, `tomato sauce` и `sauce tomato` должны быть похожими, а `milk` и `yogurt` — нет.

Если `product_type` отсутствует хотя бы с одной стороны, ставим `0.5`: это нейтральное значение, которое не помогает и не мешает слишком сильно.


In [111]:
def fuzzy_similarity(left_value, right_value):
    # Fuzzy similarity сравнивает строки напрямую, без векторов.
    left_text = normalize_for_similarity(left_value)
    right_text = normalize_for_similarity(right_value)

    if left_text == '' or right_text == '':
        return 0.5

    return SequenceMatcher(None, left_text, right_text).ratio()


candidate_pairs_scored['product_type_similarity'] = [
    fuzzy_similarity(left_value, right_value)
    for left_value, right_value in zip(
        candidate_pairs_scored['product_type_block_sk'],
        candidate_pairs_scored['product_type_block_am']
    )
]

candidate_pairs_scored['product_type_similarity'] = candidate_pairs_scored['product_type_similarity'].clip(0, 1)

print('Product type fuzzy similarity summary:')
display(candidate_pairs_scored['product_type_similarity'].describe().to_frame())

print('Примеры пар с низким product_type similarity:')
display(
    candidate_pairs_scored
    .sort_values('product_type_similarity')
    [[
        'product_type_similarity',
        'product_type_block_sk', 'product_type_block_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(15)
)


Product type fuzzy similarity summary:


,product_type_similarity
count,53436.000000
mean,0.949577
std,0.178548
min,0.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,1.000000


Примеры пар с низким product_type similarity:


,product_type_similarity,product_type_block_sk,product_type_block_am,product_name_sk,product_name_am
53121,0.0,ham,wine,GRIGORIOU Turkey Ham Slices 100g 224637,Grigoriou Wine Lountza Sliced 100 g
4837,0.0,pudding,cream,YIOTIS Instant Pudding Vanilla Flavour 62g 104174,Yiotis Instant Cream Vanilla Flavour 62 g
4934,0.0,flour,beans,Flour MITSIDES Self Raising 1kg 116510,Mitsides Medium Beans 1 kg
1054,0.0,pudding,cheese,ARLA Protein Pudding with Salted Caramel flavor Lactose free No added sugar 200g 775728,Arla Protein Cottage Cheese 200 g
1053,0.0,pudding,cheese,ARLA Protein Pudding with Salted Caramel flavor Lactose free No added sugar 200g 775728,Arla Mozzarella Cheese (10 slices) 200 g
3518,0.0,oil,sauce,AMOY Sesame Oil 150ml 233885,Amoy Light Soy Sauce 150 ml
3519,0.0,oil,sauce,AMOY Sesame Oil 150ml 233885,Amoy Soy Sauce with Reduced Salt 150 ml
3476,0.0,sauce,oil,AMOY Soy Sauce Reduced Salt 150ml 135542,Amoy Sesame Oil 150 ml
10263,0.0,juice,brandy,KEO Juice 7 Fruit 1L 134078,Keo VSOP Brandy 38% 1 L
1055,0.0,pudding,chocolate,ARLA Protein Pudding with Salted Caramel flavor Lactose free No added sugar 200g 775728,Arla Protein Chocolate Flavour Pudding 200 g


### 26.4 Fuzzy similarity для `brand` и brand penalty

Для бренда тоже используем fuzzy similarity.

Но бренд — особое поле. Если бренд есть у обоих товаров и он явно разный, это сильный сигнал против match. Поэтому отдельно добавляем `brand_penalty`.

Логика такая:

- если бренд отсутствует хотя бы с одной стороны, `brand_similarity = 0.5`
- если бренд есть у обоих, считаем fuzzy similarity
- если бренд есть у обоих и `brand_similarity < 0.30`, добавляем штраф `-0.15`


In [112]:
candidate_pairs_scored['brand_similarity'] = [
    fuzzy_similarity(left_value, right_value)
    for left_value, right_value in zip(
        candidate_pairs_scored['brand_block_sk'],
        candidate_pairs_scored['brand_block_am']
    )
]

candidate_pairs_scored['brand_similarity'] = candidate_pairs_scored['brand_similarity'].clip(0, 1)

both_brands_present = (
    candidate_pairs_scored['brand_block_sk'].map(normalize_for_similarity).ne('')
    & candidate_pairs_scored['brand_block_am'].map(normalize_for_similarity).ne('')
)

candidate_pairs_scored['brand_penalty'] = np.where(
    both_brands_present & candidate_pairs_scored['brand_similarity'].lt(0.30),
    -0.15,
    0.0
)

print('Brand fuzzy similarity summary:')
display(candidate_pairs_scored['brand_similarity'].describe().to_frame())

print('Сколько пар получили brand penalty:')
display(candidate_pairs_scored['brand_penalty'].value_counts().rename_axis('brand_penalty').reset_index(name='pair_count'))

print('Примеры пар с brand penalty:')
display(
    candidate_pairs_scored
    .loc[candidate_pairs_scored['brand_penalty'].lt(0)]
    .sort_values('brand_similarity')
    [[
        'brand_similarity', 'brand_penalty',
        'brand_block_sk', 'brand_block_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(15)
)


Brand fuzzy similarity summary:


,brand_similarity
count,53436.000000
mean,0.517203
std,0.279016
min,0.000000
25%,0.333333
50%,0.500000
75%,0.500000
max,1.000000


Сколько пар получили brand penalty:


,brand_penalty,pair_count
0,0.00,41394
1,-0.15,12042


Примеры пар с brand penalty:


,brand_similarity,brand_penalty,brand_block_sk,brand_block_am,product_name_sk,product_name_am
53362,0.0,-0.15,becel,horio,BECEL ProActiv Margarine Light Vegan 250g 444413,Minerva Horio Fast Soft Light Margarine 220 g
53011,0.0,-0.15,grigoriou,dymes,GRIGORIOU Armenian Sausage Pastourmas 300g 224609,Dymes Pastourmas 300 g
53013,0.0,-0.15,grigoriou,lackmann,GRIGORIOU Sausages Frankfurters 500g 224613,Lackmann Boiled Sausages 500 g
53018,0.0,-0.15,dymes,grigoriou,DYMES PITSILLIAS Cyprus TraditION al Finger Sausages 200 g 776729,Grigoriou Traditional Cyprus Thin Sausages 250 g
53033,0.0,-0.15,snack,grigoriou,SNACK Ham Leg 300g 224679,Grigoriou Premium Ham Leg Sliced 150 g
53038,0.0,-0.15,grigoriou,snack,GRIGORIOU Premium Ham Leg Slices 150g 224602,Snack Ham Leg 150 g
192,0.0,-0.15,barilla,de cecco,BARILLA Gluten Free Spaghetti 400g 641548,De Cecco Spaghetti Pasta No.12 500 g
52881,0.0,-0.15,grigoriou,snack,GRIGORIOU Turkey Salami 300g 430206,Snack Salami Extra 300 g
52920,0.0,-0.15,grigoriou,snack,GRIGORIOU Salami Extra 300g 224627,Snack Salami Extra 300 g
52652,0.0,-0.15,chavroux,keses,CHAVROUX Soft Goat Cheese 150g 419227,Keses Mature Sheep & Goat Halloumi P.D.O 250 g


### 26.5 `size_similarity`

Размер считаем вручную, потому что это не обычный текст.

Сначала смотрим, есть ли размер у обоих товаров. Если размера нет хотя бы у одного товара, ставим нейтральное значение `0.5`.

Если размер есть у обоих, используем относительную разницу:

```text
diff = abs(size_1 - size_2) / max(size_1, size_2)
```

Потом переводим `diff` в score:

```text
0                  -> 1.00
<= 0.03            -> 0.90
<= 0.10            -> 0.70
<= 0.25            -> 0.40
больше 0.25        -> 0.05
```

Если единицы измерения разные, например `g` и `ml`, ставим низкое значение `0.05`.


In [113]:
def parse_size_key(size_key):
    # size_key ожидается в формате вроде 500_g или 1_l.
    text = normalize_for_similarity(size_key)
    if text == '' or '_' not in text:
        return np.nan, ''

    number_text, unit = text.split('_', 1)
    number = pd.to_numeric(pd.Series([number_text]), errors='coerce').iloc[0]
    if pd.isna(number) or unit == '':
        return np.nan, ''

    return float(number), unit


def size_similarity(left_size_key, right_size_key):
    left_size, left_unit = parse_size_key(left_size_key)
    right_size, right_unit = parse_size_key(right_size_key)

    if pd.isna(left_size) or pd.isna(right_size):
        return 0.5

    if left_unit == '' or right_unit == '':
        return 0.5

    if left_unit != right_unit:
        return 0.05

    max_size = max(left_size, right_size)
    if max_size == 0:
        return 0.5

    diff = abs(left_size - right_size) / max_size

    if diff == 0:
        return 1.0
    if diff <= 0.03:
        return 0.9
    if diff <= 0.10:
        return 0.7
    if diff <= 0.25:
        return 0.4
    return 0.05


candidate_pairs_scored['size_similarity'] = [
    size_similarity(left_value, right_value)
    for left_value, right_value in zip(
        candidate_pairs_scored['size_key_sk'],
        candidate_pairs_scored['size_key_am']
    )
]

candidate_pairs_scored['size_similarity'] = candidate_pairs_scored['size_similarity'].clip(0, 1)

print('Size similarity summary:')
display(candidate_pairs_scored['size_similarity'].describe().to_frame())

print('Распределение size_similarity:')
display(
    candidate_pairs_scored['size_similarity']
    .value_counts(dropna=False)
    .rename_axis('size_similarity')
    .reset_index(name='pair_count')
    .sort_values('size_similarity', ascending=False)
)

print('Примеры пар с низким size_similarity:')
display(
    candidate_pairs_scored
    .sort_values('size_similarity')
    [[
        'size_similarity',
        'size_key_sk', 'size_key_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(15)
)


Size similarity summary:


,size_similarity
count,53436.000000
mean,0.723101
std,0.406407
min,0.050000
25%,0.400000
50%,1.000000
75%,1.000000
max,1.000000


Распределение size_similarity:


,size_similarity,pair_count
0,1.00,35349
5,0.90,117
3,0.70,1075
4,0.50,859
2,0.40,3433
1,0.05,12603


Примеры пар с низким size_similarity:


,size_similarity,size_key_sk,size_key_am,product_name_sk,product_name_am
12607,0.05,1000_ml,1980_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Orange 6x330 ml
12608,0.05,1000_ml,1980_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Zero Orange 6x330 ml
12611,0.05,1000_ml,1500_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Orange 1.5 L
12613,0.05,1000_ml,330_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Zero Orange 330 ml
12616,0.05,1000_ml,250_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Vimto Mix Fruit Fizzy Soft Drink 250 ml
12618,0.05,1000_ml,2640_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Dr Pepper Sparkling Fruit Flavour Soft Drink 8x330 ml
12619,0.05,1000_ml,150_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Orange 150 ml
12620,0.05,1000_ml,2000_ml,ROBINSONS Real Fruit Orange Soft Drink 1L 136110,Fanta Orange 4x500 ml
35628,0.05,40_ml,192_ml,"GARNIER Color SensatION Permanent Hair Color No 8.0 Light Blonde, 40 ml 558689",L'Oréal Paris Excellence Creme Hair Color 8U Light Blonde 192 ml
35629,0.05,40_ml,192_ml,"GARNIER Color SensatION Permanent Hair Color No 8.0 Light Blonde, 40 ml 558689",L'Oréal Paris Excellence Creme Hair Color 7U Blonde 192 ml


### 26.6 `grade` bonus / penalty

`grade` встречается не везде, поэтому не даём ему большой основной вес.

Используем его как маленькую корректировку:

- если `grade` есть у обоих и совпал: `+0.05`
- если `grade` есть у обоих, но не совпал: `-0.03`
- если `grade` отсутствует хотя бы с одной стороны: `0`

Важно: раньше в candidate blocking мы уже убирали пары с явно разным `grade`. Поэтому здесь отрицательных случаев может почти не быть. Это нормально.


In [114]:
def grade_adjustment(left_grade, right_grade):
    left_text = normalize_for_similarity(left_grade)
    right_text = normalize_for_similarity(right_grade)

    if left_text == '' or right_text == '':
        return 0.0

    if left_text == right_text:
        return 0.05

    return -0.03


candidate_pairs_scored['grade_adjustment'] = [
    grade_adjustment(left_value, right_value)
    for left_value, right_value in zip(
        candidate_pairs_scored['grade_block_sk'],
        candidate_pairs_scored['grade_block_am']
    )
]

print('Распределение grade_adjustment:')
display(
    candidate_pairs_scored['grade_adjustment']
    .value_counts(dropna=False)
    .rename_axis('grade_adjustment')
    .reset_index(name='pair_count')
    .sort_values('grade_adjustment', ascending=False)
)

print('Примеры пар с grade bonus/penalty:')
display(
    candidate_pairs_scored
    .loc[candidate_pairs_scored['grade_adjustment'].ne(0)]
    .sort_values('grade_adjustment', ascending=False)
    [[
        'grade_adjustment',
        'grade_block_sk', 'grade_block_am',
        'product_name_sk', 'product_name_am'
    ]]
    .head(20)
)


Распределение grade_adjustment:


,grade_adjustment,pair_count
1,0.05,244
0,0.00,53192


Примеры пар с grade bonus/penalty:


,grade_adjustment,grade_block_sk,grade_block_am,product_name_sk,product_name_am
34,0.05,size 6,size 6,AGIA SKEPI Organic Free Range Eggs Large Size 6 Pcs 653366,Petrides Fresh Free Range Eggs Large Size 6 Pieces
35,0.05,size 6,size 6,AGIA SKEPI Organic Free Range Eggs Large Size 6 Pcs 653366,Nikiforou Bio Fresh Eggs Large Size 6 Pieces
41,0.05,size 6,size 6,AGIA SKEPI Organic Free Range Eggs Large Size 6 Pcs 653366,Nikiforou Fresh Eggs Large Size 6 Pieces
261,0.05,0% fat,0% fat,MEVGAL Authentic Greek Yoghurt Strawberry 0% Fat 150g 801933,Kri Kri High Protein Super Spoon Yoghurt Dessert 0% Fat With Strawberry 200 g
273,0.05,0% fat,0% fat,MEVGAL Authentic Greek Yoghurt Strawberry 0% Fat 150g 801933,Charalambides Christis Straggato Fruta Strawberry Yoghurt Dessert 0% Fat 3x150 g
2250,0.05,1.5% fat,1.5% fat,"LAURA Homogenized Milk 1,5% Fat 1lt 647151",Olympos Bio Goat Milk 1.5% Fat 1 L
2251,0.05,1.5% fat,1.5% fat,"LAURA Homogenized Milk 1,5% Fat 1lt 647151",Olympos Freelact Lactose-Free Milk 1.5% Fat 1 L
2252,0.05,1.5% fat,1.5% fat,"LAURA Homogenized Milk 1,5% Fat 1lt 647151",Olympos Freelact Lactose Free Goats Milk 1.5% Fat 1 L
2253,0.05,1.5% fat,1.5% fat,"LAURA Homogenized Milk 1,5% Fat 1lt 647151",Charalambides Christis Charalambides Fresh Goat Milk 1.5% Fat 1 L
2255,0.05,1.5% fat,1.5% fat,"LAURA Homogenized Milk 1,5% Fat 1lt 647151",Meggle UHT Semi Skimmed Milk 1.5% Fat 1 L


### 26.7 Итоговая методология scoring

Финальный score собираем из прозрачных частей.

```text
base_score =
    0.25 * tfidf_char_core_similarity
  + 0.20 * tfidf_word_core_similarity
  + 0.20 * product_type_similarity
  + 0.25 * size_similarity
  + 0.10 * brand_similarity
```

Потом добавляем корректировки:

```text
brand_penalty = -0.15
если бренд есть у обоих товаров и brand_similarity < 0.30
```

```text
grade_adjustment = +0.05
если grade есть у обоих товаров и совпал
```

```text
grade_adjustment = -0.03
если grade есть у обоих товаров, но не совпал
```

Итог:

```text
final_match_score = base_score + brand_penalty + grade_adjustment
```

После этого ограничиваем результат диапазоном от `0` до `1`, чтобы score всегда был читаемым.


In [115]:
SCORING_WEIGHTS = {
    'tfidf_char_core_similarity': 0.25,
    'tfidf_word_core_similarity': 0.20,
    'product_type_similarity': 0.20,
    'size_similarity': 0.25,
    'brand_similarity': 0.10,
}

candidate_pairs_scored['base_match_score'] = 0.0
for column_name, weight in SCORING_WEIGHTS.items():
    candidate_pairs_scored['base_match_score'] += candidate_pairs_scored[column_name] * weight

candidate_pairs_scored['final_match_score'] = (
    candidate_pairs_scored['base_match_score']
    + candidate_pairs_scored['brand_penalty']
    + candidate_pairs_scored['grade_adjustment']
).clip(0, 1)

# Добавляем ссылки на картинки из исходных таблиц магазинов.
# Это удобно для ручной проверки match-кандидатов в Excel.
sk_image_url_map = sk.set_index('sk_item_id')['image_url']
alpha_image_url_map = alpha.set_index('am_item_id')['image_url']

candidate_pairs_scored['image_url_sk'] = candidate_pairs_scored['item_id_sk'].map(sk_image_url_map).fillna('')
candidate_pairs_scored['image_url_am'] = candidate_pairs_scored['item_id_am'].map(alpha_image_url_map).fillna('')

score_display_columns = [
    'final_match_score', 'base_match_score',
    'tfidf_char_core_similarity', 'tfidf_word_core_similarity',
    'product_type_similarity', 'size_similarity', 'brand_similarity',
    'brand_penalty', 'grade_adjustment',
    'blocking_rules',
    'brand_block_sk', 'brand_block_am',
    'product_type_block_sk', 'product_type_block_am',
    'size_key_sk', 'size_key_am',
    'grade_block_sk', 'grade_block_am',
    'image_url_sk', 'image_url_am',
    'product_name_sk', 'product_name_am'
]

candidate_pairs_scored = candidate_pairs_scored.sort_values(
    'final_match_score',
    ascending=False
).reset_index(drop=True)

MATCH_SCORING_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_v2.csv'
MATCH_SCORING_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_v2.xlsx'
MATCH_SCORING_WITH_IMAGE_LINKS_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_with_image_links_v2.xlsx'

candidate_pairs_scored.to_csv(MATCH_SCORING_CSV_PATH, index=False, encoding='utf-8-sig')
# candidate_pairs_scored.to_excel(MATCH_SCORING_EXCEL_PATH, index=False)
candidate_pairs_scored.to_excel(MATCH_SCORING_WITH_IMAGE_LINKS_EXCEL_PATH, index=False)

print('Scored candidate pairs:', len(candidate_pairs_scored))
print('Saved CSV to:', MATCH_SCORING_CSV_PATH)
print('Saved Excel to:', MATCH_SCORING_EXCEL_PATH)
print('Saved Excel with image links to:', MATCH_SCORING_WITH_IMAGE_LINKS_EXCEL_PATH)

print('Final score summary:')
display(candidate_pairs_scored['final_match_score'].describe().to_frame())

print('Распределение по score buckets:')
score_bins = [-0.001, 0.50, 0.65, 0.80, 0.90, 1.00]
score_labels = ['<= 0.50', '0.50-0.65', '0.65-0.80', '0.80-0.90', '0.90-1.00']
score_bucket_summary = (
    pd.cut(candidate_pairs_scored['final_match_score'], bins=score_bins, labels=score_labels)
    .value_counts(sort=False)
    .rename_axis('score_bucket')
    .reset_index(name='pair_count')
)
score_bucket_summary['share_pct'] = (
    score_bucket_summary['pair_count'] / score_bucket_summary['pair_count'].sum() * 100
).round(1)
display(score_bucket_summary)

print('Топ candidate pairs по final_match_score:')
display(candidate_pairs_scored[score_display_columns].head(30))


Scored candidate pairs: 53436
Saved CSV to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_scored_v2.csv
Saved Excel to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_scored_v2.xlsx
Saved Excel with image links to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_scored_with_image_links_v2.xlsx
Final score summary:


,final_match_score
count,53436.000000
mean,0.481450
std,0.143691
min,0.083361
25%,0.380662
50%,0.507612
75%,0.545871
max,1.000000


Распределение по score buckets:


,score_bucket,pair_count,share_pct
0,<= 0.50,24011,44.9
1,0.50-0.65,24452,45.8
2,0.65-0.80,3634,6.8
3,0.80-0.90,869,1.6
4,0.90-1.00,470,0.9


Топ candidate pairs по final_match_score:


,final_match_score,base_match_score,tfidf_char_core_similarity,tfidf_word_core_similarity,product_type_similarity,size_similarity,brand_similarity,brand_penalty,grade_adjustment,blocking_rules,...,product_type_block_sk,product_type_block_am,size_key_sk,size_key_am,grade_block_sk,grade_block_am,image_url_sk,image_url_am,product_name_sk,product_name_am
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + informative_token | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_size,...,pomegranate,pomegranate,250_ml,250_ml,,,https://imageproxy.wolt.com/assets/667945e97fc2644ad07687dd,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/811619.jpg&AlternativeImage=/Images/missing_image.jpg,KALAMATA PAPADIMITRIOU Balsamic Cream With Pomegranate 250ml 673077,Kalamata Papadimitriou Balsamic Cream with Pomegranate 250 ml
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_size,...,kataifi,kataifi,500_g,500_g,,,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/70a11bc6-3639-11f0-97d7-d674caf95488_5201231001100.jpeg,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/130285.jpg&AlternativeImage=/Images/missing_image.jpg,XRISI ZIMI Kataifi 500g 145636,Xrisi Zimi Kataifi 500 g
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + informative_token | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_size,...,salt,salt,250_g,250_g,,,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/1dcee4cc-355e-11f0-85e8-0211d2d964dc_5290022000026.jpg,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/115452.jpg&AlternativeImage=/Images/missing_image.jpg,SAILOR Table Salt 250 g 140072,Sailor Table Salt 250 g
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + informative_token | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_size,...,disinfectant,disinfectant,1000_ml,1000_ml,,,https://wolt-menu-images-cdn.wolt.com/menu-images/672a02ff45404fdb3b19f4a3/d6855e22-143b-11f0-a793-baa3506544fb_5208070003586.jpg,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/736850.jpg&AlternativeImage=/Images/missing_image.jpg,DETTOL Liquid Disinfectant for Surface DisinfectION 1lt 400072,Dettol Liquid Disinfectant for Surface Disinfection 1 L
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + informative_token | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_size,...,pomegranate,pomegranate,330_ml,330_ml,,,https://imageproxy.wolt.com/assets/668d0e5b567b2151f8000015,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/854588.jpg&AlternativeImage=/Images/missing_image.jpg,ALAMBRA Natural Pomegranate Juice 330ml 713366,Alambra Natural Pomegranate Juice 330 ml
5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.00,brand + exact_size | brand + product_type | product_type + exact_size | product_type + similar_brand_fragment | similar_brand_fragment + exact_size | tfidf_0_60 + size_similarity_0_70 | tfidf_0_85_any_

In [116]:
# посмотреть топ 50 пар:
candidate_pairs_scored[
    [
        'final_match_score',
        'product_name_sk',
        'product_name_am',
        'brand_block_sk',
        'brand_block_am',
        'product_type_block_sk',
        'product_type_block_am',
        'size_key_sk',
        'size_key_am'
    ]
].head(50)

,final_match_score,product_name_sk,product_name_am,brand_block_sk,brand_block_am,product_type_block_sk,product_type_block_am,size_key_sk,size_key_am
0,1.0,KALAMATA PAPADIMITRIOU Balsamic Cream With Pomegranate 250ml 673077,Kalamata Papadimitriou Balsamic Cream with Pomegranate 250 ml,kalamata papadimitriou,kalamata papadimitriou,pomegranate,pomegranate,250_ml,250_ml
1,1.0,XRISI ZIMI Kataifi 500g 145636,Xrisi Zimi Kataifi 500 g,xrisi zimi,xrisi zimi,kataifi,kataifi,500_g,500_g
2,1.0,SAILOR Table Salt 250 g 140072,Sailor Table Salt 250 g,sailor,sailor,salt,salt,250_g,250_g
3,1.0,DETTOL Liquid Disinfectant for Surface DisinfectION 1lt 400072,Dettol Liquid Disinfectant for Surface Disinfection 1 L,dettol,dettol,disinfectant,disinfectant,1000_ml,1000_ml
4,1.0,ALAMBRA Natural Pomegranate Juice 330ml 713366,Alambra Natural Pomegranate Juice 330 ml,alambra,alambra,pomegranate,pomegranate,330_ml,330_ml
5,1.0,ARTEMIS Apple Vinegar 500 ml 386207,Artemis Apple Vinegar 500 ml,artemis,artemis,apple vinegar,apple vinegar,500_ml,500_ml
6,1.0,GRIGORIOU Pizza Margarita 430g 498875,Grigoriou Pizza Margarita 430 g,grigoriou,grigoriou,pizza,pizza,430_g,430_g
7,1.0,KALAMATA PAPADIMITRIOU Balsamic Vinegar 500 ml 384110,Kalamata Papadimitriou Balsamic Vinegar 500 ml,kalamata papadimitriou,kalamata papadimitriou,balsamic vinegar,balsamic vinegar,500_ml,500_ml
8,1.0,GRIGORIOU Pizza Special 490g 498877,Grigoriou Pizza Special 490 g,grigoriou,grigoriou,pizza,pizza,490_g,490_g
9,1.0,KYKKOS Natural Mineral Water 6x1.5lt 134615,Kykkos Natural Mineral Water 6x1.5 L,kykkos,kykkos,mineral water,mineral water,9000_ml,9000_ml


## 27. Transfer manual labels to актуальный scored dataset v2

Теперь делаем более правильный training dataset для будущей ML-модели.

Раньше ручная разметка `match/not_match` была сделана в старом scored-файле `v1`. После этого мы обновили:

- brand rules
- product type rules
- candidate blocking v2
- scoring export v2

Поэтому теперь нельзя просто взять старый размеченный Excel как training dataset. Нужно перенести ручные метки на актуальную таблицу пар.

Главная идея:

```text
label переносим по ключу пары = item_id_sk + item_id_am
```

Так порядок строк уже не важен. Если пара была размечена раньше и она есть в новом актуальном scored dataset, ее метка переносится.

В этом блоке сохраняем две версии:

1. `full` — полная audit-версия, где есть и отдельные признаки, и наши ручные суммарные scores (`base_match_score`, `final_match_score`).
2. `ml_clean` — версия для ML-обучения без суммарных score-колонок, чтобы модель не училась просто повторять наш ручной weighted scoring.


In [117]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

# Старый файл, где ты руками размечала match/not_match.
ANNOTATED_PAIRS_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_v1_(labeled_manually).xlsx'
if not ANNOTATED_PAIRS_EXCEL_PATH.exists():
    ANNOTATED_PAIRS_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_v1.xlsx'

# Новый актуальный scored dataset после blocking v2/scoring v2.
CURRENT_SCORED_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_v2.csv'
CURRENT_SCORED_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_scored_with_image_links_v2.xlsx'

manual_pairs = pd.read_excel(ANNOTATED_PAIRS_EXCEL_PATH)

try:
    # Если блок scoring v2 уже выполнялся выше в этом же notebook run,
    # берем актуальную таблицу прямо из переменной.
    current_scored_pairs = candidate_pairs_scored.copy()
    current_scored_source = 'notebook variable: candidate_pairs_scored'
except NameError:
    if CURRENT_SCORED_CSV_PATH.exists():
        current_scored_pairs = pd.read_csv(CURRENT_SCORED_CSV_PATH, encoding='utf-8-sig')
        current_scored_source = CURRENT_SCORED_CSV_PATH
    elif CURRENT_SCORED_EXCEL_PATH.exists():
        current_scored_pairs = pd.read_excel(CURRENT_SCORED_EXCEL_PATH)
        current_scored_source = CURRENT_SCORED_EXCEL_PATH
    else:
        raise FileNotFoundError(
            'Не нашла актуальный scored v2 dataset. '
            'Сначала запусти blocking v2 и scoring v2 выше.'
        )

print('Manual labels source:', ANNOTATED_PAIRS_EXCEL_PATH)
print('Current scored source:', current_scored_source)
print('Manual rows:', len(manual_pairs))
print('Current scored rows:', len(current_scored_pairs))

required_columns = ['item_id_sk', 'item_id_am']
for column_name in required_columns:
    if column_name not in manual_pairs.columns:
        raise ValueError(f'В manual file нет колонки {column_name}')
    if column_name not in current_scored_pairs.columns:
        raise ValueError(f'В current scored dataset нет колонки {column_name}')

if 'match/not_match' not in manual_pairs.columns:
    raise ValueError('Не нашла колонку match/not_match в старом размеченном Excel.')


def make_pair_key(df):
    return (
        df['item_id_sk'].fillna('').astype(str).str.strip()
        + '||'
        + df['item_id_am'].fillna('').astype(str).str.strip()
    )


manual_pairs = manual_pairs.copy()
current_scored_pairs = current_scored_pairs.copy()

manual_pairs['_pair_key'] = make_pair_key(manual_pairs)
current_scored_pairs['_pair_key'] = make_pair_key(current_scored_pairs)

manual_pairs['match_label_clean'] = (
    manual_pairs['match/not_match']
    .fillna('')
    .astype(str)
    .str.strip()
    .replace({'1.0': '1', '0.0': '0'})
)

manual_labeled = manual_pairs.loc[
    manual_pairs['match_label_clean'].isin(['0', '1', '/'])
].copy()

label_counts_raw = (
    manual_pairs['match_label_clean']
    .replace({'': '(empty)'})
    .value_counts(dropna=False)
    .rename_axis('match_label_clean')
    .reset_index(name='row_count')
)

print('Manual label summary before transfer:')
display(label_counts_raw)

# Если одна и та же пара размечена несколько раз одинаково, это не страшно.
# Если одна и та же пара получила разные метки, такую пару лучше не переносить автоматически.
labels_per_pair = (
    manual_labeled
    .groupby('_pair_key')['match_label_clean']
    .agg(lambda values: sorted(set(values)))
    .reset_index(name='unique_labels')
)
labels_per_pair['label_count'] = labels_per_pair['unique_labels'].map(len)

conflicting_label_keys = set(
    labels_per_pair.loc[labels_per_pair['label_count'].gt(1), '_pair_key']
)

manual_label_map = (
    labels_per_pair
    .loc[labels_per_pair['label_count'].eq(1)]
    .copy()
)
manual_label_map['transferred_match_label'] = manual_label_map['unique_labels'].str[0]
manual_label_map = manual_label_map[['_pair_key', 'transferred_match_label']]

current_scored_labeled = current_scored_pairs.merge(
    manual_label_map,
    on='_pair_key',
    how='left'
)

current_scored_labeled['manual_match'] = pd.to_numeric(
    current_scored_labeled['transferred_match_label'].where(
        current_scored_labeled['transferred_match_label'].isin(['0', '1'])
    ),
    errors='coerce'
).astype('Int64')

transferred_summary = (
    current_scored_labeled['transferred_match_label']
    .fillna('(unlabeled)')
    .value_counts(dropna=False)
    .rename_axis('transferred_match_label')
    .reset_index(name='row_count')
)

manual_keys = set(manual_labeled['_pair_key'])
current_keys = set(current_scored_pairs['_pair_key'])

print('Transferred label summary on current scored v2:')
display(transferred_summary)
print('Manual labeled unique pairs:', manual_labeled['_pair_key'].nunique())
print('Manual labeled pairs found in current scored v2:', len(manual_keys & current_keys))
print('Manual labeled pairs not found in current scored v2:', len(manual_keys - current_keys))
print('Conflicting manually labeled pair keys:', len(conflicting_label_keys))


Manual labels source: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_candidate_pairs_scored_v1_(labeled_manually).xlsx
Current scored source: notebook variable: candidate_pairs_scored
Manual rows: 52268
Current scored rows: 53436
Manual label summary before transfer:


,match_label_clean,row_count
0,(empty),51330
1,0,446
2,1,436
3,/,56


Transferred label summary on current scored v2:


,transferred_match_label,row_count
0,(unlabeled),52498
1,0,446
2,1,436
3,/,56


Manual labeled unique pairs: 938
Manual labeled pairs found in current scored v2: 938
Manual labeled pairs not found in current scored v2: 0
Conflicting manually labeled pair keys: 0


### 27.1 Save v2 training datasets: full и ml_clean

Теперь сохраняем результат.

`full` версия нужна для проверки глазами: там остаются `base_match_score` и `final_match_score`, чтобы удобно сортировать и понимать, как наша старая ручная формула оценивала пару.

`ml_clean` версия нужна именно для будущего обучения модели. В ней убираем суммарные score-колонки:

- `base_match_score`
- `final_match_score`

Почему убираем: эти колонки уже являются ручной смесью других признаков. Если дать их модели, она может научиться повторять нашу формулу, а не самостоятельно подобрать веса по отдельным признакам.

Дополнительно сохраняем `for_ml_predict` версию. Это такая же `ml_clean` таблица, но для строк без обучающей метки `0/1`. Ее можно будет подать в будущую модель, чтобы получить предсказания для еще не размеченных пар.


In [118]:
ML_TRAINING_FULL_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_full.xlsx'
ML_TRAINING_FULL_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_full.csv'

ML_TRAINING_CLEAN_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_ml_clean.xlsx'
ML_TRAINING_CLEAN_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_ml_clean.csv'

ML_UNSURE_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_unsure_labels_v2.xlsx'
ML_UNLABELED_POOL_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_unlabeled_pool_v2.xlsx'
ML_PREDICT_POOL_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_dataset_v2_ml_clean_for_ml_predict.xlsx'
ML_PREDICT_POOL_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_dataset_v2_ml_clean_for_ml_predict.csv'
ML_CONFLICTS_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_label_conflicts_v2.xlsx'

training_pairs_full = current_scored_labeled.loc[
    current_scored_labeled['transferred_match_label'].isin(['0', '1'])
].copy()

unsure_pairs_v2 = current_scored_labeled.loc[
    current_scored_labeled['transferred_match_label'].eq('/')
].copy()

unlabeled_pool_v2 = current_scored_labeled.loc[
    current_scored_labeled['transferred_match_label'].isna()
].copy()

# Для будущего predict-файла берем все строки без обучающей метки 0/1.
# Сюда входят и полностью пустые метки, и строки с '/', потому что модель должна будет предсказать их заново.
predict_pool_v2 = current_scored_labeled.loc[
    ~current_scored_labeled['transferred_match_label'].isin(['0', '1'])
].copy()
predict_pool_v2['transferred_match_label'] = ''
predict_pool_v2['manual_match'] = pd.NA

score_columns_to_drop_for_ml = [
    'base_match_score',
    'final_match_score',
    'final_score_bucket',
]

training_pairs_ml_clean = training_pairs_full.drop(
    columns=[column_name for column_name in score_columns_to_drop_for_ml if column_name in training_pairs_full.columns]
).copy()

predict_pool_ml_clean = predict_pool_v2.drop(
    columns=[column_name for column_name in score_columns_to_drop_for_ml if column_name in predict_pool_v2.columns]
).copy()
predict_pool_ml_clean = predict_pool_ml_clean[training_pairs_ml_clean.columns]

class_balance_v2 = (
    training_pairs_full['manual_match']
    .value_counts(sort=False)
    .rename_axis('manual_match')
    .reset_index(name='row_count')
)
class_balance_v2['share_pct'] = (
    class_balance_v2['row_count'] / class_balance_v2['row_count'].sum() * 100
).round(1)

print('Training rows with transferred 0/1 labels:', len(training_pairs_full))
print('Unsure rows transferred with / label:', len(unsure_pairs_v2))
print('Unlabeled current scored v2 rows:', len(unlabeled_pool_v2))
print('Rows for ML predict pool:', len(predict_pool_ml_clean))
print('Training + predict rows:', len(training_pairs_ml_clean) + len(predict_pool_ml_clean))
print('Current scored v2 rows:', len(current_scored_labeled))
print('Training + predict equals current scored v2:', (len(training_pairs_ml_clean) + len(predict_pool_ml_clean)) == len(current_scored_labeled))
print('Columns in full dataset:', len(training_pairs_full.columns))
print('Columns in ml_clean dataset:', len(training_pairs_ml_clean.columns))
print('Columns in for_ml_predict dataset:', len(predict_pool_ml_clean.columns))
print('ml_clean and for_ml_predict have same columns:', list(training_pairs_ml_clean.columns) == list(predict_pool_ml_clean.columns))
print('Dropped from ml_clean:', [c for c in score_columns_to_drop_for_ml if c in training_pairs_full.columns])

print('Class balance for transferred v2 training dataset:')
display(class_balance_v2)

if 'final_match_score' in training_pairs_full.columns:
    score_bins = [-0.001, 0.30, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
    score_labels = ['<0.30', '0.30-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-1.00']

    training_pairs_full['final_score_bucket_for_audit'] = pd.cut(
        training_pairs_full['final_match_score'],
        bins=score_bins,
        labels=score_labels
    )

    score_bucket_label_summary_v2 = pd.crosstab(
        training_pairs_full['final_score_bucket_for_audit'],
        training_pairs_full['manual_match'],
        dropna=False
    ).reset_index()

    print('Transferred labels by current final_match_score bucket:')
    display(score_bucket_label_summary_v2)

exact_pair_duplicates_v2 = training_pairs_full.duplicated(['item_id_sk', 'item_id_am']).sum()
name_pair_duplicates_v2 = training_pairs_full.duplicated(['product_name_sk', 'product_name_am']).sum()

print('Exact item_id pair duplicate rows:', exact_pair_duplicates_v2)
print('Product name pair duplicate rows:', name_pair_duplicates_v2)

training_pairs_full.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_TRAINING_FULL_EXCEL_PATH, index=False)
training_pairs_full.drop(columns=['_pair_key'], errors='ignore').to_csv(ML_TRAINING_FULL_CSV_PATH, index=False, encoding='utf-8-sig')

# пока закомментировала, файл открыт у меня
training_pairs_ml_clean.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_TRAINING_CLEAN_EXCEL_PATH, index=False)
training_pairs_ml_clean.drop(columns=['_pair_key'], errors='ignore').to_csv(ML_TRAINING_CLEAN_CSV_PATH, index=False, encoding='utf-8-sig')

unsure_pairs_v2.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_UNSURE_EXCEL_PATH, index=False)
unlabeled_pool_v2.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_UNLABELED_POOL_EXCEL_PATH, index=False)

predict_pool_ml_clean.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_PREDICT_POOL_EXCEL_PATH, index=False)
predict_pool_ml_clean.drop(columns=['_pair_key'], errors='ignore').to_csv(ML_PREDICT_POOL_CSV_PATH, index=False, encoding='utf-8-sig')

if conflicting_label_keys:
    conflict_pairs = manual_labeled.loc[
        manual_labeled['_pair_key'].isin(conflicting_label_keys)
    ].copy()
    conflict_pairs.drop(columns=['_pair_key'], errors='ignore').to_excel(ML_CONFLICTS_EXCEL_PATH, index=False)
    print('Saved conflicts Excel to:', ML_CONFLICTS_EXCEL_PATH)

print('Saved full training Excel to:', ML_TRAINING_FULL_EXCEL_PATH)
print('Saved full training CSV to:', ML_TRAINING_FULL_CSV_PATH)
print('Saved ml_clean training Excel to:', ML_TRAINING_CLEAN_EXCEL_PATH)
print('Saved ml_clean training CSV to:', ML_TRAINING_CLEAN_CSV_PATH)
print('Saved unsure labels Excel to:', ML_UNSURE_EXCEL_PATH)
print('Saved unlabeled pool Excel to:', ML_UNLABELED_POOL_EXCEL_PATH)
print('Saved for ML predict Excel to:', ML_PREDICT_POOL_EXCEL_PATH)
print('Saved for ML predict CSV to:', ML_PREDICT_POOL_CSV_PATH)


Training rows with transferred 0/1 labels: 882
Unsure rows transferred with / label: 56
Unlabeled current scored v2 rows: 52498
Rows for ML predict pool: 52554
Training + predict rows: 53436
Current scored v2 rows: 53436
Training + predict equals current scored v2: True
Columns in full dataset: 30
Columns in ml_clean dataset: 28
Columns in for_ml_predict dataset: 28
ml_clean and for_ml_predict have same columns: True
Dropped from ml_clean: ['base_match_score', 'final_match_score']
Class balance for transferred v2 training dataset:


,manual_match,row_count,share_pct
0,1,436,49.4
1,0,446,50.6


Transferred labels by current final_match_score bucket:


manual_match,final_score_bucket_for_audit,0,1
0,<0.30,0,0
1,0.30-0.50,52,0
2,0.50-0.60,91,0
3,0.60-0.70,66,2
4,0.70-0.80,133,34
5,0.80-0.90,91,79
6,0.90-1.00,13,321


Exact item_id pair duplicate rows: 0
Product name pair duplicate rows: 39
Saved full training Excel to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_match_training_dataset_v2_full.xlsx
Saved full training CSV to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_match_training_dataset_v2_full.csv
Saved ml_clean training Excel to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_match_training_dataset_v2_ml_clean.xlsx
Saved ml_clean training CSV to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_match_training_dataset_v2_ml_clean.csv
Saved unsure labels Excel to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_match_unsure_labels_v2.xlsx
Saved unlabeled pool Excel to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_

### 27.2 TF-IDF diagnostics for candidate blocking coverage

Заметила, что до 0,8 score (то есть до конца адекватного количества матчей) всего 1272 пары, причем часть из них еще и повторяется. С учетом того, что в Склавенитисе всего около 3300 товаров, это выглядит слишком мало. Поэтому проверяю, не теряет ли candidate blocking хорошие пары еще до scoring.

Здесь делаю диагностический лист для ручного просмотра:

- `unblocked_skl_top5` — берем 50 товаров Sklavenitis, которые вообще не попали в candidate blocking, и для каждого ищем top-5 товаров Alpha Omega по TF-IDF похожести.

TF-IDF считаем двумя способами:

- word TF-IDF с `ngram_range=(1, 2)`, чтобы учитывать одиночные слова и пары слов
- char TF-IDF с `ngram_range=(3, 5)`, чтобы ловить похожесть строк, опечатки, дефисы и небольшие отличия в написании

Для ранжирования использую:

```text
tfidf_rescue_similarity = max(tfidf_word_1_2_similarity, tfidf_char_3_5_similarity)
```

Это не добавляет пары в основной blocking автоматически. Это diagnostic export: я хочу глазами понять, действительно ли TF-IDF rescue находит полезные пары для товаров, которые вообще не попали в candidate blocking. Только после этого стоит решать, нужно ли добавлять новое правило в blocking v2.

Проверка по текущему blocking: он уже использует manual-поля, которые есть выше. В блоке candidate blocking бренд берется из `brand_pred_with_manual`, product type — из `product_type_pred_with_manual`, а остаточный текст — из `product_core_without_brand_with_manual`.


In [119]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

SKL_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_with_manual_triggers.csv'
ALPHA_PATH = OUTPUT_DIR / 'alphamega_brand_product_type_with_manual_triggers.csv'
CANDIDATE_PAIRS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_candidate_pairs_v1.csv'
TFIDF_BLOCKING_DIAGNOSTICS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_tfidf_blocking_diagnostics_v1.xlsx'

sk_tfidf_raw = pd.read_csv(SKL_PATH, encoding='utf-8-sig').fillna('')
alpha_tfidf_raw = pd.read_csv(ALPHA_PATH, encoding='utf-8-sig').fillna('')
candidate_pairs_for_diag = pd.read_csv(CANDIDATE_PAIRS_PATH, encoding='utf-8-sig').fillna('')

# item_id создавались в candidate blocking из номера строки после reset_index(drop=True).
# Поэтому здесь повторяем ту же схему, чтобы ID совпали.
sk_tfidf = sk_tfidf_raw.copy().reset_index(drop=True)
alpha_tfidf = alpha_tfidf_raw.copy().reset_index(drop=True)
sk_tfidf['item_id_sk'] = ['sk_' + str(i) for i in range(len(sk_tfidf))]
alpha_tfidf['item_id_am'] = ['am_' + str(i) for i in range(len(alpha_tfidf))]


def normalize_tfidf_text(value):
    text = str(value).lower().strip()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def choose_tfidf_text(row):
    # Для диагностики берем product_core, потому что именно он используется в scoring TF-IDF.
    # Если он пустой, берем очищенное название как запасной вариант.
    product_core = normalize_tfidf_text(row.get('product_core', ''))
    if product_core:
        return product_core
    product_name_clean = normalize_tfidf_text(row.get('product_name_clean', ''))
    if product_name_clean:
        return product_name_clean
    return normalize_tfidf_text(row.get('product_name', ''))


sk_tfidf['tfidf_search_text'] = sk_tfidf.apply(choose_tfidf_text, axis=1)
alpha_tfidf['tfidf_search_text'] = alpha_tfidf.apply(choose_tfidf_text, axis=1)

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w+\b',
    lowercase=False
)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    lowercase=False
)

all_tfidf_texts = pd.concat(
    [sk_tfidf['tfidf_search_text'], alpha_tfidf['tfidf_search_text']],
    ignore_index=True
)

word_vectorizer.fit(all_tfidf_texts)
char_vectorizer.fit(all_tfidf_texts)

alpha_word_vectors = word_vectorizer.transform(alpha_tfidf['tfidf_search_text'])
alpha_char_vectors = char_vectorizer.transform(alpha_tfidf['tfidf_search_text'])

existing_candidate_pair_keys = set(
    zip(candidate_pairs_for_diag['item_id_sk'], candidate_pairs_for_diag['item_id_am'])
)

# Sklavenitis товары, которые вообще не попали в candidate blocking.
sk_items_with_candidates = set(candidate_pairs_for_diag['item_id_sk'])
unblocked_skl_items = (
    sk_tfidf.loc[~sk_tfidf['item_id_sk'].isin(sk_items_with_candidates)]
    .head(50)
    .copy()
)


def build_top5_tfidf_matches(target_skl_items, sheet_source_label):
    rows = []
    if target_skl_items.empty:
        return pd.DataFrame(rows)

    target_word_vectors = word_vectorizer.transform(target_skl_items['tfidf_search_text'])
    target_char_vectors = char_vectorizer.transform(target_skl_items['tfidf_search_text'])

    word_similarity_matrix = (target_word_vectors @ alpha_word_vectors.T).toarray()
    char_similarity_matrix = (target_char_vectors @ alpha_char_vectors.T).toarray()
    rescue_similarity_matrix = np.maximum(word_similarity_matrix, char_similarity_matrix)

    for target_position, (_, sk_row) in enumerate(target_skl_items.reset_index(drop=True).iterrows()):
        top_alpha_positions = np.argsort(-rescue_similarity_matrix[target_position])[:5]

        for rank, alpha_position in enumerate(top_alpha_positions, start=1):
            alpha_row = alpha_tfidf.iloc[alpha_position]
            pair_key = (sk_row['item_id_sk'], alpha_row['item_id_am'])
            existing_pair = pair_key in existing_candidate_pair_keys

            rows.append({
                'diagnostic_source': sheet_source_label,
                'tfidf_rank': rank,
                'tfidf_rescue_similarity': rescue_similarity_matrix[target_position, alpha_position],
                'tfidf_word_1_2_similarity': word_similarity_matrix[target_position, alpha_position],
                'tfidf_char_3_5_similarity': char_similarity_matrix[target_position, alpha_position],
                'already_in_candidate_blocking': existing_pair,
                'item_id_sk': sk_row['item_id_sk'],
                'item_id_am': alpha_row['item_id_am'],
                'product_name_sk': sk_row.get('product_name', ''),
                'product_name_am': alpha_row.get('product_name', ''),
                'product_core_sk': sk_row.get('product_core', ''),
                'product_core_am': alpha_row.get('product_core', ''),
                'brand_sk': sk_row.get('brand_pred_with_manual', ''),
                'brand_am': alpha_row.get('brand_pred_with_manual', ''),
                'product_type_sk': sk_row.get('product_type_pred_with_manual', ''),
                'product_type_am': alpha_row.get('product_type_pred_with_manual', ''),
                'size_total_base_sk': sk_row.get('size_total_base', ''),
                'size_base_unit_sk': sk_row.get('size_base_unit', ''),
                'size_total_base_am': alpha_row.get('size_total_base', ''),
                'size_base_unit_am': alpha_row.get('size_base_unit', ''),
                'image_url_sk': sk_row.get('image_url', ''),
                'image_url_am': alpha_row.get('image_url', ''),
            })

    return pd.DataFrame(rows)


unblocked_top5_tfidf = build_top5_tfidf_matches(
    unblocked_skl_items,
    'sklavenitis_without_any_candidate_pairs'
)

# пока закомментила, чтобы при перезапуске kernel не создавал опять
# with pd.ExcelWriter(TFIDF_BLOCKING_DIAGNOSTICS_PATH) as writer:
#     unblocked_top5_tfidf.to_excel(writer, sheet_name='unblocked_skl_top5', index=False)

print('Sklavenitis items without any candidate pairs:', len(unblocked_skl_items))
print('Rows in unblocked_skl_top5 sheet:', len(unblocked_top5_tfidf))
print('Saved TF-IDF blocking diagnostics to:', TFIDF_BLOCKING_DIAGNOSTICS_PATH)

print('Preview: unblocked_skl_top5')
display(unblocked_top5_tfidf.head(20))


Sklavenitis items without any candidate pairs: 50
Rows in unblocked_skl_top5 sheet: 250
Saved TF-IDF blocking diagnostics to: C:\Users\valeriya.brovchuk\Documents\projects\price_comparison_grocery\output\sklavenitis_alphamega_tfidf_blocking_diagnostics_v1.xlsx
Preview: unblocked_skl_top5


,diagnostic_source,tfidf_rank,tfidf_rescue_similarity,tfidf_word_1_2_similarity,tfidf_char_3_5_similarity,already_in_candidate_blocking,item_id_sk,item_id_am,product_name_sk,product_name_am,...,brand_sk,brand_am,product_type_sk,product_type_am,size_total_base_sk,size_base_unit_sk,size_total_base_am,size_base_unit_am,image_url_sk,image_url_am
0,sklavenitis_without_any_candidate_pairs,1,0.235023,0.235023,0.222353,False,sk_1,am_7578,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,Ion Milk Chocolate with Almonds 100 g,...,macedonian,ion,halva,chocolate,400.0,g,100.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/707007.jpg&AlternativeImage=/Images/missing_image.jpg
1,sklavenitis_without_any_candidate_pairs,2,0.235023,0.235023,0.222353,False,sk_1,am_7135,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,Ion Milk Chocolate with Almonds 45 g,...,macedonian,ion,halva,chocolate,400.0,g,45.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/104308.jpg&AlternativeImage=/Images/missing_image.jpg
2,sklavenitis_without_any_candidate_pairs,3,0.235023,0.235023,0.222353,False,sk_1,am_7368,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,Ion Milk Chocolate with Almonds 200 g,...,macedonian,ion,halva,chocolate,400.0,g,200.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/138997.jpg&AlternativeImage=/Images/missing_image.jpg
3,sklavenitis_without_any_candidate_pairs,4,0.210210,0.210210,0.142637,False,sk_1,am_319,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,Algida 4 Cornetto Ice Creams Vegan & Gluten Free 360 ml,...,macedonian,,halva,,400.0,g,360.0,ml,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/831834.jpg&AlternativeImage=/Images/missing_image.jpg
4,sklavenitis_without_any_candidate_pairs,5,0.196341,0.176145,0.196341,False,sk_1,am_7069,MACEDONIAN Halva with Almonds Vegan Gluten-Free 400g 119461,Ion Milk Chocolate with Almonds and Stevia 60 g,...,macedonian,ion,halva,chocolate,400.0,g,60.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d156,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/808838.jpg&AlternativeImage=/Images/missing_image.jpg
5,sklavenitis_without_any_candidate_pairs,1,0.218465,0.218465,0.136806,False,sk_2,am_319,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",Algida 4 Cornetto Ice Creams Vegan & Gluten Free 360 ml,...,makedonikos,,halva,,400.0,g,360.0,ml,https://imageproxy.wolt.com/assets/697364789b938fe622d8d157,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/831834.jpg&AlternativeImage=/Images/missing_image.jpg
6,sklavenitis_without_any_candidate_pairs,2,0.177813,0.177813,0.138535,False,sk_2,am_8578,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",AB Tahini with Cocoa 300 g,...,makedonikos,,halva,tahini,400.0,g,300.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d157,/Admin/Public/GetImage.ashx?width=640&crop=5&FillCanvas=True&DoNotUpscale=true&Compression=75&image=/Files/Images/Products/897717.jpg&AlternativeImage=/Images/missing_image.jpg
7,sklavenitis_without_any_candidate_pairs,3,0.177417,0.177417,0.128252,False,sk_2,am_7552,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten Free 400g 119459",Olympos Tahini with Cocoa 300 g,...,makedonikos,olympos,halva,tahini,400.0,g,300.0,g,https://imageproxy.wolt.com/assets/697364789b938fe622d8d157,/Admin/Public

## 28. Train baseline logistic regression matcher

Следующий этап — обучить первую простую ML-модель, которая будет предсказывать вероятность match для каждой candidate pair.

### Цель

Хотим заменить ручные веса scoring более гибкой моделью + доделать матчинг всех пар

Сейчас мы вручную считаем:

```text
final_match_score = weighted similarity score + penalties + grade adjustment
```

А ML-модель будет учиться на проставленных мной ручных метках:

```text
manual_match = 1 или 0
```

В общем, модель посмотрит на уже посчитанные признаки и сама оценит, какие из них сильнее помогают отличать match от non-match.

### Первую модель делаем простой

Начинаем с logistic regression, потому что она:

- хорошо подходит для задачи binary classification: match или not match
- даёт вероятность `ml_match_probability` от `0` до `1`
- достаточно прозрачная: можно посмотреть, какие признаки повышают или понижают вероятность match
- проще для проверки и обучения, чем более сложные модели

### Какие признаки используем

Для первой версии берём отдельные признаки пары:

- `tfidf_char_core_similarity`
- `tfidf_word_core_similarity`
- `product_type_similarity`
- `brand_similarity`
- `brand_penalty`
- `size_similarity`
- `grade_adjustment`

В `full` dataset ещё есть `base_match_score` и `final_match_score`, но для `ml_clean` версии мы их убираем. Эти две колонки уже являются нашей ручной weighted-формулой, а ML-модель должна учиться весам по отдельным признакам сама.

### Как будем проверять качество

Training dataset разделим на две части:

- train: модель учится на этих строках
- test: модель не видит эти строки во время обучения, а мы проверяем качество на них

Смотреть будем не только accuracy, потому что в matching-задачах важны ошибки двух типов:

- false positive: модель сказала match, но товары разные
- false negative: модель сказала not match, но товары одинаковые

Поэтому полезные метрики:

- precision
- recall
- F1-score
- confusion matrix
- ручной просмотр самых спорных предсказаний

### Как модель поможет разметить остальное

После обучения применим модель ко всем candidate pairs без ручной метки и получим:

```text
ml_match_probability
```

Дальше буду смотреть:

- пары, где модель сильно уверена в match
- пары, где модель сильно уверена в not match
- пары в спорной зоне, например `0.35-0.75`
- пары, где `final_match_score` и `ml_match_probability` сильно расходятся

Так получится active learning: ты размечаешь самые полезные случаи, мы переобучаем модель, и постепенно качество matcher становится лучше.


### 28.1 Cross-validation evaluator по рассчитанным признакам

Цель этого блока:

- взять размеченный `v2_ml_clean` dataset
- использовать только уже рассчитанные числовые признаки пары
- сравнить несколько простых моделей
- проверить качество через `StratifiedGroupKFold`
- подобрать разные probability thresholds

В признаки не добавляем исходные названия товаров, бренды, product type, картинки или другие raw-поля. Берем только рассчитанные similarity/penalty/adjustment показатели.

Группировку для cross-validation делаем по `item_id_sk`, чтобы пары с одним и тем же товаром Sklavenitis не попадали одновременно и в train, и в validation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

# SimpleImputer нужен, чтобы заполнять пропуски в признаках. Заполняет NaN медианой или другим значением
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# метрики качества модели
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Stratified — сохраняет баланс классов 0/1 в fold'ах
# Group — не разрывает один item_id_sk между train и validation
# KFold — делит данные на несколько частей
from sklearn.model_selection import StratifiedGroupKFold

# Pipeline нужен, чтобы собрать несколько шагов в одну цепочку:
# заполни пропуски → отмасштабируй → обучи модель
from sklearn.pipeline import Pipeline

# StandardScaler приводит признаки к похожему масштабу.
# Например, если один признак от 0 до 1, а другой от 0 до 100, логистической регрессии может быть неудобно. Scaler делает признаки более сопоставимыми.
# Для деревьев он обычно не нужен, но для логистической регрессии — полезен.
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

ML_TRAINING_CLEAN_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_ml_clean.csv'
ML_CV_RESULTS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_crossval_threshold_results_v1.csv'
ML_CV_PREDICTIONS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_crossval_oof_predictions_v1.csv'
ML_CV_TOP_CANDIDATES_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_crossval_top_candidates_v1.csv'

ml_train = pd.read_csv(ML_TRAINING_CLEAN_CSV_PATH, encoding='utf-8-sig')

# проверка, что есть разметка на 0 и 1 
if 'manual_match' not in ml_train.columns:
    raise ValueError('Не нашла manual_match в ml_clean dataset.') #если этой колонки нет, ML обучать невозможно, потому что модели не на чем учиться

# список признаков для обучения - берем только уже рассчитанные числовые показатели, не берем исходные колонки и суммирующие итоги
candidate_feature_columns = [
    'tfidf_char_core_similarity',
    'tfidf_word_core_similarity',
    'product_type_similarity',
    'brand_similarity',
    'brand_penalty',
    'size_similarity',
    'grade_adjustment',
]

feature_columns = [
    column_name
    for column_name in candidate_feature_columns
    if column_name in ml_train.columns
]

# на всякий случай, если чего-то не окажется из признаков или не распознает
missing_feature_columns = sorted(set(candidate_feature_columns) - set(feature_columns))
if missing_feature_columns:
    print('Нет части ожидаемых feature columns:', missing_feature_columns)

if not feature_columns:
    raise ValueError('Не нашла ни одной рассчитанной feature column для обучения.')


# оставляем только 0 и 1 в размеченной колонке, остальное если есть мусор - отфильтровываем (но я старалась чтобы и не было ничего кроме 0 и 1)
ml_train = ml_train.loc[ml_train['manual_match'].isin([0, 1, '0', '1'])].copy()

# если был записан как строчный, то привожу к типу число
ml_train['manual_match'] = ml_train['manual_match'].astype(int)


# X — это таблица признаков, на которых будет обучаться модель
# pd.to_numeric(errors='coerce') означает: попробуй превратить значения в числа, а если не получилось — поставь NaN
X = ml_train[feature_columns].apply(pd.to_numeric, errors='coerce')

# y — это правильные ответы (0 или 1), то есть то, что модель должна научиться предсказывать
y = ml_train['manual_match']

# groups = группы для StratifiedGroupKFold. 
# Смысл:candidate pairs для одного товара должны попадать или в train, или в validation, но не одновременно, иначе это leakage данных и модель будет считать, что она лучше предсказывает, чем на самом деле
groups = ml_train['item_id_sk'].fillna('').astype(str)

print('Training rows:', len(ml_train))
print('Feature columns:', feature_columns)
print('Groups by item_id_sk:', groups.nunique())
print('Class balance:')
display(
    y.value_counts(sort=False)
    .rename_axis('manual_match')
    .reset_index(name='row_count')
)

# выбираем количество fold'ов для cross-validation ( то есть как бы на сколько промежутков делим датасет). Обычно 5
n_splits = 5

# здесь создаётся объект, который будет делить данные на train/validation
# Он старается одновременно:
# 1. сохранить баланс классов 0/1 (Stratified)
# 2. не разрывать группы item_id_sk (Group)
# 3. перемешать группы перед разбиением (shuffle=True)
# 4. сделать результат воспроизводимым через random_state=42
cv = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)


# 1. SimpleImputer(strategy='median')
#    заполняет пропуски медианой

# 2. StandardScaler()
#    масштабирует признаки

# 3. LogisticRegression(...)
#    обучает логистическую регрессию
models = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300,   #300 деревьев
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,          #использовать все доступные ядра процессора
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(
            max_iter=150,      #количество boosting-итераций = сколько маленьких моделей строим последовательно
            learning_rate=0.05,  #маленький шаг обучения, т.е. модель учится осторожнее
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=42,
        )),
    ]),
}

# есть вероятность, что относится к 0 или 1, но чтобы точно определить в определенный класс - нужен порог (типа как >=0,5 в математике - это округление в большую сторону)

# Создаётся список порогов:
# 0.10, 0.15, 0.20, ..., 0.95
thresholds = np.round(np.arange(0.10, 0.96, 0.05), 2)

# создаются пустые списки для результатов
oof_prediction_frames = []
threshold_result_rows = []

# цикл по моделям. код по очереди запускает logistic_regression, random_forest, hist_gradient_boosting
for model_name, model in models.items():
    print('Running model:', model_name)

    oof_probability = pd.Series(np.nan, index=ml_train.index, dtype='float64') #сюда будут записываться вероятности match для каждой строки. сначала везде NaN
    fold_ids = pd.Series(np.nan, index=ml_train.index, dtype='float64')   #сюда будет записано, в каком fold строка была validation


    
    # Тру cross-validation

    # cv.split(...) возвращает индексы: train_idx — строки для обучения, valid_idx — строки для проверки

    # Для каждого fold: модель обучается на train, предсказывает на validation
    for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups=groups), start=1):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        # clone создаёт чистую копию pipeline для каждого fold, чтобы состояние модели из прошлого fold точно не переиспользовалось
        fold_model = clone(model)

        # модель обучается на train-части
        fold_model.fit(X_train, y_train)

        # модель предсказывает вероятность класса 1 для validation-части
        # out-of-fold prediction - это когда модель делает предсказания по данным, которых не было в обучающей выборке. Это честнее результат даст
        oof_probability.iloc[valid_idx] = fold_model.predict_proba(X_valid)[:, 1]
        fold_ids.iloc[valid_idx] = fold_id



# ROC-AUC оценивает, насколько хорошо модель ранжирует пары: настоящие match должны получать более высокие вероятности, настоящие not match — более низкие
# try/except нужен, потому что ROC-AUC может упасть, если в данных какой-то проблемный случай, например только один класс
    try:
        auc_value = roc_auc_score(y, oof_probability)
    except ValueError:
        auc_value = np.nan



# сохраняем OOF-предсказания для этой модели
    oof_frame = ml_train[[
        'item_id_sk',
        'item_id_am',
        'product_name_sk',
        'product_name_am',
        'manual_match',
    ]].copy()
    oof_frame['model_name'] = model_name    #какая модель дала prediction
    oof_frame['fold_id'] = fold_ids.astype('Int64')   #в каком fold строка была validation
    oof_frame['ml_match_probability'] = oof_probability   #вероятность match
    oof_prediction_frames.append(oof_frame)

    for threshold in thresholds:
        y_pred = (oof_probability >= threshold).astype(int)   # для каждого порога переводится вероятность в класс (0 или 1), зависит от того какой порог (0,5 0,7 и тд) получили
        
        # tn — true negative: реально not match, модель сказала not match
        # fp — false positive: реально not match, модель сказала match
        # fn — false negative: реально match, модель сказала not match
        # tp — true positive: реально match, модель сказала match

        # Для matching особенно важен fp, потому что это неправильные auto-match
        tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()

        threshold_result_rows.append({
            'model_name': model_name,
            'threshold': threshold,
            'roc_auc': auc_value,
            'accuracy': accuracy_score(y, y_pred),
            'precision': precision_score(y, y_pred, zero_division=0),  #zero_division=0 нужен, чтобы метрика не падала, если модель при каком-то threshold вообще не предсказала ни одного match
            'recall': recall_score(y, y_pred, zero_division=0),
            'f1': f1_score(y, y_pred, zero_division=0),
            'tp': int(tp),
            'fp': int(fp),
            'tn': int(tn),
            'fn': int(fn),
            'predicted_match_count': int(tp + fp),
        })

cv_threshold_results = pd.DataFrame(threshold_result_rows)

# Для auto-match важнее не максимальный F1, а очень высокий precision: лучше отправить сомнительную пару на review, чем автоматически склеить разные товары
MIN_AUTO_MATCH_PRECISION = 0.95
high_precision_policy = (
    cv_threshold_results
    .loc[
        cv_threshold_results['precision'].ge(MIN_AUTO_MATCH_PRECISION)
        & cv_threshold_results['predicted_match_count'].gt(0)
    ]
    .sort_values(['model_name', 'recall', 'precision', 'threshold'], ascending=[True, False, False, True])
    .groupby('model_name', as_index=False)
    .head(1)
    .sort_values(['recall', 'precision'], ascending=False)
)

# Объединяю OOF predictions всех моделей в одну таблицу
cv_oof_predictions = pd.concat(oof_prediction_frames, ignore_index=True)

# В matching-задаче важно не только оценить каждую пару, но и выбрать лучшего кандидата Alpha Omega для каждого товара Sklavenitis
cv_oof_predictions = cv_oof_predictions.sort_values(
    ['model_name', 'item_id_sk', 'ml_match_probability'],
    ascending=[True, True, False]
).reset_index(drop=True)
cv_oof_predictions['candidate_rank'] = (
    cv_oof_predictions
    .groupby(['model_name', 'item_id_sk'])['ml_match_probability']
    .rank(method='first', ascending=False)
    .astype(int)
)

# gap_to_second показывает, насколько top-1 кандидат увереннее второго кандидата: большой отрыв полезен для auto-match, маленький отрыв лучше отправлять на review
top_candidate_rows = []
for (model_name, item_id_sk), group in cv_oof_predictions.groupby(['model_name', 'item_id_sk'], sort=False):
    sorted_group = group.sort_values('ml_match_probability', ascending=False)
    top1 = sorted_group.iloc[0].copy()
    top2_probability = sorted_group['ml_match_probability'].iloc[1] if len(sorted_group) > 1 else np.nan
    top1['top2_probability'] = top2_probability
    top1['gap_to_second'] = top1['ml_match_probability'] - top2_probability if pd.notna(top2_probability) else np.nan
    top1['candidate_count_for_item_sk'] = len(sorted_group)
    top_candidate_rows.append(top1)

cv_top_candidates = pd.DataFrame(top_candidate_rows)

# Сохраняем результаты в CSV
cv_threshold_results.to_csv(ML_CV_RESULTS_PATH, index=False, encoding='utf-8-sig')
cv_oof_predictions.to_csv(ML_CV_PREDICTIONS_PATH, index=False, encoding='utf-8-sig')
cv_top_candidates.to_csv(ML_CV_TOP_CANDIDATES_PATH, index=False, encoding='utf-8-sig')

print('Saved threshold results to:', ML_CV_RESULTS_PATH)
print('Saved out-of-fold predictions to:', ML_CV_PREDICTIONS_PATH)
print('Saved top candidates to:', ML_CV_TOP_CANDIDATES_PATH)

# Лучший threshold по F1 для каждой модели
print('Best threshold per model by F1:')
best_by_f1 = (
    cv_threshold_results
    .sort_values(['model_name', 'f1', 'precision', 'recall'], ascending=[True, False, False, False])
    .groupby('model_name', as_index=False)
    .head(1)
    .sort_values('f1', ascending=False)
)
display(best_by_f1)

print(f'Best threshold per model with precision >= {MIN_AUTO_MATCH_PRECISION}:')
display(high_precision_policy)

print('Top candidates by model and item_id_sk:')
display(
    cv_top_candidates
    .sort_values(['model_name', 'ml_match_probability', 'gap_to_second'], ascending=[True, False, False])
    .head(30)
)


# Топ-30 threshold-результатов по F1
print('Threshold table sorted by F1:')
display(
    cv_threshold_results
    .sort_values(['f1', 'precision', 'recall'], ascending=False)
    .head(30)
)


### 28.2 Final ML predict для всех неразмеченных candidate pairs

В прошлом блоке мы делали cross-validation: модель училась на части размеченных строк и проверялась на другой части размеченных строк. Поэтому `cv_oof_predictions` и `cv_top_candidates` относятся только к ручной разметке.

Здесь делаем другой шаг: обучаем выбранную модель на всей ручной разметке `0/1`, а затем применяем ее к `for_ml_predict` файлу, где лежат неразмеченные candidate pairs.

Логика:

- берем только рассчитанные признаки, без исходных текстовых колонок и без суммирующего ручного score;
- выбираем рабочий threshold из CV-таблицы по правилу `precision >= 0.95`, а среди таких вариантов берем максимальный `recall`;
- обучаем финальную модель на всех размеченных строках;
- считаем `ml_match_probability` для каждой неразмеченной пары;
- добавляем `ml_predicted_match_pair`, то есть 0/1 по выбранному threshold;
- отдельно ранжируем кандидатов внутри каждого `item_id_sk`, потому что для реального matching обычно нужен лучший кандидат, а не просто независимая оценка каждой пары.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

ML_TRAINING_CLEAN_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_training_dataset_v2_ml_clean.csv'
ML_PREDICT_POOL_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_match_dataset_v2_ml_clean_for_ml_predict.csv'
ML_CV_RESULTS_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_crossval_threshold_results_v1.csv'

ML_FINAL_PREDICTIONS_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_predictions_all_candidate_pairs_v1.csv'
ML_FINAL_PREDICTIONS_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_predictions_all_candidate_pairs_v1.xlsx'
ML_FINAL_TOP1_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_predictions_top1_candidates_v1.csv'
ML_FINAL_TOP1_EXCEL_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_ml_predictions_top1_candidates_v1.xlsx'

ml_train = pd.read_csv(ML_TRAINING_CLEAN_CSV_PATH, encoding='utf-8-sig')
ml_predict_pool = pd.read_csv(ML_PREDICT_POOL_CSV_PATH, encoding='utf-8-sig')
cv_threshold_results = pd.read_csv(ML_CV_RESULTS_PATH, encoding='utf-8-sig')

# Берем только рассчитанные числовые признаки: модель должна учиться на similarity-показателях, а не на исходных названиях товаров
candidate_feature_columns = [
    'tfidf_char_core_similarity',
    'tfidf_word_core_similarity',
    'product_type_similarity',
    'brand_similarity',
    'brand_penalty',
    'size_similarity',
    'grade_adjustment',
]

feature_columns = [
    column_name
    for column_name in candidate_feature_columns
    if column_name in ml_train.columns and column_name in ml_predict_pool.columns
]

missing_feature_columns = sorted(set(candidate_feature_columns) - set(feature_columns))
if missing_feature_columns:
    print('Нет части ожидаемых feature columns:', missing_feature_columns)

if not feature_columns:
    raise ValueError('Не нашла ни одной рассчитанной feature column для final predict.')

# Для обучения оставляем только строки, где ручная разметка точно равна 0 или 1
ml_train = ml_train.loc[ml_train['manual_match'].isin([0, 1, '0', '1'])].copy()
ml_train['manual_match'] = ml_train['manual_match'].astype(int)

X_train = ml_train[feature_columns]
y_train = ml_train['manual_match']
X_predict = ml_predict_pool[feature_columns]

models = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=42,
        )),
    ]),
}

# Для auto-match выбираем не самый высокий F1, а threshold с precision >= 0.95 и максимальным recall
MIN_AUTO_MATCH_PRECISION = 0.95
high_precision_policy = (
    cv_threshold_results
    .loc[
        cv_threshold_results['precision'].ge(MIN_AUTO_MATCH_PRECISION)
        & cv_threshold_results['predicted_match_count'].gt(0)
    ]
    .sort_values(['recall', 'precision'], ascending=False)
)

if high_precision_policy.empty:
    raise ValueError('В CV-результатах нет threshold с precision >= 0.95. Нужно пересмотреть критерий auto-match.')

selected_policy = high_precision_policy.iloc[0]
selected_model_name = selected_policy['model_name']
selected_threshold = float(selected_policy['threshold'])

if selected_model_name not in models:
    raise ValueError(f'Модель из CV-таблицы не найдена в словаре models: {selected_model_name}')

prediction_output = ml_predict_pool.copy()

# Сохраняем вероятности всех моделей: это удобно для сравнения и будущего анализа ошибок
for model_name, model in models.items():
    print('Training final model:', model_name)
    model.fit(X_train, y_train)
    prediction_output[f'ml_probability_{model_name}'] = model.predict_proba(X_predict)[:, 1]

prediction_output['ml_selected_model_name'] = selected_model_name
prediction_output['ml_selected_threshold'] = selected_threshold
prediction_output['ml_match_probability'] = prediction_output[f'ml_probability_{selected_model_name}']

# Это независимое pair-level решение: каждая пара получает 1, если вероятность выше выбранного threshold
prediction_output['ml_predicted_match_pair'] = (
    prediction_output['ml_match_probability'].ge(selected_threshold)
).astype(int)

# Для matching ранжируем кандидатов Alpha Omega внутри каждого товара Sklavenitis
prediction_output = prediction_output.sort_values(
    ['item_id_sk', 'ml_match_probability'],
    ascending=[True, False]
).reset_index(drop=True)

prediction_output['ml_candidate_rank_for_item_sk'] = (
    prediction_output
    .groupby('item_id_sk')
    .cumcount()
    .add(1)
)

top2_probability_by_item = (
    prediction_output
    .loc[prediction_output['ml_candidate_rank_for_item_sk'].eq(2), ['item_id_sk', 'ml_match_probability']]
    .rename(columns={'ml_match_probability': 'ml_top2_probability_for_item_sk'})
)

prediction_output = prediction_output.merge(top2_probability_by_item, on='item_id_sk', how='left')
prediction_output['ml_gap_top1_to_top2'] = np.where(
    prediction_output['ml_candidate_rank_for_item_sk'].eq(1),
    prediction_output['ml_match_probability'] - prediction_output['ml_top2_probability_for_item_sk'],
    np.nan,
)

# auto-match кандидат - это top-1 для товара Sklavenitis, который еще и проходит выбранный threshold
prediction_output['ml_auto_match_top1'] = (
    prediction_output['ml_candidate_rank_for_item_sk'].eq(1)
    & prediction_output['ml_predicted_match_pair'].eq(1)
).astype(int)

top1_predictions = (
    prediction_output
    .loc[prediction_output['ml_candidate_rank_for_item_sk'].eq(1)]
    .copy()
)

prediction_output.to_csv(ML_FINAL_PREDICTIONS_CSV_PATH, index=False, encoding='utf-8-sig')
prediction_output.to_excel(ML_FINAL_PREDICTIONS_EXCEL_PATH, index=False)
top1_predictions.to_csv(ML_FINAL_TOP1_CSV_PATH, index=False, encoding='utf-8-sig')
top1_predictions.to_excel(ML_FINAL_TOP1_EXCEL_PATH, index=False)

print('Training rows:', len(ml_train))
print('Predict pool rows:', len(ml_predict_pool))
print('Training + predict rows:', len(ml_train) + len(ml_predict_pool))
print('Feature columns:', feature_columns)
print('Selected model:', selected_model_name)
print('Selected threshold:', selected_threshold)
print('Predicted pair-level matches:', int(prediction_output['ml_predicted_match_pair'].sum()))
print('Auto-match top1 candidates:', int(top1_predictions['ml_auto_match_top1'].sum()))
print('Saved all-pairs predictions to:', ML_FINAL_PREDICTIONS_EXCEL_PATH)
print('Saved top1 predictions to:', ML_FINAL_TOP1_EXCEL_PATH)

display(
    prediction_output
    .sort_values(['ml_auto_match_top1', 'ml_match_probability'], ascending=[False, False])
    .head(30)
)


### 28.3 Ручная доразметка датасета и сборка конечного датасета пар товаров Склавенитис - АльфаМега

- precision > 0,95 для ML-разметки оказался по границе 0,75 вероятности метода Random Forest (1-0,75)
- далее до ML-вероятности метода Random Forest = 0,5 я прошлась ручной разметкой match / not match (1/0)
- после чего я отфильтровала оставшиеся неразмеченные пары и отсортировала их по скорингу матча, который задавала вручную раннее и там отсмотрела все кандидаты выше 0,8

Когда я размечала, я сделала вывод, что скоринг работает на текущих данных лучше, чем ML-предсказание, потому что по тому как я шла ниже по датасету от 0,75 вероятности до 0,5 вероятности я не заметила увеличение количества не-матчей со спуском, более того даже сразу на 0,74 было столько не-матчей, что я сомневаюсь, что действительно выше границы 0,75 всего 5% не-матчей (precision 95%), как это насчитала модель. В скоринге же была отчетливая разница, что с понижением этого скоринга я встречала все меньше матчей 

- потом полученные матчи и не-матчи из тренировочного датасета для ML (который я собирала вручную из разных диапазонов) + размеченное ML как матч с threashold 0,75 вероятности + размеченные вручную пары от 0,75 ML-вероятности до 0,5 ML-вероятности + размеченные вручную оставшиеся пары до 0,8 вероятности скоринга собираем в один датасет
- в этом датасете фильтруем и оставляем только матчи
- смотрим, есть ли дубликаты (кейсы где к одному товару склавенитиса есть несколько товаров альфамеги и наоборот). такие случаи есть и я отсматриваю их вручную, проверяю совпадение товаров по картинкам в карточках товаров
- лишние дубликаты удаляются и остается конечный датасет сматченных пар из 841 наблюдения

### 28.4 Приведение датасета пар к конечному формату

- я добавила цены на товары Склавенитиса и АльфаМеги, а также рассчитала разницу цен между товарами - абсолютную через разность и относительную через логарифм цены АМ/цену Ск; 
- добавила колонку проверки, что размеры совпадают (только для 21 наблюдения это оказалось не так и только из-за отсутствия размерности у одного из товаров или округления/неверно занесенных объемов они различаются)

### 29. Power analysis
### Какой MDE можно обнаружить при 841 парах? С вероятностью 80 или 90%, что тест заметит эффект, если он реально существует (paired t-test)

Поскольку данные состоят из сматченных товарных пар, единицей наблюдения считается одна пара товаров. 

Анализ мощности проводится для основного парного t-теста и показывает, какую минимальную среднюю разницу в цене можно обнаружить с вероятностью 80% или 90% при уровне значимости 5%. Этот расчёт мощности напрямую относится к парному t-тесту; Wilcoxon signed-rank, sign test и bootstrap CI используются как дополнительные проверки устойчивости вывода.

In [3]:
# %pip install statsmodels

In [5]:
from pathlib import Path
import pandas as pd
from statsmodels.stats.power import TTestPower
import numpy as np

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'
POWER_ANALYSIS_CSV_PATH = OUTPUT_DIR / 'sklavenitis_alphamega_final_matches_with_prices_and_categories_v1_power_analysis.csv'

df = pd.read_csv(POWER_ANALYSIS_CSV_PATH, encoding='utf-8-sig')
df["effective_price_diff_am_minus_sk_eur"] = pd.to_numeric(df["effective_price_diff_am_minus_sk_eur"], errors="coerce")
df["effective_price_log_ratio_am_to_sk"] = pd.to_numeric(df["effective_price_log_ratio_am_to_sk"], errors="coerce")
# df["selected_for_final_one_to_one_matches"] = df["selected_for_final_one_to_one_matches"].astype(str).str.lower().eq("true")
# df = df.loc[
#     df["selected_for_final_one_to_one_matches"]
#     & df["effective_price_diff_am_minus_sk_eur"].notna()
#     & df["effective_price_log_ratio_am_to_sk"].notna()
# ].copy()

n_pairs = len(df)
alpha = 0.05

# Абсолютная разница в цене (пример: один магазин дороже другого на 0,1€ )

df["diff"] = df["effective_price_diff_am_minus_sk_eur"]
sigma_d = df["diff"].std()

analysis = TTestPower()

effect_size_80 = analysis.solve_power(
    nobs=n_pairs,
    power=0.80,
    alpha=alpha,
    alternative="two-sided"
)

effect_size_90 = analysis.solve_power(
    nobs=n_pairs,
    power=0.90,
    alpha=alpha,
    alternative="two-sided"
)

MDE_abs_80 = effect_size_80 * sigma_d
MDE_abs_90 = effect_size_90 * sigma_d



print(
    f"При {n_pairs} сматченной паре товаров тест способен обнаружить "
    f"среднюю абсолютную разницу в цене примерно от €{MDE_abs_80:.2f} "
    f"с вероятностью 80% при alpha = {alpha}."
)

print(
    f"При {n_pairs} сматченной паре товаров тест способен обнаружить "
    f"среднюю абсолютную разницу в цене примерно от €{MDE_abs_90:.2f} "
    f"с вероятностью 90% при alpha = {alpha}."
)

# Относительная разница в цене (пример: один магазин дороже другого на 5%)

df["log_ratio"] = df["effective_price_log_ratio_am_to_sk"]
sigma_log = df["log_ratio"].std()

analysis = TTestPower()

effect_size_80 = analysis.solve_power(
    nobs=n_pairs,
    power=0.80,
    alpha=alpha,
    alternative="two-sided"
)

effect_size_90 = analysis.solve_power(
    nobs=n_pairs,
    power=0.90,
    alpha=alpha,
    alternative="two-sided"
)

MDE_log_80 = effect_size_80 * sigma_log
MDE_log_90 = effect_size_90 * sigma_log

MDE_rel_80 = (np.exp(MDE_log_80) - 1) * 100
MDE_rel_90 = (np.exp(MDE_log_90) - 1) * 100


print(
    f"При {n_pairs} сматченной паре товаров тест способен обнаружить "
    f"среднюю относительную разницу в цене примерно от {MDE_rel_80:.2f}% "
    f"с вероятностью 80% при alpha = {alpha}."
)

print(
    f"При {n_pairs} сматченной паре товаров тест способен обнаружить "
    f"среднюю относительную разницу в цене примерно от {MDE_rel_90:.2f}% "
    f"с вероятностью 90% при alpha = {alpha}."
)


При 841 сматченной паре товаров тест способен обнаружить среднюю абсолютную разницу в цене примерно от €0.09 с вероятностью 80% при alpha = 0.05.
При 841 сматченной паре товаров тест способен обнаружить среднюю абсолютную разницу в цене примерно от €0.10 с вероятностью 90% при alpha = 0.05.
При 841 сматченной паре товаров тест способен обнаружить среднюю относительную разницу в цене примерно от 1.44% с вероятностью 80% при alpha = 0.05.
При 841 сматченной паре товаров тест способен обнаружить среднюю относительную разницу в цене примерно от 1.67% с вероятностью 90% при alpha = 0.05.


### 30. Brief descriptive analysis

- Средняя/медианная абсолютная разница в евро
- Средняя/медианная относительная разница в %
- Разброс: std (standard deviation), IQR (25-75 percentile), 5–95 percentile
- В каком % AlphaMega дороже / Sklavenitis дороже / цены одинаковые
- Топ товаров с самой большой разницей для sanity check

In [6]:
import numpy as np
import pandas as pd

# Берем только строки, где есть обе нужные метрики
analysis_df = df.dropna(subset=["diff", "log_ratio"]).copy()

# я знаю что их 841, но лучше проверить на всякий
n_pairs = len(analysis_df)

# Переводим log_ratio в относительную разницу в процентах
analysis_df["relative_diff_percent"] = (np.exp(analysis_df["log_ratio"]) - 1) * 100

# Считаем направления разницы
alpha_more_expensive = (analysis_df["diff"] > 0).sum()
sklavenitis_more_expensive = (analysis_df["diff"] < 0).sum()
same_price = (analysis_df["diff"] == 0).sum()

alpha_more_expensive_pct = alpha_more_expensive / n_pairs * 100
sklavenitis_more_expensive_pct = sklavenitis_more_expensive / n_pairs * 100
same_price_pct = same_price / n_pairs * 100

# Summary table
descriptive_summary = pd.DataFrame({
    "metric": [
        "Number of matched product pairs",
        "Mean absolute price difference, EUR",
        "Median absolute price difference, EUR",
        "Std of absolute price difference, EUR",
        "25th percentile absolute difference, EUR",
        "75th percentile absolute difference, EUR",
        "5th percentile absolute difference, EUR",
        "95th percentile absolute difference, EUR",
        "Mean relative price difference, %",
        "Median relative price difference, %",
        "Std of relative price difference, %",
        "25th percentile relative difference, %",
        "75th percentile relative difference, %",
        "5th percentile relative difference, %",
        "95th percentile relative difference, %",
        "AlphaMega more expensive, %",
        "Sklavenitis more expensive, %",
        "Same price, %"
    ],
    "value": [
        n_pairs,
        analysis_df["diff"].mean(),
        analysis_df["diff"].median(),
        analysis_df["diff"].std(),
        analysis_df["diff"].quantile(0.25),
        analysis_df["diff"].quantile(0.75),
        analysis_df["diff"].quantile(0.05),
        analysis_df["diff"].quantile(0.95),
        analysis_df["relative_diff_percent"].mean(),
        analysis_df["relative_diff_percent"].median(),
        analysis_df["relative_diff_percent"].std(),
        analysis_df["relative_diff_percent"].quantile(0.25),
        analysis_df["relative_diff_percent"].quantile(0.75),
        analysis_df["relative_diff_percent"].quantile(0.05),
        analysis_df["relative_diff_percent"].quantile(0.95),
        alpha_more_expensive_pct,
        sklavenitis_more_expensive_pct,
        same_price_pct
    ]
})

# мне впадлу это делать как-то красиво через print, поэтому я просто выведу так, все равно я делаю это для себя
descriptive_summary

,metric,value
0,Number of matched product pairs,841.000000
1,"Mean absolute price difference, EUR",0.048133
2,"Median absolute price difference, EUR",0.040000
3,"Std of absolute price difference, EUR",0.903768
4,"25th percentile absolute difference, EUR",-0.010000
5,"75th percentile absolute difference, EUR",0.200000
6,"5th percentile absolute difference, EUR",-0.600000
7,"95th percentile absolute difference, EUR",0.640000
8,"Mean relative price difference, %",3.203536
9,"Median relative price difference, %",1.343284
